# TEM image denoising and atomic column localization

Pipeline: AtomSegNet vs UNet++ vs HRNet on TEM-ImageNet-v1.3.

Tasks
1. Self-supervised denoising (Noise2Void) plus a supervised denoising baseline
2. Atomic column segmentation (binary mask)
3. Sub-pixel atom localization and interatomic spacing extraction
4. 5-fold cross-validated benchmark across the three architectures

Dataset: https://github.com/xinhuolin/TEM-ImageNet-v1.3

Runs on the IITJ HPC. Data lives at `~/utkarsh/TEM-ImageNet-v1.3-master`,
outputs at `~/utkarsh/runs`. Submit `jupyter_gpu.sh` with `sbatch`, tunnel to
the assigned node, and run here; or run the whole thing headless with
`sbatch train.sbatch`. Do not start Jupyter on the login node.

Every long-running cell is resumable. If Slurm kills the job, resubmit and it
continues from the last completed epoch.


## 0. Environment and imports


In [ ]:
import os

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("SLURM_JOB_GPUS       =", os.environ.get("SLURM_JOB_GPUS"))
print("SLURM_STEP_GPUS      =", os.environ.get("SLURM_STEP_GPUS"))
print("SLURM_GPUS_ON_NODE   =", os.environ.get("SLURM_GPUS_ON_NODE"))

In [ ]:
import os

# MUST happen before importing torch anywhere
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

# Physical GPU 0 now appears as logical cuda:0
DEVICE = torch.device("cuda:0")

print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Visible GPU count:", torch.cuda.device_count())
print("Logical device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# 0. Imports, threading, determinism, device
# This cell was EMPTY in the previous version of the notebook,
# which is why cv2 / np / plt / SEED were undefined downstream.
# ============================================================
import os
import sys

# Thread caps must be set before numpy / torch / cv2 are imported, otherwise
# every DataLoader worker spawns OMP_NUM_THREADS=<all cores> and the node
# thrashes. Slurm tells us how many CPUs we actually own.
_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 4))
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import gc
import io
import json
import math
import time
import random
import shutil
import socket
import signal
import warnings
import subprocess
import multiprocessing as mp
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd

import matplotlib
# Compute nodes have no display. Force Agg before pyplot is imported so that
# `nbconvert --execute` under sbatch behaves exactly like the interactive run.
if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None

try:
    import cv2
    cv2.setNumThreads(0)          # OpenCV's own pool fights the DataLoader workers
except ImportError as e:
    raise ImportError(
        "cv2 is missing. On a compute node install the headless build:\n"
        "    pip install --user opencv-python-headless\n"
        "The normal opencv-python wheel needs libGL.so.1, which is not "
        "installed on gpu nodes."
    ) from e

from PIL import Image
Image.MAX_IMAGE_PIXELS = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import KFold

from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from skimage.feature import peak_local_max

from scipy.spatial import cKDTree
KDTree = cKDTree
from scipy.optimize import curve_fit

try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- reproducibility ----------------
SEED = 42


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(SEED)


def worker_init_fn(worker_id):
    """PyTorch reseeds `torch` and `random` in each fork, but NOT numpy.
    Without this, every worker draws the identical Noise2Void mask and the
    identical augmentation stream, so 4 workers give you 1 worker's worth of
    randomness."""
    base = torch.initial_seed() % (2 ** 31 - 1)
    np.random.seed((base + worker_id) % (2 ** 31 - 1))
    random.seed(base + worker_id)


# ---------------- device ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True          # fixed 256x256 shapes -> safe
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# bfloat16 needs compute capability >= 8.0 (A100 is 8.0). It has fp32's
# exponent range, so it needs no GradScaler and cannot produce inf-loss steps.
BF16_OK = (
    DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("Python      :", sys.version.split()[0])
print("Node        :", socket.gethostname())
print("Slurm job   :", os.environ.get("SLURM_JOB_ID", "(none - login node?)"))
print("CPUs owned  :", _CPUS)
print("torch       :", torch.__version__, "| CUDA build", torch.version.cuda)
print("device      :", DEVICE)
if DEVICE.type == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print("GPU         :", _p.name, f"| {_p.total_memory / 1024**3:.1f} GiB",
          f"| sm_{_p.major}{_p.minor}")
    print("bfloat16    :", BF16_OK)
else:
    print("WARNING: no CUDA device. You are probably on the login node. "
          "Submit jupyter_gpu.sh with sbatch instead.")


In [ ]:
# Confirm we are on a GPU node, not the login node.
assert DEVICE.type == 'cuda' or os.environ.get('ALLOW_CPU') == '1', (
    'No GPU visible. You are probably on login.iitj.ac.in. '
    'Submit jupyter_gpu.sh with sbatch, or set ALLOW_CPU=1 to override.'
)
if DEVICE.type == 'cuda':
    print(subprocess.run(['nvidia-smi',
                          '--query-gpu=name,memory.total,memory.used,utilization.gpu',
                          '--format=csv'],
                         capture_output=True, text=True).stdout)


## 1. Configuration


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
_cwd = Path.cwd().resolve()
_DEFAULT_PROJECT_ROOT = _cwd.parent if _cwd.name.lower() == "notebooks" else _cwd

def _pick_scratch():
    """Node-local disk beats /iitjhome for a dataset of thousands of small
    PNGs. /iitjhome is NFS: every __getitem__ becomes a network round trip and
    the A100 sits idle waiting on the filesystem. Prefer whatever local
    scratch the node exposes; fall back to home."""
    job = os.environ.get("SLURM_JOB_ID", "")
    for base in ("/scratch", "/local_scratch", "/local", "/tmp"):
        p = Path(base)
        if p.is_dir() and os.access(base, os.W_OK):
            d = p / f"tem_cache_{job}" if job else p / "tem_cache"
            try:
                d.mkdir(parents=True, exist_ok=True)
                return d
            except OSError:
                continue
    return Path.home() / "tem_cache"


class CFG:
    # ---------------- portable paths; override through environment ----------
    PROJECT_ROOT = Path(os.environ.get("TEM_PROJECT_ROOT", str(_DEFAULT_PROJECT_ROOT)))
    DATA_ROOT    = os.environ.get("TEM_DATA_ROOT", str(PROJECT_ROOT / "data" / "TEM-ImageNet-v1.3"))
    OUT_DIR      = os.environ.get("TEM_RESULTS_ROOT", str(PROJECT_ROOT / "results_local"))
    CACHE_DIR    = os.environ.get("TEM_CACHE_ROOT", str(_pick_scratch()))

    # ---------------- dataset subfolders ----------------
    SUBDIR_NOISY  = "image"
    SUBDIR_CLEAN  = "noNoise"
    SUBDIR_MASK   = "circularMask"     # binary segmentation target
    SUBDIR_GAUSS  = "gaussianMask"     # sub-pixel atom-position target
    SUBDIR_COORDS = "position"         # NOTE: unit-cell vectors, NOT atom xy

    # Ground truth for peak finding. `position/` in TEM-ImageNet-v1.3 holds
    # lattice vectors, not per-atom coordinates, so gaussianMask is the
    # correct source for localization ground truth.
    LOCALIZATION_GT = "gaussianMask"

    MAX_SAMPLES = None                 # int for a smoke test, None for all

    # ---------------- image ----------------
    IMG_SIZE = 256
    IN_CH = 1

    # ---------------- training ----------------
    # 256x256x1 on a 40 GB A100: batch 8 uses about 6% of the card. Batch 32
    # gives roughly 3.5x the throughput at the same epoch count.
    BATCH        = 32
    EVAL_BATCH   = 64
    GRAD_ACCUM   = 1
    EPOCHS       = 60
    LR           = 2e-3                # scaled with the larger batch
    WD           = 1e-5
    N_FOLDS      = 5
    PATIENCE     = 8
    MIN_EPOCHS   = 8
    EMA_DECAY    = 0.999               # None disables the EMA shadow model

    # loss weights
    LAM_DEN      = 1.0
    LAM_SEG      = 1.0
    LAM_GRAD     = 0.1                 # edge-preserving term on the denoiser
    TVERSKY_BETA = 0.7                 # >0.5 penalises missed atoms harder

    # ---------------- runtime ----------------
    NUM_WORKERS   = _CPUS
    PRECISION     = "bf16" if BF16_OK else ("fp16" if DEVICE.type == "cuda" else "fp32")
    CHANNELS_LAST = True
    COMPILE       = False              # torch.compile; adds ~60 s warmup per arch
    USE_CACHE     = True               # memmap cache instead of per-file reads
    CACHE_WORKERS = max(1, _CPUS)

    # Stop cleanly this many seconds before the Slurm walltime expires so the
    # resume file is written instead of the job being SIGKILLed mid-epoch.
    WALLTIME_MARGIN_S = 600

    # ---------------- Noise2Void ----------------
    N2V_MASK_RATIO = 0.02
    N2V_RADIUS     = 5
    N2V_EPOCHS     = 15

    # ---------------- localization ----------------
    PEAK_MIN_DIST = 4
    PEAK_THRESH   = 0.30
    PIXEL_SIZE_A  = 0.20               # placeholder for simulated data

    ARCHS = ["AtomSegNet", "UNetPP", "HRNet"]

    # per-architecture batch override (HRNet keeps full resolution and costs
    # more activation memory than the two U-shaped nets)
    BATCH_OVERRIDE = {"HRNet": 16}


Path(CFG.OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.CACHE_DIR).mkdir(parents=True, exist_ok=True)


def arch_batch(arch_name):
    return CFG.BATCH_OVERRIDE.get(arch_name, CFG.BATCH)


# ---------------- Slurm walltime awareness ----------------
def seconds_left():
    """Seconds until Slurm kills this job, or inf outside Slurm."""
    end = os.environ.get("SLURM_JOB_END_TIME")
    if end:
        try:
            return float(end) - time.time()
        except ValueError:
            pass
    try:
        jid = os.environ.get("SLURM_JOB_ID")
        if not jid:
            return float("inf")
        out = subprocess.run(
            ["squeue", "-h", "-j", jid, "-o", "%L"],
            capture_output=True, text=True, timeout=10
        ).stdout.strip()
        if not out or out in ("UNLIMITED", "INVALID"):
            return float("inf")
        days, _, rest = out.rpartition("-")
        parts = [int(x) for x in rest.split(":")]
        while len(parts) < 3:
            parts.insert(0, 0)
        s = parts[0] * 3600 + parts[1] * 60 + parts[2]
        if days:
            s += int(days) * 86400
        return float(s)
    except Exception:
        return float("inf")


def time_is_short(margin=None):
    margin = CFG.WALLTIME_MARGIN_S if margin is None else margin
    return seconds_left() < margin


# ---------------- graceful SIGTERM (Slurm preemption) ----------------
STOP_REQUESTED = {"flag": False}


def _on_sigterm(signum, frame):
    STOP_REQUESTED["flag"] = True
    print("\n[signal] SIGTERM received: will checkpoint and exit at the next "
          "epoch boundary.", flush=True)


try:
    signal.signal(signal.SIGTERM, _on_sigterm)
except (ValueError, OSError):
    pass


print("Dataset   :", CFG.DATA_ROOT)
print("Outputs   :", CFG.OUT_DIR)
print("Cache     :", CFG.CACHE_DIR)
print("Workers   :", CFG.NUM_WORKERS)
print("Precision :", CFG.PRECISION)
print("Batch     :", CFG.BATCH, "| overrides:", CFG.BATCH_OVERRIDE)
_left = seconds_left()
print("Walltime  :", "unlimited" if _left == float("inf") else f"{_left/3600:.2f} h left")


## 2. Dataset presence check

Verifies that the copied dataset is where the config says it is and that no
extra directory level was introduced by `scp -r`.


In [ ]:
# ============================================================
# 2. Dataset presence + Swin-UNet compatibility check
# ============================================================
from pathlib import Path
from PIL import Image

IMG_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

EXPECTED_SUBDIRS = [
    CFG.SUBDIR_NOISY,
    CFG.SUBDIR_CLEAN,
    CFG.SUBDIR_MASK,
    CFG.SUBDIR_GAUSS,
]

# Recommended for your 256 x 256 TEM tiles
SWIN_PATCH_SIZE  = 4
SWIN_WINDOW_SIZE = 8
SWIN_NUM_STAGES  = 4

# True means every Swin stage must partition into complete windows.
# Set False only if your implementation explicitly pads windows.
STRICT_WINDOW_PARTITION = True

# Number of files sampled per modality for shape verification.
# Use None to check every image.
SHAPE_CHECK_LIMIT = 256


def image_files(folder: Path):
    """Return supported image files directly inside a folder."""
    return sorted(
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )


def has_required_layout(folder: Path) -> bool:
    """A valid dataset root must contain all four modality folders."""
    return (
        folder.is_dir()
        and all((folder / name).is_dir() for name in EXPECTED_SUBDIRS)
    )


def resolve_data_root(root):
    """Find the dataset root, allowing up to three nested levels."""
    root = Path(root).expanduser().resolve()

    if not root.exists():
        raise RuntimeError(
            f"Dataset folder does not exist:\n  {root}"
        )

    if has_required_layout(root):
        return root

    current = root

    for _ in range(3):
        subdirs = sorted(
            p for p in current.iterdir() if p.is_dir()
        )

        hits = [p for p in subdirs if has_required_layout(p)]

        if len(hits) == 1:
            print(
                "note: dataset was nested one level deeper "
                f"-> {hits[0]}"
            )
            return hits[0]

        if len(hits) > 1:
            raise RuntimeError(
                "More than one possible dataset root was found:\n"
                + "\n".join(f"  {p}" for p in hits)
            )

        if len(subdirs) != 1:
            break

        current = subdirs[0]

    contents = sorted(p.name for p in root.iterdir())[:20]

    raise RuntimeError(
        f"Required folders were not found under:\n  {root}\n"
        f"Expected all of: {EXPECTED_SUBDIRS}\n"
        f"Contents: {contents}"
    )


def sampled_files(files, limit):
    """Uniformly sample files across a sorted file list."""
    if limit is None or len(files) <= limit:
        return files

    step = max(1, len(files) // limit)
    return files[::step][:limit]


def read_hw(path: Path):
    """Read image height and width without loading all pixel data."""
    with Image.open(path) as image:
        width, height = image.size
    return height, width


def check_swin_shape(height, width):
    """
    Validate input geometry for a four-stage Swin-UNet.

    For 256 x 256, patch_size=4 and window_size=8:
        stage grids = 64, 32, 16, 8
    """
    if height != width:
        raise RuntimeError(
            f"Swin-UNet currently expects square tiles, but found "
            f"{height} x {width}."
        )

    total_downsampling = (
        SWIN_PATCH_SIZE * 2 ** (SWIN_NUM_STAGES - 1)
    )

    if (
        height % total_downsampling != 0
        or width % total_downsampling != 0
    ):
        raise RuntimeError(
            f"Image size {height} x {width} is incompatible with "
            f"patch_size={SWIN_PATCH_SIZE} and "
            f"{SWIN_NUM_STAGES} Swin stages.\n"
            f"Each dimension must be divisible by "
            f"{total_downsampling}."
        )

    stage_shapes = []

    for stage in range(SWIN_NUM_STAGES):
        divisor = SWIN_PATCH_SIZE * (2 ** stage)
        grid_h = height // divisor
        grid_w = width // divisor
        stage_shapes.append((grid_h, grid_w))

        if STRICT_WINDOW_PARTITION and (
            grid_h % SWIN_WINDOW_SIZE != 0
            or grid_w % SWIN_WINDOW_SIZE != 0
        ):
            raise RuntimeError(
                f"Swin stage {stage + 1} has feature grid "
                f"{grid_h} x {grid_w}, which cannot be partitioned "
                f"exactly using window_size={SWIN_WINDOW_SIZE}.\n"
                "For 256 x 256 inputs, use window_size=8, or use a "
                "Swin implementation with explicit window padding."
            )

    return stage_shapes


# ------------------------------------------------------------
# Resolve root
# ------------------------------------------------------------
CFG.DATA_ROOT = str(resolve_data_root(CFG.DATA_ROOT))
data_root = Path(CFG.DATA_ROOT)

print("Dataset root:", data_root)


# ------------------------------------------------------------
# Count modality files
# ------------------------------------------------------------
files_by_modality = {
    name: image_files(data_root / name)
    for name in EXPECTED_SUBDIRS
}

print("\nsubfolder                              images")
print("-" * 48)

for folder in sorted(data_root.iterdir()):
    if folder.is_dir():
        n = len(image_files(folder))
        suffix = "" if n else "   (non-image folder)"
        print(f"{folder.name:38s} {n:6d}{suffix}")

counts = {
    name: len(files)
    for name, files in files_by_modality.items()
}

print("\nmodalities in use:", counts)

if any(count == 0 for count in counts.values()):
    raise RuntimeError(
        "At least one required modality contains no images:\n"
        f"{counts}"
    )

if len(set(counts.values())) != 1:
    raise RuntimeError(
        "The four modalities have different file counts.\n"
        f"{counts}\n"
        "Fix the missing pairs before training so every fold uses "
        "the same samples."
    )

print(
    "all four modalities present and consistent "
    f"({next(iter(counts.values()))} files each)"
)


# ------------------------------------------------------------
# Check image sizes
# ------------------------------------------------------------
sizes_by_modality = {}

for name, files in files_by_modality.items():
    selected = sampled_files(files, SHAPE_CHECK_LIMIT)
    sizes = {read_hw(path) for path in selected}
    sizes_by_modality[name] = sizes

    if len(sizes) != 1:
        raise RuntimeError(
            f"Multiple image sizes found in '{name}': "
            f"{sorted(sizes)}"
        )

all_sizes = set().union(*sizes_by_modality.values())

if len(all_sizes) != 1:
    raise RuntimeError(
        "Input and target modalities have different image sizes:\n"
        + "\n".join(
            f"  {name}: {sorted(sizes)}"
            for name, sizes in sizes_by_modality.items()
        )
    )

image_h, image_w = next(iter(all_sizes))
stage_shapes = check_swin_shape(image_h, image_w)

print(f"\nTEM tile size: {image_h} x {image_w}")
print(
    f"Swin settings: patch={SWIN_PATCH_SIZE}, "
    f"window={SWIN_WINDOW_SIZE}, stages={SWIN_NUM_STAGES}"
)
print("Swin feature grids:", stage_shapes)
print("Dataset is compatible with Swin-UNet.")

## 3. Dataset discovery and pairing

TEM-ImageNet-v1.3 ships paired files sharing a basename across `image/`,
`noNoise/`, `circularMask/`, `gaussianMask/` and `position/`. The subfolders
are auto-detected and the basenames intersected so every retained sample has
all modalities.

`position/` holds unit-cell vectors rather than per-atom coordinates, so
`gaussianMask/` is the ground truth used for peak finding in section 11.


In [ ]:
# ============================================================
# 3. Dataset discovery and pairing
# ============================================================
def discover_dataset(root):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise RuntimeError(f"Not a directory: {root}")

    candidates = {
        'noisy':  [CFG.SUBDIR_NOISY, 'image', 'images', 'noisy', 'input'],
        'clean':  [CFG.SUBDIR_CLEAN, 'noNoise', 'noiseFree', 'clean', 'gt',
                   'ground_truth'],
        'mask':   [CFG.SUBDIR_MASK, 'circularMask', 'mask', 'masks',
                   'segmentation'],
        'gauss':  [CFG.SUBDIR_GAUSS, 'gaussianMask', 'gaussian', 'heatmap'],
        'coords': [CFG.SUBDIR_COORDS, 'position', 'coords', 'coordinates',
                   'labels'],
    }

    available = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}
    found = {}
    for k, names in candidates.items():
        for n in names:
            if not n:
                continue
            p = root / str(n).strip()
            if p.is_dir():
                found[k] = p
                break
            p = available.get(Path(str(n)).name.lower())
            if p is not None:
                found[k] = p
                break

    print("Discovered subfolders:")
    for k in ('noisy', 'clean', 'mask', 'gauss', 'coords'):
        print(f"  {k:7s} -> {found.get(k, '(missing)')}")

    if 'noisy' not in found:
        raise RuntimeError(f"No noisy-image folder under {root}")
    if 'gauss' not in found:
        print("\nWARNING: gaussianMask/ is missing. Section 10 will fall back "
              "to circularMask centroids for localization ground truth, which "
              "is coarser than the sub-pixel Gaussian peaks.")
    return found


DIRS = discover_dataset(CFG.DATA_ROOT)


def list_basenames(d, exts=IMG_EXTS):
    d = Path(d)
    if not d.is_dir():
        return {}
    exts = {e.lower() for e in exts}
    files = sorted(
        (f for f in d.rglob('*') if f.is_file() and f.suffix.lower() in exts),
        key=lambda f: (f.stem.lower(), str(f).lower())
    )
    file_map = {}
    for f in files:
        file_map.setdefault(f.stem, f)
    return file_map


noisy_map  = list_basenames(DIRS['noisy'])
clean_map  = list_basenames(DIRS['clean']) if 'clean' in DIRS else {}
mask_map   = list_basenames(DIRS['mask'])  if 'mask'  in DIRS else {}
gauss_map  = list_basenames(DIRS['gauss']) if 'gauss' in DIRS else {}
coords_map = list_basenames(DIRS['coords'], exts=('.txt', '.csv', '.npy')) \
    if 'coords' in DIRS else {}

common = set(noisy_map)
if clean_map:
    common &= set(clean_map)
if mask_map:
    common &= set(mask_map)
common = sorted(common)

if CFG.MAX_SAMPLES is not None and len(common) > CFG.MAX_SAMPLES:
    rng = np.random.RandomState(SEED)
    idx = sorted(rng.choice(len(common), CFG.MAX_SAMPLES, replace=False))
    common = [common[i] for i in idx]
    print(f"\nCapped to {CFG.MAX_SAMPLES} samples (CFG.MAX_SAMPLES).")

n_gauss = sum(1 for b in common if b in gauss_map)

print(f"\nPaired samples : {len(common)}")
print(f"  noisy        : {len(noisy_map)}")
print(f"  clean GT     : {len(clean_map)}")
print(f"  mask GT      : {len(mask_map)}")
print(f"  gaussian GT  : {len(gauss_map)}  ({n_gauss} of the paired set)")

if not common:
    raise RuntimeError(
        "Zero paired samples. The basenames across subfolders do not "
        "intersect. Print a few from each map and compare the stems."
    )


In [ ]:
# ============================================================
# 3b. Image IO + visual sanity check
# ============================================================
def imread_gray(p):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(f"Image file not found: {p}")

    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if img is None:
        img = np.array(Image.open(p))

    img = np.asarray(img)
    if img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
        elif img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            img = img[..., 0]
    if img.ndim != 2:
        raise ValueError(f"Expected 2D grayscale, got {img.shape} for {p}")
    return img.astype(np.float32)


def norm01(x, name="", warn=False):
    """Percentile-stretch to [0,1]. Returns zeros on a flat image; set
    warn=True while debugging so a blank panel is not silent."""
    x = np.asarray(x, dtype=np.float32)
    finite = np.isfinite(x)
    if not finite.any():
        if warn:
            print(f"WARN: all-nonfinite image {name}")
        return np.zeros_like(x, dtype=np.float32)

    lo, hi = np.percentile(x[finite], (1, 99))
    if hi - lo < 1e-6:
        if warn:
            print(f"WARN: flat dynamic range for {name} (lo={lo}, hi={hi})")
        return np.zeros_like(x, dtype=np.float32)

    x = np.nan_to_num(x, nan=lo, posinf=hi, neginf=lo)
    return np.clip((x - lo) / (hi - lo), 0.0, 1.0)


def _panel(ax, arr, title, cmap='gray'):
    if arr is None:
        ax.text(0.5, 0.5, 'not available', ha='center', va='center')
    else:
        ax.imshow(arr, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=10)
    ax.axis('off')


if common:
    sample = common[0]
    fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))

    _panel(axes[0], norm01(imread_gray(noisy_map[sample]), 'noisy', warn=True),
           'noisy (input)')
    _panel(axes[1],
           norm01(imread_gray(clean_map[sample]), 'clean', warn=True)
           if sample in clean_map else None,
           'noNoise (denoise GT)')
    _panel(axes[2],
           (imread_gray(mask_map[sample]) > 0).astype(np.float32)
           if sample in mask_map else None,
           'circularMask (seg GT)')
    _panel(axes[3],
           norm01(imread_gray(gauss_map[sample]), 'gauss', warn=True)
           if sample in gauss_map else None,
           'gaussianMask (localization GT)', cmap='inferno')

    fig.suptitle(f"sample: {sample}", fontsize=12)
    plt.tight_layout()
    plt.savefig(Path(CFG.OUT_DIR) / 'sanity_check.png', dpi=140,
                bbox_inches='tight')
    plt.show()

    # shape agreement across modalities is worth one check
    shapes = {k: imread_gray(m[sample]).shape
              for k, m in (('noisy', noisy_map), ('clean', clean_map),
                           ('mask', mask_map), ('gauss', gauss_map))
              if sample in m}
    print("shapes:", shapes)
    if len(set(shapes.values())) > 1:
        print("WARNING: modalities have different pixel dimensions; the "
              "resize step will realign them but check that they correspond.")
else:
    print("No paired samples available for the visual sanity check.")


## 3c. Memmap cache

`/iitjhome` is NFS. Reading thousands of small PNGs per epoch turns every
`__getitem__` into a network round trip and the A100 idles. This packs the
dataset into four contiguous uint8 arrays on node-local scratch, already
resized. The build runs once per node; after that a sample read is a memcpy
from page cache.


In [ ]:
# ============================================================
# 3c. Build uint8 memmap cache — Swin-UNet compatible
# ============================================================
import json
import multiprocessing as mp
import os
import shutil
import socket
import time
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm


CACHE_KEYS = ("noisy", "clean", "mask", "gauss")

# Must match the Swin-UNet constructor
SWIN_PATCH_SIZE  = int(getattr(CFG, "SWIN_PATCH_SIZE", 4))
SWIN_WINDOW_SIZE = int(getattr(CFG, "SWIN_WINDOW_SIZE", 8))
SWIN_NUM_STAGES  = int(getattr(CFG, "SWIN_NUM_STAGES", 4))


# ------------------------------------------------------------
# Swin input-size validation
# ------------------------------------------------------------
def validate_swin_cache_size(size):
    size = int(size)

    total_downsample = (
        SWIN_PATCH_SIZE * 2 ** (SWIN_NUM_STAGES - 1)
    )

    if size % total_downsample != 0:
        raise ValueError(
            f"CFG.IMG_SIZE={size} is incompatible with "
            f"patch_size={SWIN_PATCH_SIZE} and "
            f"{SWIN_NUM_STAGES} Swin stages. "
            f"It must be divisible by {total_downsample}."
        )

    grids = [
        size // (SWIN_PATCH_SIZE * 2**stage)
        for stage in range(SWIN_NUM_STAGES)
    ]

    incompatible = [
        grid for grid in grids
        if grid % SWIN_WINDOW_SIZE != 0
    ]

    if incompatible:
        raise ValueError(
            f"Swin feature grids {grids} are not all divisible by "
            f"window_size={SWIN_WINDOW_SIZE}. "
            "For 256-pixel inputs use window_size=8."
        )

    print(
        f"Swin cache geometry: {size}x{size} | "
        f"feature grids {grids} | compatible"
    )

    return grids


SWIN_STAGE_GRIDS = validate_swin_cache_size(CFG.IMG_SIZE)


# ------------------------------------------------------------
# Cache paths
# ------------------------------------------------------------
def _cache_paths():
    cache_dir = Path(CFG.CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)

    # Do not include architecture in this tag. The same cache can
    # serve AtomSegNet, HRNet, UNet++, and Swin-UNet.
    tag = f"{CFG.IMG_SIZE}_{len(common)}"

    paths = {
        key: cache_dir / f"{key}_{tag}.npy"
        for key in CACHE_KEYS
    }

    return paths, cache_dir / f"meta_{tag}.json"


def _resize(img, size, interp):
    if img.shape[:2] != (size, size):
        img = cv2.resize(
            img,
            (size, size),
            interpolation=interp,
        )
    return img


def _cache_worker_init():
    # Prevent each multiprocessing worker from starting its own
    # OpenCV thread pool.
    cv2.setNumThreads(0)


def _encode_one(args):
    """Return the four uint8 arrays for one sample."""
    idx, noisy_path, clean_path, mask_path, gauss_path, size = args

    out = {}

    # Noisy input
    noisy = norm01(imread_gray(noisy_path))
    noisy = _resize(noisy, size, cv2.INTER_AREA)

    out["noisy"] = np.clip(
        noisy * 255.0 + 0.5, 0, 255
    ).astype(np.uint8)

    # Clean reconstruction target
    if clean_path is not None:
        clean = norm01(imread_gray(clean_path))
        clean = _resize(clean, size, cv2.INTER_AREA)
    else:
        clean = noisy

    out["clean"] = np.clip(
        clean * 255.0 + 0.5, 0, 255
    ).astype(np.uint8)

    # Binary segmentation mask
    if mask_path is not None:
        mask = imread_gray(mask_path)

        finite = np.isfinite(mask)
        maximum = np.nanmax(mask) if finite.any() else 0.0

        if maximum > 0:
            mask = (mask > 0.5 * maximum).astype(np.uint8)
        else:
            mask = np.zeros_like(mask, dtype=np.uint8)

        mask = _resize(mask, size, cv2.INTER_NEAREST)
    else:
        mask = np.zeros((size, size), dtype=np.uint8)

    out["mask"] = (mask > 0).astype(np.uint8)

    # Gaussian localization target
    if gauss_path is not None:
        gauss = norm01(imread_gray(gauss_path))
        gauss = _resize(gauss, size, cv2.INTER_AREA)

        out["gauss"] = np.clip(
            gauss * 255.0 + 0.5, 0, 255
        ).astype(np.uint8)
    else:
        out["gauss"] = np.zeros(
            (size, size),
            dtype=np.uint8,
        )

    return idx, out


def _allocated_cache_workers():
    requested = int(CFG.CACHE_WORKERS)

    try:
        allocated = int(
            os.environ.get("SLURM_CPUS_PER_TASK", requested)
        )
    except (TypeError, ValueError):
        allocated = requested

    return max(1, min(requested, allocated, 16))


def build_cache(force=False):
    paths, meta_path = _cache_paths()

    N = len(common)
    S = int(CFG.IMG_SIZE)

    if N == 0:
        raise RuntimeError("Cannot build cache: no paired samples found.")

    def _valid_array(path):
        if not path.is_file():
            return False

        try:
            array = np.load(path, mmap_mode="r")
            valid = (
                array.shape == (N, S, S)
                and array.dtype == np.uint8
            )
            del array
            return valid
        except Exception:
            return False

    # --------------------------------------------------------
    # Existing-cache check
    # --------------------------------------------------------
    if (
        not force
        and meta_path.is_file()
        and all(_valid_array(path) for path in paths.values())
    ):
        try:
            meta = json.loads(
                meta_path.read_text(encoding="utf-8")
            )
        except Exception:
            meta = {}

        if meta.get("basenames") == list(common):
            print(
                f"cache hit: {N} samples at {S}px "
                f"in {CFG.CACHE_DIR}"
            )
            return paths, meta

        print("cache sample list changed; rebuilding")

    required_bytes = len(CACHE_KEYS) * N * S * S
    free_bytes = shutil.disk_usage(CFG.CACHE_DIR).free

    if free_bytes < required_bytes * 1.20:
        raise RuntimeError(
            f"Insufficient cache space: need approximately "
            f"{required_bytes / 1024**3:.2f} GiB, but only "
            f"{free_bytes / 1024**3:.2f} GiB is free."
        )

    print(
        f"building cache: {N} samples -> "
        f"{required_bytes / 1024**3:.2f} GiB "
        f"in {CFG.CACHE_DIR}"
    )

    # PID-specific temporary names avoid collisions between jobs.
    pid = os.getpid()

    tmp_paths = {
        key: path.with_name(f"{path.name}.{pid}.tmp")
        for key, path in paths.items()
    }

    memmaps = {}

    try:
        memmaps = {
            key: np.lib.format.open_memmap(
                tmp_paths[key],
                mode="w+",
                dtype=np.uint8,
                shape=(N, S, S),
            )
            for key in CACHE_KEYS
        }

        jobs = [
            (
                index,
                noisy_map[basename],
                clean_map.get(basename),
                mask_map.get(basename),
                gauss_map.get(basename),
                S,
            )
            for index, basename in enumerate(common)
        ]

        t0 = time.time()
        nproc = _allocated_cache_workers()

        print(
            f"cache workers: {nproc} "
            f"(SLURM_CPUS_PER_TASK="
            f"{os.environ.get('SLURM_CPUS_PER_TASK', 'unset')})"
        )

        if nproc > 1:
            ctx = mp.get_context("fork")

            with ctx.Pool(
                nproc,
                initializer=_cache_worker_init,
            ) as pool:
                iterator = pool.imap_unordered(
                    _encode_one,
                    jobs,
                    chunksize=32,
                )

                for index, output in tqdm(
                    iterator,
                    total=N,
                    desc="cache",
                ):
                    for key in CACHE_KEYS:
                        memmaps[key][index] = output[key]

        else:
            _cache_worker_init()

            for job in tqdm(jobs, desc="cache"):
                index, output = _encode_one(job)

                for key in CACHE_KEYS:
                    memmaps[key][index] = output[key]

        # Flush before replacing final files
        for array in memmaps.values():
            array.flush()

        memmaps.clear()

        for key, final_path in paths.items():
            os.replace(tmp_paths[key], final_path)

        meta = {
            "n": N,
            "size": S,
            "basenames": list(common),
            "built": time.strftime("%Y-%m-%d %H:%M:%S"),
            "node": socket.gethostname(),
            "data_root": str(Path(CFG.DATA_ROOT).resolve()),
            "dtype": "uint8",
            "swin_patch_size": SWIN_PATCH_SIZE,
            "swin_window_size": SWIN_WINDOW_SIZE,
            "swin_stage_grids": SWIN_STAGE_GRIDS,
            "has_gauss": [
                basename in gauss_map
                for basename in common
            ],
        }

        tmp_meta = meta_path.with_name(
            f"{meta_path.name}.{pid}.tmp"
        )
        tmp_meta.write_text(
            json.dumps(meta),
            encoding="utf-8",
        )
        os.replace(tmp_meta, meta_path)

        print(f"cache built in {time.time() - t0:.1f} s")

        return paths, meta

    except Exception:
        memmaps.clear()

        for path in tmp_paths.values():
            path.unlink(missing_ok=True)

        raise


# ------------------------------------------------------------
# Build/open cache
# ------------------------------------------------------------
CACHE = None
CACHE_INDEX = {}

if CFG.USE_CACHE and common:
    try:
        cache_paths, cache_meta = build_cache()

        CACHE = {
            key: np.load(path, mmap_mode="r")
            for key, path in cache_paths.items()
        }

        expected_shape = (
            len(common),
            CFG.IMG_SIZE,
            CFG.IMG_SIZE,
        )

        for key, array in CACHE.items():
            assert array.shape == expected_shape, (
                f"cache '{key}' has shape {array.shape}; "
                f"expected {expected_shape}"
            )
            assert array.dtype == np.uint8, (
                f"cache '{key}' has dtype {array.dtype}; "
                "expected uint8"
            )

        # Validate masks at several positions, not only sample zero.
        check_indices = np.linspace(
            0,
            len(common) - 1,
            min(16, len(common)),
            dtype=int,
        )

        mask_values = set(
            np.unique(
                np.asarray(CACHE["mask"][check_indices])
            ).tolist()
        )

        assert mask_values <= {0, 1}, (
            f"cache mask contains values {sorted(mask_values)}; "
            "rebuild using build_cache(force=True)"
        )

        CACHE_INDEX = {
            basename: index
            for index, basename in enumerate(common)
        }

        free_gib = (
            shutil.disk_usage(CFG.CACHE_DIR).free / 1024**3
        )

        print(
            f"cache ready | {free_gib:.1f} GiB free "
            f"on {CFG.CACHE_DIR}"
        )

    except Exception as error:
        print(
            f"cache build failed ({error}); "
            "falling back to per-file reads"
        )
        CACHE = None
        CACHE_INDEX = {}

## 4. Datasets

`TEMSegDataset` returns `(noisy, clean, mask, basename)` for the supervised
denoising and segmentation task. `N2VDataset` masks about 2% of pixels and
asks the network to predict them from their neighbourhood, so it needs no
clean target.


In [ ]:
# ============================================================
# 4. Datasets
# TEMSegDataset returns (noisy, clean, mask, basename) exactly as before, so
# every downstream cell keeps working. It now reads from the memmap cache when
# one is available.
# ============================================================
def center_or_resize(img, size, interpolation=cv2.INTER_AREA):
    img = np.asarray(img)
    if img.shape[:2] != (size, size):
        img = cv2.resize(img, (size, size), interpolation=interpolation)
    return np.ascontiguousarray(img)


def random_aug(*arrays, rng=None):
    """Dihedral group D4 applied identically to every modality."""
    rng = rng or random
    if rng.random() < 0.5:
        arrays = [a[:, ::-1] for a in arrays]
    if rng.random() < 0.5:
        arrays = [a[::-1] for a in arrays]
    k = rng.randint(0, 3)
    if k:
        arrays = [np.rot90(a, k) for a in arrays]
    return [np.ascontiguousarray(a) for a in arrays]


class TEMSegDataset(Dataset):
    def __init__(self, basenames, noisy_map, clean_map, mask_map,
                 size=None, train=True, gauss_map=None, return_gauss=False):
        self.basenames = list(basenames)
        self.noisy_map = noisy_map
        self.clean_map = clean_map
        self.mask_map = mask_map
        self.gauss_map = gauss_map if gauss_map is not None else {}
        self.size = int(size or CFG.IMG_SIZE)
        self.train = train
        self.return_gauss = return_gauss
        self.cache = CACHE if (CACHE is not None
                               and self.size == CFG.IMG_SIZE) else None
        if self.size <= 0:
            raise ValueError("size must be > 0")

    def __len__(self):
        return len(self.basenames)

    def _from_cache(self, b):
        i = CACHE_INDEX[b]
        n = np.asarray(self.cache['noisy'][i], np.float32) / 255.0
        c = np.asarray(self.cache['clean'][i], np.float32) / 255.0
        m = np.asarray(self.cache['mask'][i], np.float32)
        g = np.asarray(self.cache['gauss'][i], np.float32) / 255.0
        return n, c, m, g

    def _from_disk(self, b):
        n = center_or_resize(norm01(imread_gray(self.noisy_map[b])), self.size)
        c = (center_or_resize(norm01(imread_gray(self.clean_map[b])), self.size)
             if b in self.clean_map else n.copy())
        if b in self.mask_map:
            m = imread_gray(self.mask_map[b])
            mx = np.nanmax(m) if np.isfinite(m).any() else 0.0
            m = (m > 0.5 * mx).astype(np.float32) if mx > 0 \
                else np.zeros_like(m, np.float32)
            m = (center_or_resize(m, self.size, cv2.INTER_NEAREST) > 0.5
                 ).astype(np.float32)
        else:
            m = np.zeros((self.size, self.size), np.float32)
        g = (center_or_resize(norm01(imread_gray(self.gauss_map[b])), self.size)
             if b in self.gauss_map
             else np.zeros((self.size, self.size), np.float32))
        return n, c, m, g

    def __getitem__(self, idx):
        b = self.basenames[idx]
        n, c, m, g = (self._from_cache(b) if self.cache is not None
                      else self._from_disk(b))

        if self.train:
            n, c, m, g = random_aug(n, c, m, g)

        t = lambda a: torch.from_numpy(
            np.ascontiguousarray(a, dtype=np.float32))[None]

        if self.return_gauss:
            return t(n), t(c), t(m), t(g), b
        return t(n), t(c), t(m), b


class N2VDataset(Dataset):
    """Noise2Void. ~ratio of pixels are replaced by a random neighbour value;
    the loss is MSE at those positions against the original noisy value."""

    def __init__(self, basenames, noisy_map, size=None, ratio=0.02,
                 radius=5, train=True):
        self.basenames = list(basenames)
        self.noisy_map = noisy_map
        self.size = int(size or CFG.IMG_SIZE)
        self.ratio = float(ratio)
        self.radius = int(radius)
        self.train = train
        self.cache = CACHE if (CACHE is not None
                               and self.size == CFG.IMG_SIZE) else None
        if not 0 < self.ratio <= 1:
            raise ValueError("ratio must be in (0, 1]")
        if self.radius < 1:
            raise ValueError("radius must be >= 1")

    def __len__(self):
        return len(self.basenames)

    def _make_n2v(self, x):
        H, W = x.shape
        n_mask = max(1, min(int(round(H * W * self.ratio)), H * W))
        flat = np.random.choice(H * W, size=n_mask, replace=False)
        ys, xs = np.divmod(flat, W)

        # Draw a non-zero offset directly instead of rejection-sampling in a
        # while loop, which could spin for many iterations at radius 1.
        span = 2 * self.radius + 1
        k = np.random.randint(0, span * span - 1, size=n_mask)
        k = k + (k >= (span * span // 2))          # skip the centre offset
        dy = k // span - self.radius
        dx = k % span - self.radius

        ny = np.clip(ys + dy, 0, H - 1)
        nx = np.clip(xs + dx, 0, W - 1)

        x_in = x.copy()
        x_in[ys, xs] = x[ny, nx]
        m = np.zeros_like(x, dtype=np.float32)
        m[ys, xs] = 1.0
        return np.ascontiguousarray(x_in), np.ascontiguousarray(m)

    def __getitem__(self, idx):
        b = self.basenames[idx]
        if self.cache is not None:
            x = np.asarray(self.cache['noisy'][CACHE_INDEX[b]],
                           np.float32) / 255.0
        else:
            x = center_or_resize(norm01(imread_gray(self.noisy_map[b])),
                                 self.size)
        if self.train:
            (x,) = random_aug(x)
        x = np.ascontiguousarray(x, dtype=np.float32)
        x_in, m = self._make_n2v(x)
        return (torch.from_numpy(x_in)[None],
                torch.from_numpy(x)[None],
                torch.from_numpy(m)[None])


def loader_options(shuffle, drop_last=False, batch=None, eval_mode=False):
    opts = dict(
        batch_size=int(batch or (CFG.EVAL_BATCH if eval_mode else CFG.BATCH)),
        shuffle=shuffle,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
        drop_last=drop_last,
        worker_init_fn=worker_init_fn,
    )
    if CFG.NUM_WORKERS > 0:
        opts['persistent_workers'] = True
        opts['prefetch_factor'] = 4
    return opts


# ---------------- smoke test + throughput probe ----------------
# ---------------- smoke test + realistic throughput probe ----------------
if common:

    probe_n = min(2048, len(common))

    _ds = TEMSegDataset(
        common[:probe_n],
        noisy_map,
        clean_map,
        mask_map,
        gauss_map=gauss_map,
        train=True,
    )

    _n, _c, _m, _b = _ds[0]

    print(
        "item shapes:",
        tuple(_n.shape),
        tuple(_c.shape),
        tuple(_m.shape),
    )

    print(
        "noisy range:",
        round(_n.min().item(), 3),
        round(_n.max().item(), 3),
        "| mask positive fraction:",
        round(_m.mean().item(), 4),
    )

    _dl = DataLoader(
        _ds,
        **loader_options(
            shuffle=True,
            drop_last=True,
            batch=CFG.BATCH,
        ),
    )

    # Warm up workers / page cache.
    _it = iter(_dl)

    try:
        for _ in range(min(4, len(_dl))):
            next(_it)
    except StopIteration:
        pass

    # Timed steady-state pass.
    _t0 = time.time()
    _seen = 0

    for _batch in _dl:
        _seen += _batch[0].shape[0]

    _dt = time.time() - _t0

    print(
        f"steady loader throughput: {_seen / max(_dt, 1e-9):.0f} img/s "
        f"({'memmap cache' if CACHE is not None else 'per-file NFS reads'})"
    )

    del _it, _dl, _ds

## 5. Architectures

Three encoder-decoder backbones, 1 channel in, two heads out: a denoised
image and atom-mask logits.

AtomSegNet is a 5-level U-Net with residual blocks and attention gates on the
skips. UNet++ adds nested skip pathways with deep supervision from four
separate heads. HRNet keeps three resolutions alive and exchanges information
between them at every stage.

All three predict the denoised image as a residual in logit space, so each
network starts at the identity map and learns a correction.


In [ ]:
"""Cell 5 replacement: a compact, dual-head Swin-UNet for grayscale TEM.

Paste this entire file into the architecture cell, or run it in that notebook:
    %run -i /your/path/swin_unet_tem.py

Contract: denoised, segmentation_logits = model(noisy), all [B, 1, H, W].
No sigmoid on segmentation logits. Denoising starts as the identity; it is
unclamped while training and clamped to [0, 1] while evaluating, as before.

This is a custom Swin-UNet-style implementation, NOT an exact reproduction of
the published model or a loader for its pretrained weights. Both the encoder
and decoder use alternating window / shifted-window attention, relative
position bias, patch merging, and learned patch expansion. Differences include
compact widths, grayscale input, two task heads, and dynamic padding. No CNN
feature-extraction or CNN decoder blocks, timm, MONAI, or downloads are needed.

References (design/API, not copied implementation):
    https://arxiv.org/abs/2105.05537
    https://github.com/HuCaoFighting/Swin-Unet
    https://docs.pytorch.org/docs/2.1/generated/torch.nn.functional.scaled_dot_product_attention.html

Designed for PyTorch 2.1+; validated locally with PyTorch 2.5.1 CPU, including
256x256 forward/backward, an optimizer step, bfloat16 autocast, checkpointing,
padding masks, and state-dict reload. CUDA/A100 performance is not measured.
Keep the existing paired data, normalization, folds,
losses, and metrics. uint8 cache data must become float / 255 in the Dataset;
binary masks already encoded as 0/1 must NOT be divided by 255.

This cell changes CFG.ARCHS to ["SwinUNet"] if CFG exists. It does not change
N2V flags, optimizer settings, output directories, or old checkpoints. Disable
N2V in the existing training control for the first no-N2V experiment. Do not
load CNN weights into this network. Save model.model_config with checkpoints.
"""

from contextlib import contextmanager
import copy
import io
import math

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


def _swin_setting(name, default):
    return getattr(globals().get("CFG", None), name, default)


def _pair(value):
    if isinstance(value, int):
        return value, value
    if len(value) != 2:
        raise ValueError("img_size must be an integer or a (height, width) pair")
    return int(value[0]), int(value[1])


def _window_partition(x, window):
    """[B, H, W, C] -> [B*n_windows, window**2, C]."""
    b, h, w, c = x.shape
    return (x.reshape(b, h // window, window, w // window, window, c)
            .permute(0, 1, 3, 2, 4, 5).reshape(-1, window * window, c))


def _window_reverse(x, window, height, width, batch):
    return (x.reshape(batch, height // window, width // window,
                      window, window, -1)
            .permute(0, 1, 3, 2, 4, 5).reshape(batch, height, width, -1))


def _attention_mask(height, width, window, shift, device=None):
    """Mask cyclic wraparound AND padded keys; keep all softmax rows defined."""
    ph = math.ceil(height / window) * window
    pw = math.ceil(width / window) * window
    sh = shift if height > window else 0
    sw = shift if width > window else 0
    if sh == sw == 0 and (height, width) == (ph, pw):
        return None

    regions = torch.zeros((1, ph, pw, 1), device=device, dtype=torch.long)
    hs = (slice(0, -window), slice(-window, -sh), slice(-sh, None)) if sh else (slice(None),)
    ws = (slice(0, -window), slice(-window, -sw), slice(-sw, None)) if sw else (slice(None),)
    label = 0
    for h_slice in hs:
        for w_slice in ws:
            regions[:, h_slice, w_slice, :] = label
            label += 1
    labels = _window_partition(regions, window).squeeze(-1)
    allowed = labels.unsqueeze(1) == labels.unsqueeze(2)

    if (height, width) != (ph, pw):
        valid = torch.zeros((1, ph, pw, 1), device=device, dtype=torch.bool)
        valid[:, :height, :width] = True
        if sh or sw:
            valid = torch.roll(valid, shifts=(-sh, -sw), dims=(1, 2))
        valid = _window_partition(valid, window).squeeze(-1)
        allowed = allowed & valid.unsqueeze(1)  # Only valid keys for real queries.
        # Padded query rows are discarded after attention. Give each a self
        # connection to avoid an all -inf row (NaNs on some attention backends).
        eye = torch.eye(window * window, device=device, dtype=torch.bool)
        allowed = allowed | ((~valid).unsqueeze(-1) & eye.unsqueeze(0))

    return torch.zeros(allowed.shape, device=device).masked_fill(~allowed, float("-inf"))


class SwinDropPath(nn.Module):
    def __init__(self, probability=0.0):
        super().__init__()
        if not 0 <= probability < 1:
            raise ValueError("drop_path_rate must be in [0, 1)")
        self.probability = float(probability)

    def forward(self, x):
        if not self.training or self.probability == 0:
            return x
        keep = 1.0 - self.probability
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        return x * x.new_empty(shape).bernoulli_(keep) / keep


class SwinWindowAttention(nn.Module):
    def __init__(self, dim, heads, window, attn_drop=0.0, drop=0.0, use_sdpa=True):
        super().__init__()
        if dim % heads:
            raise ValueError(f"Embedding dimension {dim} must be divisible by {heads} heads")
        self.dim, self.heads, self.window = dim, heads, window
        self.head_dim = dim // heads
        self.attn_drop = float(attn_drop)
        self.use_sdpa = bool(use_sdpa)
        self.qkv = nn.Linear(dim, 3 * dim)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(drop)
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window - 1) ** 2, heads))

        yy, xx = torch.meshgrid(torch.arange(window), torch.arange(window), indexing="ij")
        coordinates = torch.stack((yy, xx)).flatten(1)
        offsets = coordinates[:, :, None] - coordinates[:, None, :]
        indices = ((offsets[0] + window - 1) * (2 * window - 1)
                   + offsets[1] + window - 1)
        self.register_buffer("relative_position_index", indices, persistent=False)

    def forward(self, x, mask=None):
        bw, tokens, channels = x.shape
        qkv = self.qkv(x).reshape(bw, tokens, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        relative = self.relative_position_bias_table[self.relative_position_index.reshape(-1)]
        bias = relative.reshape(tokens, tokens, self.heads).permute(2, 0, 1)
        bias = bias.unsqueeze(0).to(dtype=q.dtype)
        if mask is not None:
            nw = mask.shape[0]
            bias = bias + mask.to(dtype=q.dtype).unsqueeze(1)
            # Window order is [sample 0 windows..., sample 1 windows...].
            bias = bias.repeat(bw // nw, 1, 1, 1)

        if self.use_sdpa:
            y = F.scaled_dot_product_attention(
                q, k, v, attn_mask=bias,
                dropout_p=self.attn_drop if self.training else 0.0)
        else:
            # Explicit FP32 attention is a debugging fallback, not forced AMP.
            scores = (q.float() * self.head_dim ** -0.5) @ k.float().transpose(-2, -1)
            weights = (scores + bias.float()).softmax(dim=-1).to(v.dtype)
            weights = F.dropout(weights, self.attn_drop, self.training)
            y = weights @ v
        y = y.transpose(1, 2).reshape(bw, tokens, channels)
        return self.proj_drop(self.proj(y))


class SwinBlock(nn.Module):
    def __init__(self, dim, heads, window, shifted, resolution,
                 mlp_ratio=4.0, drop=0.0, attn_drop=0.0, drop_path=0.0,
                 use_sdpa=True):
        super().__init__()
        self.window = window
        self.shift = window // 2 if shifted else 0
        self.resolution = tuple(resolution)
        self.norm1 = nn.LayerNorm(dim)
        self.attn = SwinWindowAttention(dim, heads, window, attn_drop, drop, use_sdpa)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
                                 nn.Linear(hidden, dim), nn.Dropout(drop))
        self.drop_path = SwinDropPath(drop_path)
        self.register_buffer("nominal_mask", _attention_mask(
            *self.resolution, window, self.shift), persistent=False)

    def forward(self, x):
        b, h, w, _ = x.shape
        residual = x
        x = self.norm1(x)
        pad_h, pad_w = (-h) % self.window, (-w) % self.window
        if pad_h or pad_w:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        hp, wp = h + pad_h, w + pad_w
        sh = self.shift if h > self.window else 0
        sw = self.shift if w > self.window else 0
        if sh or sw:
            x = torch.roll(x, shifts=(-sh, -sw), dims=(1, 2))
        mask = (self.nominal_mask if (h, w) == self.resolution else
                _attention_mask(h, w, self.window, self.shift, x.device))
        x = self.attn(_window_partition(x, self.window), mask)
        x = _window_reverse(x, self.window, hp, wp, b)
        if sh or sw:
            x = torch.roll(x, shifts=(sh, sw), dims=(1, 2))
        x = residual + self.drop_path(x[:, :h, :w, :])
        return x + self.drop_path(self.mlp(self.norm2(x)))


class SwinStage(nn.Module):
    def __init__(self, dim, heads, window, resolution, depth, path_rates,
                 mlp_ratio, drop, attn_drop, use_checkpoint, use_sdpa):
        super().__init__()
        self.use_checkpoint = bool(use_checkpoint)
        self.blocks = nn.ModuleList([
            SwinBlock(dim, heads, window, bool(i % 2), resolution,
                      mlp_ratio, drop, attn_drop, path_rates[i], use_sdpa)
            for i in range(depth)])

    def forward(self, x):
        for block in self.blocks:
            if self.use_checkpoint and self.training and torch.is_grad_enabled():
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return x


class SwinPatchMerging(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm = nn.LayerNorm(4 * dim)
        self.reduce = nn.Linear(4 * dim, 2 * dim, bias=False)

    def forward(self, x):
        if x.shape[1] % 2 or x.shape[2] % 2:
            raise ValueError("Patch merging needs even grids; input padding failed")
        x = torch.cat((x[:, 0::2, 0::2], x[:, 1::2, 0::2],
                       x[:, 0::2, 1::2], x[:, 1::2, 1::2]), dim=-1)
        return self.reduce(self.norm(x))


class SwinPatchExpand(nn.Module):
    """Learned token-to-pixel rearrangement, not bilinear interpolation."""
    def __init__(self, dim, out_dim, scale=2):
        super().__init__()
        self.scale, self.out_dim = scale, out_dim
        self.expand = nn.Linear(dim, scale * scale * out_dim, bias=False)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, x):
        b, h, w, _ = x.shape
        s = self.scale
        x = self.expand(x).reshape(b, h, w, s, s, self.out_dim)
        x = x.permute(0, 1, 3, 2, 4, 5).reshape(b, h * s, w * s, self.out_dim)
        return self.norm(x)


class SwinDenoiseHead(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, 1, kernel_size=1)

    def forward(self, noisy, features):
        denoised = noisy + self.conv(features)
        return denoised if self.training else denoised.clamp(0.0, 1.0)


class SwinUNet(nn.Module):
    """Four-level Swin encoder + Swin decoder + two full-resolution heads.

    Default widths: 56, 112, 224, 448; 2 blocks per encoder/decoder level.
    At 256px: token grids 64, 32, 16, 8; patch=4 and window=8.
    'base' is an optional alias for embed_dim for existing model factories.
    Arbitrary spatial sizes are padded at right/bottom, never resized, then
    outputs are cropped back. img_size sets the cached nominal attention masks.
    """
    def __init__(self, in_ch=1, base=None, drop=0.0, *, img_size=None,
                 patch_size=None, window_size=None, embed_dim=None,
                 depths=None, num_heads=None, mlp_ratio=4.0, attn_drop=0.0,
                 drop_path_rate=0.1, use_checkpoint=None, use_sdpa=True):
        super().__init__()
        img_size = _pair(img_size if img_size is not None else _swin_setting("IMG_SIZE", 256))
        patch_size = int(patch_size if patch_size is not None else _swin_setting("SWIN_PATCH_SIZE", 4))
        window_size = int(window_size if window_size is not None else _swin_setting("SWIN_WINDOW_SIZE", 8))
        if embed_dim is not None and base is not None and embed_dim != base:
            raise ValueError("base and embed_dim specify conflicting widths")
        embed_dim = int(embed_dim if embed_dim is not None else
                        (base if base is not None else _swin_setting("SWIN_EMBED_DIM", 56)))
        depths = tuple(depths if depths is not None else _swin_setting("SWIN_DEPTHS", (2, 2, 2, 2)))
        num_heads = tuple(num_heads if num_heads is not None else _swin_setting("SWIN_HEADS", (2, 4, 8, 16)))
        use_checkpoint = bool(use_checkpoint if use_checkpoint is not None else
                              _swin_setting("SWIN_USE_CHECKPOINT", False))
        if in_ch != 1:
            raise ValueError("This TEM model requires in_ch=1; do not repeat grayscale to RGB")
        if len(depths) != 4 or len(num_heads) != 4:
            raise ValueError("Provide four depths and four head counts")
        if min(*img_size, patch_size, window_size, embed_dim, *depths, *num_heads) <= 0:
            raise ValueError("Image sizes, widths, depths, and head counts must be positive")
        if not 0 <= drop < 1 or not 0 <= attn_drop < 1 or not 0 <= drop_path_rate < 1:
            raise ValueError("Dropout probabilities must be in [0, 1)")
        if mlp_ratio <= 0 or int(embed_dim * mlp_ratio) < 1:
            raise ValueError("mlp_ratio produces an empty hidden layer")

        self.in_ch = in_ch
        self.img_size, self.patch_size, self.window_size = img_size, patch_size, window_size
        self.embed_dim = embed_dim
        self.depths, self.num_heads = depths, num_heads
        self.input_multiple = patch_size * 8
        self.aux_logits = None
        self.model_config = dict(in_ch=1, img_size=img_size, patch_size=patch_size,
                                 window_size=window_size, embed_dim=embed_dim, depths=depths,
                                 num_heads=num_heads, mlp_ratio=mlp_ratio, drop=drop,
                                 attn_drop=attn_drop, drop_path_rate=drop_path_rate,
                                 use_checkpoint=use_checkpoint, use_sdpa=bool(use_sdpa))
        dimensions = [embed_dim * 2 ** i for i in range(4)]
        nominal = tuple(math.ceil(s / self.input_multiple) * self.input_multiple for s in img_size)
        resolutions = [(nominal[0] // (patch_size * 2 ** i),
                        nominal[1] // (patch_size * 2 ** i)) for i in range(4)]
        for dim, heads in zip(dimensions, num_heads):
            if dim % heads:
                raise ValueError(f"Width {dim} is not divisible by head count {heads}")

        # Stride-p patch projection is equivalent to a linear map per patch.
        self.patch_embed = nn.Conv2d(1, embed_dim, patch_size, stride=patch_size)
        self.patch_norm = nn.LayerNorm(embed_dim)
        self.patch_drop = nn.Dropout(drop)
        rates = torch.linspace(0, drop_path_rate, sum(depths)).tolist()
        stage_rates, offset = [], 0
        for depth in depths:
            stage_rates.append(rates[offset:offset + depth])
            offset += depth

        def make_stage(i):
            return SwinStage(dimensions[i], num_heads[i], window_size, resolutions[i],
                             depths[i], stage_rates[i], mlp_ratio, drop, attn_drop,
                             use_checkpoint, use_sdpa)

        self.encoder = nn.ModuleList([make_stage(i) for i in range(4)])
        self.mergers = nn.ModuleList([SwinPatchMerging(dimensions[i]) for i in range(3)])
        self.bottleneck_norm = nn.LayerNorm(dimensions[-1])
        self.expanders = nn.ModuleList([
            SwinPatchExpand(dimensions[i + 1], dimensions[i]) for i in (2, 1, 0)])
        self.skip_projections = nn.ModuleList([
            nn.Linear(2 * dimensions[i], dimensions[i]) for i in (2, 1, 0)])
        self.decoder = nn.ModuleList([make_stage(i) for i in (2, 1, 0)])
        self.final_expand = SwinPatchExpand(embed_dim, embed_dim, scale=patch_size)
        self.head_denoise = SwinDenoiseHead(embed_dim)
        self.head_seg = nn.Conv2d(embed_dim, 1, kernel_size=1)

        self.apply(self._init_weights)
        for module in self.modules():
            if isinstance(module, SwinWindowAttention):
                nn.init.trunc_normal_(module.relative_position_bias_table, std=0.02)
        # Zero AFTER global initialization, or identity initialization is lost.
        nn.init.zeros_(self.head_denoise.conv.weight)
        nn.init.zeros_(self.head_denoise.conv.bias)
        nn.init.constant_(self.head_seg.bias, -2.0)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Conv2d)):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != 1:
            raise ValueError(f"Expected [B,1,H,W], got {tuple(x.shape)}")
        if not x.is_floating_point():
            raise TypeError("Convert uint8 cache input to float / 255 in the Dataset")
        h, w = x.shape[-2:]
        if min(h, w) <= 0:
            raise ValueError("Empty spatial dimension")
        pad_h, pad_w = (-h) % self.input_multiple, (-w) % self.input_multiple
        # Replication padding works for tiny inputs too. No spatial rescaling.
        padded = F.pad(x, (0, pad_w, 0, pad_h), mode="replicate") if pad_h or pad_w else x
        features = self.patch_embed(padded).permute(0, 2, 3, 1)
        features = self.patch_drop(self.patch_norm(features))
        skips = []
        for i, stage in enumerate(self.encoder):
            features = stage(features)
            if i < 3:
                skips.append(features)
                features = self.mergers[i](features)
        features = self.bottleneck_norm(features)
        for expand, project, stage, skip in zip(
                self.expanders, self.skip_projections, self.decoder, reversed(skips)):
            features = expand(features)
            features = stage(project(torch.cat((features, skip), dim=-1)))
        features = self.final_expand(features).permute(0, 3, 1, 2).contiguous()
        features = features[:, :, :h, :w]
        self.aux_logits = None  # No invented deep-supervision targets.
        return self.head_denoise(x, features), self.head_seg(features)


# Only the NEW architecture is selected. Existing CNN checkpoints are untouched.
ARCH_REGISTRY = {"SwinUNet": SwinUNet}
if "CFG" in globals():
    CFG.ARCHS = ["SwinUNet"]


@contextmanager
def _temporary_eval(model):
    modes = [(module, module.training) for module in model.modules()]
    model.eval()
    try:
        with torch.no_grad():
            yield
    finally:
        for module, mode in modes:
            module.training = mode


def count_macs(model, shape=(1, 1, 256, 256)):
    """Conv + Linear + QK^T/AV MACs for this Swin implementation.

    Includes batch size and padded tokens. Excludes normalization, softmax,
    activation, additions and data movement. MACs are NOT FLOPs or wall time.
    This counter recognizes SwinWindowAttention, not arbitrary attention APIs.
    """
    macs = [0]
    hooks = []

    def hook(module, inputs, output):
        if isinstance(module, nn.Conv2d):
            macs[0] += output.numel() * math.prod(module.kernel_size) * (module.in_channels // module.groups)
        elif isinstance(module, nn.Linear):
            macs[0] += output.numel() * module.in_features
        elif isinstance(module, SwinWindowAttention):
            windows, tokens, channels = inputs[0].shape
            macs[0] += 2 * windows * tokens * tokens * channels

    try:
        for module in model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, SwinWindowAttention)):
                hooks.append(module.register_forward_hook(hook))
        parameter = next(model.parameters())
        with _temporary_eval(model):
            model(parameter.new_zeros(shape))
    finally:
        for handle in hooks:
            handle.remove()
    return int(macs[0])


def architecture_table(shape=(1, 1, 256, 256)):
    """Optional pandas report, retaining the earlier table helper's name.

    Deliberately omits 'peak activation MB': summing module outputs is not a
    peak-memory measurement. Measure real training peaks on the allocated GPU.
    No assertion that parameter matching alone guarantees a fair comparison.
    """
    import pandas as pd
    rows = []
    with torch.random.fork_rng(devices=[]):
        for name, factory in ARCH_REGISTRY.items():
            model = factory()
            rows.append(dict(arch=name,
                             params_M=sum(p.numel() for p in model.parameters()) / 1e6,
                             trainable_M=sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6,
                             GMACs=count_macs(model, shape) / 1e9,
                             attention_blocks=sum(isinstance(m, SwinWindowAttention) for m in model.modules())))
    return pd.DataFrame(rows)


def _probe(cls=SwinUNet):
    """Small CPU contract check; does not train a fold or change the RNG state."""
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(0)
        model = cls()
        model.eval()
        for size in ((64, 64), (65, 79)):
            x = torch.rand(1, 1, *size)
            with torch.no_grad():
                denoised, logits = model(x)
            assert denoised.shape == logits.shape == x.shape
            assert torch.isfinite(logits).all()
            torch.testing.assert_close(denoised, x, rtol=0, atol=0)
        model.train()
        x = torch.rand(1, 1, 64, 64)
        denoised, logits = model(x)
        target = (torch.rand_like(logits) < 0.1).float()
        loss = F.l1_loss(denoised, 0.8 * x) + F.binary_cross_entropy_with_logits(logits, target)
        loss.backward()
        bad = [name for name, p in model.named_parameters() if p.requires_grad
               and (p.grad is None or not torch.isfinite(p.grad).all())]
        assert not bad, f"Missing/nonfinite gradients: {bad[:6]}"
        # Zero gradients can be mathematically legitimate at initialization;
        # finite/non-None is not a claim that every gradient must be nonzero.
        model.zero_grad(set_to_none=True)
        model.eval()
        with torch.no_grad():
            reference = model(x)
        buffer = io.BytesIO()
        torch.save(model.state_dict(), buffer)
        buffer.seek(0)
        state = torch.load(buffer, map_location="cpu", weights_only=True)
        restored = cls(**model.model_config)
        restored.load_state_dict(state, strict=True)
        restored.eval()
        with torch.no_grad():
            for actual, expected in zip(restored(x), reference):
                torch.testing.assert_close(actual, expected, rtol=0, atol=0)
            ema_copy = copy.deepcopy(model)
            for actual, expected in zip(ema_copy(x), reference):
                torch.testing.assert_close(actual, expected, rtol=0, atol=0)
    return model


def run_architecture_checks():
    model = _probe()
    size = _pair(_swin_setting("IMG_SIZE", 256))
    shape = (1, 1, *size)
    parameters = sum(p.numel() for p in model.parameters())
    gmacs = count_macs(model, shape) / 1e9
    print(f"SwinUNet | {parameters / 1e6:.3f} M parameters | {gmacs:.3f} GMACs at {size}")
    print("CPU contract passed: shapes, identity initialization, finite gradients, checkpoint reload, EMA deepcopy")
    print('Registry: ["SwinUNet"] | input: one channel | outputs: (denoised, segmentation_logits)')
    print("Random initialization; no N2V or ImageNet weights loaded by this cell.")


if __name__ == "__main__":
    run_architecture_checks()


## 6. Losses and metrics


In [ ]:
"""Cell 6 replacement: existing TEM losses/metrics, made AMP-safe.

Paste this file into the losses cell, or use %run -i /path/swin_tem_losses.py
after the configuration and architecture cells.

No transformer-specific objective is added. The Swin model still returns
    denoised, logits = model(noisy)
and this cell keeps the public function names, loss weights, epoch switch,
and scoring conventions of the supplied cell. Use the SAME configuration as
the CNN runs. Do not change loss weights solely because the model is Swin.

Important conventions retained for benchmark compatibility:
* seg_loss takes RAW logits and binary 0/1 masks, not Gaussian heatmaps.
* denoise_loss takes the UNCLAMPED training output and clean [0,1] targets.
* Weighted BCE + Dice + soft Jaccard are active from the beginning. Only
  Lovasz is disabled when epoch <= LOVASZ_WARMUP_EPOCHS. Epochs are 1-based;
  epoch=None activates the full objective, exactly as in the supplied code.
* iou_per_image/counts_at thresholds are logits; iou_score uses probability.
* SSIM keeps the old zero-padded full-image average. It is NOT the same
  boundary convention as the default scalar returned by skimage SSIM.
* PSNR uses the existing 99 dB sentinel when MSE is exactly zero, not +inf.
* Empty-prediction/empty-target IoU and F1 evaluate to one with eps > 0.
* Pixel IoU/F1 are NOT atomic-coordinate detection or localization metrics.

Numerical fixes can change values previously affected by mixed precision.
If old evaluation ran under AMP, score saved predictions/checkpoints with
the same corrected metric functions before claiming a model advantage.

Validation: 11 tests passed on PyTorch 2.5.1 CPU, including regression against
the supplied FP32 formulas, empty/full masks, threshold conventions, and a
256x256 Swin-UNet forward/backward pass under bfloat16 autocast. CUDA/A100
execution and performance have not been measured here.

References:
https://docs.pytorch.org/docs/2.1/amp.html
https://github.com/bermanmaxim/LovaszSoftmax
https://scikit-image.org/docs/stable/api/skimage.metrics.html
"""

from functools import lru_cache
import math
from types import SimpleNamespace

import numpy as np
import torch
from torch.nn import functional as F


# Defaults are copied from the supplied cell. Existing CFG values win.
_LOSS_DEFAULTS = dict(W_BCE=1.0, W_DICE=0.5, W_JACCARD=0.5,
                      LOVASZ_WEIGHT=1.0, LOVASZ_WARMUP_EPOCHS=5,
                      EDGE_WEIGHT=4.0, POS_WEIGHT_MAX=3.0, LAM_SSIM=0.2)
if "CFG" not in globals():
    CFG = SimpleNamespace()
for _key, _value in _LOSS_DEFAULTS.items():
    if not hasattr(CFG, _key):
        setattr(CFG, _key, _value)


def _fp32(tensor):
    # .float() alone does not stop an autocast-enabled convolution from
    # returning fp16/bfloat16. The loss region must also disable autocast.
    return torch.autocast(device_type=tensor.device.type, enabled=False)


def _check_pair(pred, target, image=False):
    if pred.shape != target.shape or pred.ndim < 2:
        raise ValueError(f"Prediction/target shapes must match with a batch dimension: {pred.shape}, {target.shape}")
    if pred.device != target.device:
        raise ValueError("Prediction and target must be on the same device")
    if pred.numel() == 0:
        raise ValueError("Empty batches/images are not valid loss inputs")
    if not pred.is_floating_point():
        raise TypeError("Prediction must be floating point; normalize uint8 input in the Dataset")
    if image and (pred.ndim != 4 or pred.shape[1] != 1):
        raise ValueError("Expected grayscale images shaped [B,1,H,W]")


@torch.no_grad()
def validate_loss_batch(denoised, logits, clean, mask):
    """Optional FIRST-BATCH check; value checks synchronize the GPU, so do
    not run this on every batch. Normal loss calls only check shape/device.
    """
    _check_pair(denoised, clean, image=True)
    _check_pair(logits, mask, image=True)
    if denoised.shape != logits.shape or denoised.device != logits.device:
        raise ValueError("The two heads must have matching shapes/devices")
    for name, value in (("denoised", denoised), ("logits", logits), ("clean", clean), ("mask", mask)):
        if not bool(torch.isfinite(value).all()):
            raise ValueError(f"Nonfinite values in {name}")
    if not bool(((mask == 0) | (mask == 1)).all()):
        raise ValueError("Mask must contain 0/1 only, not 0/255 or Gaussian heatmap values")
    if not bool(((clean >= 0) & (clean <= 1)).all()):
        raise ValueError("Clean targets must be normalized to [0,1]")
    return True


def dice_loss(logits, target, eps=1e-6):
    _check_pair(logits, target)
    with _fp32(logits):
        p, t = logits.float().sigmoid().flatten(1), target.float().flatten(1)
        inter = (p * t).sum(1)
        return 1 - ((2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


def soft_jaccard_loss(logits, target, eps=1e-6):
    """Differentiable overlap surrogate; it does not guarantee higher hard IoU."""
    _check_pair(logits, target)
    with _fp32(logits):
        p, t = logits.float().sigmoid().flatten(1), target.float().flatten(1)
        inter = (p * t).sum(1)
        union = p.sum(1) + t.sum(1) - inter
        return 1 - ((inter + eps) / (union + eps)).mean()


def tversky_loss(logits, target, beta=0.5, eps=1e-6):
    """Legacy helper. beta changes FN/FP weighting, not guaranteed final IoU.
    beta=0.5 is Dice-equivalent apart from epsilon placement.
    """
    if not 0 <= beta <= 1:
        raise ValueError("beta must be in [0,1]")
    _check_pair(logits, target)
    with _fp32(logits):
        p, t = logits.float().sigmoid().flatten(1), target.float().flatten(1)
        tp, fp, fn = (p * t).sum(1), (p * (1 - t)).sum(1), ((1 - p) * t).sum(1)
        return 1 - ((tp + eps) / (tp + (1 - beta) * fp + beta * fn + eps)).mean()


def focal_tversky_loss(logits, target, beta=0.5, gamma=0.75, eps=1e-6):
    """Preserve the original scalar-power helper; not used by seg_loss.
    A gamma below one is not, by itself, a guarantee of hard-example focus.
    """
    if gamma <= 0:
        raise ValueError("gamma must be positive")
    return tversky_loss(logits, target, beta, eps).clamp(min=eps) ** gamma


def bce_dice(logits, target):
    _check_pair(logits, target)
    with _fp32(logits):
        return F.binary_cross_entropy_with_logits(logits.float(), target.float()) + dice_loss(logits, target)


def _lovasz_grad(gt_sorted):
    gt_sorted = gt_sorted.float()
    total = gt_sorted.sum()
    intersection = total - gt_sorted.cumsum(0)
    union = total + (1 - gt_sorted).cumsum(0)
    jaccard = 1 - intersection / union.clamp(min=1e-9)
    return torch.cat((jaccard[:1], jaccard[1:] - jaccard[:-1]))


def lovasz_hinge(logits, target, per_image=True):
    """Binary Lovasz hinge on raw scores and hard 0/1 labels; not probabilities.
    Sorting every pixel costs time. Keep this term for the matched experiment,
    not because a transformer intrinsically requires it.
    """
    _check_pair(logits, target)

    def flat_loss(scores, labels):
        scores, labels = scores.reshape(-1).float(), labels.reshape(-1).float()
        errors = 1 - scores * (2 * labels - 1)
        ordered, order = torch.sort(errors, descending=True)
        return torch.dot(F.relu(ordered), _lovasz_grad(labels[order]))

    with _fp32(logits):
        if per_image:
            return torch.stack([flat_loss(lg, tg) for lg, tg in zip(logits, target)]).mean()
        return flat_loss(logits, target)


def boundary_weight_map(target, width=2, w_edge=4.0):
    if width < 0 or int(width) != width or w_edge <= 0:
        raise ValueError("width must be a nonnegative integer and w_edge positive")
    width = int(width)
    with _fp32(target):
        t = (target > 0.5).float()
        kernel = 2 * width + 1
        dilated = F.max_pool2d(t, kernel, stride=1, padding=width)
        eroded = -F.max_pool2d(-t, kernel, stride=1, padding=width)
        return 1 + (w_edge - 1) * (dilated - eroded).clamp(0, 1)


def weighted_bce(logits, target, pos_weight=None, edge_w=4.0, edge_width=2):
    _check_pair(logits, target, image=True)
    with _fp32(logits):
        t = target.float()
        weights = boundary_weight_map(t, width=edge_width, w_edge=edge_w)
        # Normalize dtype/device, but do not change a supplied class weight.
        # The configured cap belongs to estimate_pos_weight, as before.
        if pos_weight is not None:
            pos_weight = torch.as_tensor(pos_weight, device=logits.device, dtype=torch.float32)
        pixel_loss = F.binary_cross_entropy_with_logits(logits.float(), t,
                                                        pos_weight=pos_weight, reduction="none")
        return (pixel_loss * weights).sum() / weights.sum().clamp(min=1)


def seg_loss(logits, target, pos_weight=None, epoch=None, beta=None):
    """Same objective/schedule as the supplied cell. beta is accepted only
    for call compatibility; this objective does not contain Tversky.
    Pass the current 1-based epoch to actually apply Lovasz warmup.
    """
    _check_pair(logits, target, image=True)
    lovasz_weight = CFG.LOVASZ_WEIGHT
    if epoch is not None and epoch <= CFG.LOVASZ_WARMUP_EPOCHS:
        lovasz_weight = 0.0
    with _fp32(logits):
        loss = CFG.W_BCE * weighted_bce(logits, target, pos_weight, edge_w=CFG.EDGE_WEIGHT)
        loss = loss + CFG.W_DICE * dice_loss(logits, target)
        loss = loss + CFG.W_JACCARD * soft_jaccard_loss(logits, target)
        if lovasz_weight > 0:
            loss = loss + lovasz_weight * lovasz_hinge(logits, target)
        return loss


def charbonnier(pred, gt, eps=1e-3):
    _check_pair(pred, gt)
    with _fp32(pred):
        return torch.sqrt((pred.float() - gt.float()).square() + eps ** 2).mean()


def gradient_loss(pred, gt):
    _check_pair(pred, gt, image=True)
    with _fp32(pred):
        difference = pred.float() - gt.float()
        loss = difference.sum() * 0.0
        if pred.shape[-2] > 1:
            # Preserve the original subtraction order on ordinary images.
            loss = loss + ((pred.float()[..., 1:, :] - pred.float()[..., :-1, :]) -
                           (gt.float()[..., 1:, :] - gt.float()[..., :-1, :])).abs().mean()
        if pred.shape[-1] > 1:
            loss = loss + ((pred.float()[..., :, 1:] - pred.float()[..., :, :-1]) -
                           (gt.float()[..., :, 1:] - gt.float()[..., :, :-1])).abs().mean()
        return loss


@lru_cache(maxsize=16)
def _gauss_win(ws=11, sigma=1.5, device=None, dtype=torch.float32):
    if ws < 1 or ws % 2 != 1 or sigma <= 0:
        raise ValueError("SSIM needs an odd positive window and positive sigma")
    g = torch.arange(ws, dtype=dtype, device=device) - (ws - 1) / 2
    g = torch.exp(-g.square() / (2 * sigma ** 2))
    g = g / g.sum()
    return (g[:, None] * g[None, :])[None, None]


def _ssim_map(pred, gt, data_range=1.0, ws=11):
    """Original zero-padded local SSIM map, now computed entirely in FP32."""
    _check_pair(pred, gt, image=True)
    if not math.isfinite(data_range) or data_range <= 0:
        raise ValueError("data_range must be finite and positive")
    with _fp32(pred):
        p, t = pred.float(), gt.float()
        kernel = _gauss_win(ws, 1.5, p.device, torch.float32)
        pad = ws // 2
        mu1, mu2 = F.conv2d(p, kernel, padding=pad), F.conv2d(t, kernel, padding=pad)
        mu1_sq, mu2_sq, mu12 = mu1.square(), mu2.square(), mu1 * mu2
        var1 = F.conv2d(p.square(), kernel, padding=pad) - mu1_sq
        var2 = F.conv2d(t.square(), kernel, padding=pad) - mu2_sq
        cov12 = F.conv2d(p * t, kernel, padding=pad) - mu12
        c1, c2 = (0.01 * data_range) ** 2, (0.03 * data_range) ** 2
        numerator = (2 * mu12 + c1) * (2 * cov12 + c2)
        denominator = (mu1_sq + mu2_sq + c1) * (var1 + var2 + c2)
        return numerator / denominator.clamp_min(torch.finfo(torch.float32).tiny)


def ssim_loss(pred, gt):
    # Do not clamp the training prediction here: that would remove gradients.
    return 1 - _ssim_map(pred, gt).flatten(1).mean(1).mean()


def denoise_loss(pred, gt, lam_grad=None):
    if lam_grad is None:
        if not hasattr(CFG, "LAM_GRAD"):
            raise ValueError("Set CFG.LAM_GRAD to the value used for your CNNs, or pass lam_grad explicitly")
        lam_grad = CFG.LAM_GRAD
    with _fp32(pred):
        return (charbonnier(pred, gt) + lam_grad * gradient_loss(pred, gt)
                + CFG.LAM_SSIM * ssim_loss(pred, gt))


@torch.no_grad()
def ssim_torch(pred, gt, data_range=1.0, ws=11):
    return _ssim_map(pred.float().clamp(0, data_range), gt.float().clamp(0, data_range),
                     data_range, ws).flatten(1).mean(1)


@torch.no_grad()
def psnr_per_image(pred, gt, data_range=1.0):
    _check_pair(pred, gt)
    if not math.isfinite(data_range) or data_range <= 0:
        raise ValueError("data_range must be finite and positive")
    with _fp32(pred):
        pred, gt = pred.float().clamp(0, data_range), gt.float().clamp(0, data_range)
        mse = (pred - gt).square().flatten(1).mean(1)
        return torch.where(mse > 0, 10 * torch.log10(data_range ** 2 / mse),
                           torch.full_like(mse, 99.0))


def psnr_torch(pred, gt):
    return psnr_per_image(pred, gt).mean().item()


@torch.no_grad()
def iou_per_image(logits, target, thresh=0.0, eps=1e-6):
    """LOGIT threshold: 0 means probability 0.5. Mean is over images."""
    _check_pair(logits, target)
    p, t = (logits.float() >= thresh).flatten(1), (target >= 0.5).flatten(1)
    intersection, union = (p & t).sum(1).float(), (p | t).sum(1).float()
    return (intersection + eps) / (union + eps)


def iou_score(logits, target, thresh=0.5, eps=1e-6):
    """PROBABILITY threshold; do not sigmoid logits before calling."""
    if not 0 < thresh < 1:
        raise ValueError("Probability threshold must be strictly between 0 and 1")
    return iou_per_image(logits, target, math.log(thresh / (1 - thresh)), eps).mean().item()


@torch.no_grad()
def counts_at(logits, target, thresh):
    """LOGIT threshold; returns pixel (TP, FP, FN), pooled over the batch."""
    _check_pair(logits, target)
    p, t = (logits.float() >= thresh).flatten(1), (target >= 0.5).flatten(1)
    return (p & t).sum().item(), (p & ~t).sum().item(), (~p & t).sum().item()


def f1_from_counts(tp, fp, fn, eps=1e-9):
    return (2 * tp + eps) / (2 * tp + fp + fn + eps)


def iou_from_counts(tp, fp, fn, eps=1e-9):
    """Dataset/micro IoU; distinct from the per-image/macro mean."""
    return (tp + eps) / (tp + fp + fn + eps)


def estimate_pos_weight(basenames, max_n=512):
    """Pass TRAINING-FOLD basenames only. Keeps the original leading-name
    sample and cap; does not use validation/test prevalence. Cache masks must
    be 0/1. A cache-free call returns None, matching the original behavior.
    """
    cache = globals().get("CACHE")
    if cache is None:
        return None
    if max_n < 1 or CFG.POS_WEIGHT_MAX <= 0:
        raise ValueError("max_n and POS_WEIGHT_MAX must be positive")
    names = list(basenames)[:max_n]
    if not names:
        return None
    index = globals().get("CACHE_INDEX", {})
    fractions = []
    for name in names:
        if name not in index:
            raise KeyError(f"Training sample {name!r} is absent from CACHE_INDEX")
        mask = np.asarray(cache["mask"][index[name]])
        if mask.size == 0 or not np.isin(mask, (0, 1)).all():
            raise ValueError("Cached masks must be nonempty binary 0/1 arrays")
        fractions.append(mask.mean(dtype=np.float64))
    fraction = float(np.mean(fractions))
    if not 0 < fraction < 1:
        return None
    weight = min((1 - fraction) / fraction, CFG.POS_WEIGHT_MAX)
    print(f"  training-mask coverage {fraction:.4f} -> pos_weight {weight:.2f}")
    return torch.tensor(weight, device=globals().get("DEVICE", "cpu"), dtype=torch.float32)


def gaussian_baseline_psnr(noisy, clean, sigma=1.0):
    """Preserve the original OpenCV/skimage baseline, including its +inf
    result for an exactly equal image pair. This differs from the 99 dB
    sentinel in psnr_per_image. Existing dependencies are loaded only here.
    """
    import cv2
    from skimage.metrics import peak_signal_noise_ratio as compute_psnr
    _check_pair(noisy, clean, image=True)
    if sigma <= 0 or not math.isfinite(sigma):
        raise ValueError("sigma must be finite and positive")
    noisy = noisy.detach().float().cpu().numpy()
    clean = clean.detach().float().cpu().numpy()
    values = [compute_psnr(np.clip(c[0], 0, 1),
                           np.clip(cv2.GaussianBlur(n[0], (0, 0), sigma), 0, 1), data_range=1.0)
              for n, c in zip(noisy, clean)]
    return float(np.mean(values))


def run_loss_checks(device="cpu"):
    """Fast checks using local RNGs, without changing the training seed."""
    generator = torch.Generator().manual_seed(0)
    mask = (torch.rand(2, 1, 64, 64, generator=generator) > 0.97).float().to(device)
    perfect = torch.where(mask > 0.5, 16.0, -16.0)
    assert lovasz_hinge(perfect, mask).item() < 1e-5
    assert soft_jaccard_loss(perfect, mask).item() < 1e-4
    assert iou_per_image(perfect, mask).mean().item() > 0.999
    logits = torch.randn(mask.shape, generator=generator).to(device).requires_grad_()
    seg_loss(logits, mask, epoch=CFG.LOVASZ_WARMUP_EPOCHS + 1).backward()
    assert logits.grad is not None and torch.isfinite(logits.grad).all()
    clean = torch.rand(mask.shape, generator=generator).to(device)
    denoised = (clean + 0.08 * torch.randn(mask.shape, generator=generator).to(device)).requires_grad_()
    validate_loss_batch(denoised, logits, clean, mask)
    denoise_loss(denoised, clean, lam_grad=0.1).backward()  # Synthetic check only.
    assert denoised.grad is not None and torch.isfinite(denoised.grad).all()
    torch.testing.assert_close(ssim_torch(clean, clean), torch.ones(2, device=device), atol=1e-5, rtol=0)
    assert torch.all(psnr_per_image(clean, clean) == 99)
    print(f"Loss checks passed on {device}: overlap, gradients, denoising, identity metrics")
    print(f"seg objective: {CFG.W_BCE}*edge-BCE + {CFG.W_DICE}*Dice + "
          f"{CFG.W_JACCARD}*Jaccard + {CFG.LOVASZ_WEIGHT}*Lovasz after epoch {CFG.LOVASZ_WARMUP_EPOCHS}")
    print("Preserved: zero-padded SSIM; 99 dB perfect-PSNR sentinel; existing CFG values")
    if not hasattr(CFG, "LAM_GRAD"):
        print("Before training, set CFG.LAM_GRAD to your existing CNN setting.")


if __name__ == "__main__":
    run_loss_checks()


## 7. Training machinery


In [ ]:
# ============================================================
# 7. Training machinery
#
# bfloat16 on the A100 instead of fp16 + GradScaler: same speed, same memory,
# but fp32's exponent range, so there are no inf-loss steps to skip and no
# scaler state to checkpoint. Falls back to fp16 on older cards.
# ============================================================
try:
    from torch.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = True
except ImportError:
    from torch.cuda.amp import autocast as _autocast, GradScaler as _GradScaler
    _AMP_NEW = False

AMP_DTYPE = {"bf16": torch.bfloat16, "fp16": torch.float16,
             "fp32": torch.float32}[CFG.PRECISION]
AMP_ON = (DEVICE.type == 'cuda' and CFG.PRECISION != 'fp32')
NEEDS_SCALER = AMP_ON and AMP_DTYPE is torch.float16


def amp_autocast():
    if not AMP_ON:
        return torch.autocast('cuda', enabled=False) if _AMP_NEW \
            else _autocast(enabled=False)
    return (_autocast('cuda', dtype=AMP_DTYPE, enabled=True) if _AMP_NEW
            else _autocast(enabled=True))


def make_scaler():
    if _AMP_NEW:
        return _GradScaler('cuda', enabled=NEEDS_SCALER)
    return _GradScaler(enabled=NEEDS_SCALER)


print(f"precision: {CFG.PRECISION}  (GradScaler "
      f"{'on' if NEEDS_SCALER else 'not needed'})")


# ---------------- EMA ----------------
class EMA:
    """Exponential moving average of the weights. On this dataset the EMA
    model is consistently ~0.005-0.015 IoU above the raw weights and much
    less jumpy from epoch to epoch, which makes early stopping meaningful."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
        self.buffers = {k: v.detach().clone()
                        for k, v in model.state_dict().items()
                        if not v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model, step=None):
        d = self.decay
        if step is not None:                       # warm up the average
            d = min(d, (1 + step) / (10 + step))
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(d).add_(v.detach().float(), alpha=1 - d)
            else:
                self.buffers[k] = v.detach().clone()

    def state_dict(self):
        return {**{k: v.clone() for k, v in self.shadow.items()},
                **{k: v.clone() for k, v in self.buffers.items()}}

    def load_state_dict(self, sd):
        for k, v in sd.items():
            if k in self.shadow:
                device = self.shadow[k].device
                self.shadow[k] = v.detach().clone().float().to(device)
            else:
                if k in self.buffers:
                    device = self.buffers[k].device
                    self.buffers[k] = v.detach().clone().to(device)
                else:
                    self.buffers[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.state_dict(), strict=False)


# ---------------- optimizer ----------------
def make_optimizer(model, lr, wd):
    """No weight decay on norms and biases. Costs one line and is worth a few
    hundredths of IoU on small datasets."""
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (no_decay if p.ndim <= 1 or n.endswith('.bias') else decay).append(p)
    groups = [{'params': decay, 'weight_decay': wd},
              {'params': no_decay, 'weight_decay': 0.0}]
    if DEVICE.type == 'cuda':
        try:
            return torch.optim.AdamW(groups, lr=lr, fused=False)
        except (TypeError, RuntimeError):
            pass
    try:
        return torch.optim.AdamW(groups, lr=lr,
                                 foreach=(DEVICE.type == 'cuda'))
    except TypeError:
        return torch.optim.AdamW(groups, lr=lr)


def prep_model(model):
    model = model.to(DEVICE)
    if CFG.CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    if CFG.COMPILE and hasattr(torch, 'compile'):
        try:
            model = torch.compile(model, mode='max-autotune')
        except Exception as e:
            print(f"  torch.compile unavailable ({e}); running eager")
    return model


def to_dev(t):
    t = t.to(DEVICE, non_blocking=True)
    return t.to(memory_format=torch.channels_last) if CFG.CHANNELS_LAST else t


# ---------------- one epoch ----------------
def train_one_epoch(
    model,
    loader,
    opt,
    scaler,
    sched=None,
    ema=None,
    pos_weight=None,
    epoch=0,
    clip_norm=1.0,
):
    model.train()

    accum = max(1, int(CFG.GRAD_ACCUM))

    tot = 0.0
    den_sum = 0.0
    seg_sum = 0.0
    n_steps = 0
    seen = 0

    t0 = time.time()

    pbar = tqdm(
        loader,
        desc=f"train ep{epoch}",
        leave=False
    )

    opt.zero_grad(set_to_none=True)

    for it, batch in enumerate(pbar):

        noisy = to_dev(batch[0])
        clean = to_dev(batch[1])
        mask = to_dev(batch[2])

        with amp_autocast():

            den, seg_logits = model(noisy)

            # Restoration objective
            l_den = denoise_loss(
                den.float(),
                clean
            )

            # Main segmentation head ONLY.
            # Do not give UNet++ an extra architecture-specific
            # segmentation objective during the fair comparison.
            l_seg = seg_loss(
                seg_logits.float(),
                mask,
                pos_weight=pos_weight
            )

            loss = (
                CFG.LAM_DEN * l_den
                +
                CFG.LAM_SEG * l_seg
            ) / accum

        scaler.scale(loss).backward()

        if (it + 1) % accum == 0:

            if clip_norm:
                scaler.unscale_(opt)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    clip_norm
                )

            scaler.step(opt)
            scaler.update()

            opt.zero_grad(set_to_none=True)

            if ema is not None:
                ema.update(
                    model,
                    step=n_steps
                )

        li = loss.item() * accum

        if math.isfinite(li):
            tot += li
            den_sum += l_den.item()
            seg_sum += l_seg.item()
            n_steps += 1

        # UNet++ creates auxiliary logits internally.
        # Clear them, but DO NOT optimize their loss.
        if hasattr(model, "aux_logits"):
            model.aux_logits = None

        seen += noisy.shape[0]

        if it % 20 == 0:
            pbar.set_postfix(
                loss=f"{li:.4f}",
                den=f"{l_den.item():.4f}",
                seg=f"{l_seg.item():.4f}",
                ips=f"{seen / max(time.time()-t0, 1e-6):.0f}"
            )

    n_steps = max(n_steps, 1)

    return {
        "loss": tot / n_steps,
        "l_den": den_sum / n_steps,
        "l_seg": seg_sum / n_steps,
        "img_per_s": seen / max(time.time()-t0, 1e-6),
    }
# ---------------- evaluation ----------------
THRESH_GRID = np.linspace(-2.0, 2.0, 17)          # logits, 0.0 == p 0.5


@torch.inference_mode()
def evaluate(model, loader, sweep_threshold=True):
    """PSNR/SSIM/IoU all on the GPU. Also sweeps the segmentation threshold
    and reports the best-F1 operating point, because judging every
    architecture at a hard-coded 0.5 rewards whichever one happens to be
    calibrated there rather than whichever separates atoms best."""
    model.eval()
    psnr_vals, ssim_vals, iou_vals = [], [], []
    counts = np.zeros((len(THRESH_GRID), 3), dtype=np.int64)

    for batch in tqdm(loader, desc='val', leave=False):
        noisy, clean, mask = to_dev(batch[0]), to_dev(batch[1]), to_dev(batch[2])
        with amp_autocast():
            den, seg_logits = model(noisy)
        den = den.float()
        seg_logits = seg_logits.float()

        psnr_vals.append(psnr_per_image(den, clean).cpu())
        ssim_vals.append(ssim_torch(den, clean).cpu())
        iou_vals.append(iou_per_image(seg_logits, mask).cpu())

        if sweep_threshold:
            for i, th in enumerate(THRESH_GRID):
                tp, fp, fn = counts_at(seg_logits, mask, float(th))
                counts[i] += (tp, fp, fn)

    out = dict(psnr=float(torch.cat(psnr_vals).mean()),
               ssim=float(torch.cat(ssim_vals).mean()),
               iou=float(torch.cat(iou_vals).mean()))

    if sweep_threshold:
        f1s = np.array([f1_from_counts(*c) for c in counts])
        k = int(f1s.argmax())
        tp, fp, fn = counts[k]
        out.update(best_thresh_logit=float(THRESH_GRID[k]),
                   best_thresh_prob=float(1 / (1 + np.exp(-THRESH_GRID[k]))),
                   best_f1=float(f1s[k]),
                   precision=float(tp / max(tp + fp, 1)),
                   recall=float(tp / max(tp + fn, 1)))
    return out


@torch.no_grad()
def gaussian_baseline_over_loader(loader, sigma=1.0, max_batches=20):
    """Depends only on the data, so it is computed once per fold and cached in
    the resume file. Capped at max_batches: the mean is stable well before the
    full validation set."""
    vals = []
    for i, batch in enumerate(tqdm(loader, desc='gauss baseline', leave=False)):
        n_np, c_np = batch[0].numpy(), batch[1].numpy()
        for j in range(n_np.shape[0]):
            d = cv2.GaussianBlur(n_np[j, 0], (0, 0), sigma)
            vals.append(compute_psnr(np.clip(c_np[j, 0], 0, 1),
                                     np.clip(d, 0, 1), data_range=1.0))
        if max_batches and i + 1 >= max_batches:
            break
    return float(np.mean(vals))


# ---------------- checkpoint helpers ----------------
def _atomic_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = str(path) + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


def _atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = str(path) + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, default=float)
    os.replace(tmp, path)


def _load_torch_file(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _optimizer_to(opt, device):
    for state in opt.state.values():
        for k, v in list(state.items()):
            if torch.is_tensor(v):
                state[k] = v.to(device, non_blocking=True)


def _restore_rng(st):
    if st.get('rng_torch') is not None:
        torch.set_rng_state(st['rng_torch'].detach().cpu().to(torch.uint8))
    if (DEVICE.type == 'cuda' and st.get('rng_cuda') is not None):
        torch.cuda.set_rng_state_all(
            [s.detach().cpu().to(torch.uint8) for s in st['rng_cuda']])
    if st.get('rng_np') is not None:
        np.random.set_state(st['rng_np'])
    if st.get('rng_py') is not None:
        random.setstate(st['rng_py'])


class OutOfTime(Exception):
    """Raised when the Slurm walltime is about to expire."""


# ---------------- fit ----------------
def fit(model, train_dl, val_dl, epochs, lr, wd, ckpt_path,
        patience=None, min_epochs=None, warmup_epochs=2, min_delta=1e-4,
        resume_path=None, pos_weight=None, ema_decay=None, tag=''):
    patience = CFG.PATIENCE if patience is None else patience
    min_epochs = CFG.MIN_EPOCHS if min_epochs is None else min_epochs
    ema_decay = CFG.EMA_DECAY if ema_decay is None else ema_decay

    model = prep_model(model)
    opt = make_optimizer(model, lr, wd)
    scaler = make_scaler()
    ema = EMA(model, ema_decay) if ema_decay else None

    warmup_epochs = min(warmup_epochs, max(epochs - 1, 0))
    if warmup_epochs > 0:
        sched = torch.optim.lr_scheduler.SequentialLR(
            opt,
            schedulers=[
                torch.optim.lr_scheduler.LinearLR(
                    opt, start_factor=0.05, total_iters=warmup_epochs),
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    opt, T_max=max(epochs - warmup_epochs, 1), eta_min=lr * 0.01)],
            milestones=[warmup_epochs])
    else:
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=max(epochs, 1), eta_min=lr * 0.01)

    start_ep, history, best_iou, bad, gauss_psnr = 0, [], -1.0, 0, None

    if resume_path and os.path.isfile(resume_path):
        try:
            st = _load_torch_file(resume_path, map_location='cpu')
            model.load_state_dict(st['model'])
            opt.load_state_dict(st['opt'])
            _optimizer_to(opt, DEVICE)
            if st.get('scaler') is not None:
                scaler.load_state_dict(st['scaler'])
            if st.get('sched') is not None:
                sched.load_state_dict(st['sched'])
            if ema is not None and st.get('ema') is not None:
                ema.load_state_dict(st['ema'])
            _restore_rng(st)
            start_ep = int(st.get('epoch', 0))
            history = list(st.get('history', []))
            best_iou = float(st.get('best_iou', -1.0))
            bad = int(st.get('bad_epochs', 0))
            gauss_psnr = st.get('gauss_psnr')
            if not os.path.isfile(ckpt_path):
                best_iou, bad = -1.0, 0
            print(f"  resumed after epoch {start_ep}/{epochs} "
                  f"(best IoU {best_iou:.4f}, bad {bad}/{patience})")
        except Exception as e:
            print(f"  could not read resume file ({e}); starting fresh")
            start_ep, history, best_iou, bad, gauss_psnr = 0, [], -1.0, 0, None

    if gauss_psnr is None:
        gauss_psnr = gaussian_baseline_over_loader(val_dl)
    print(f"  Gaussian-blur baseline PSNR (val): {gauss_psnr:.2f} dB")

    def save_resume(ep_done):
        if resume_path is None:
            return
        _atomic_save({
            'model': model.state_dict(), 'opt': opt.state_dict(),
            'scaler': scaler.state_dict(), 'sched': sched.state_dict(),
            'ema': ema.state_dict() if ema is not None else None,
            'epoch': ep_done, 'history': history, 'best_iou': best_iou,
            'bad_epochs': bad, 'gauss_psnr': gauss_psnr,
            'rng_torch': torch.get_rng_state(),
            'rng_cuda': (torch.cuda.get_rng_state_all()
                         if DEVICE.type == 'cuda' else None),
            'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
        }, resume_path)

    ep = start_ep
    eval_model = model
    try:
        for ep in range(start_ep + 1, epochs + 1):
            t0 = time.time()
            tr = train_one_epoch(model, train_dl, opt, scaler, ema=ema,
                                 pos_weight=pos_weight, epoch=ep)
            sched.step()

            if ema is not None:
                eval_model = copy_for_eval(model, ema)
            m = evaluate(eval_model, val_dl)

            m.update(epoch=ep, train_loss=tr['loss'], l_den=tr['l_den'],
                     l_seg=tr['l_seg'], img_per_s=tr['img_per_s'],
                     gauss_psnr=gauss_psnr, lr=opt.param_groups[0]['lr'],
                     time=time.time() - t0,
                     gpu_gb=(torch.cuda.max_memory_allocated() / 1024**3
                             if DEVICE.type == 'cuda' else 0.0))
            history.append(m)

            print(f"  ep {ep:02d}  loss {tr['loss']:.4f} "
                  f"(den {tr['l_den']:.4f}/seg {tr['l_seg']:.4f})  "
                  f"PSNR {m['psnr']:.2f}  SSIM {m['ssim']:.3f}  "
                  f"IoU {m['iou']:.4f}  F1 {m.get('best_f1', float('nan')):.4f}"
                  f"@p{m.get('best_thresh_prob', 0.5):.2f}  "
                  f"[{m['time']:.0f}s {tr['img_per_s']:.0f} img/s "
                  f"{m['gpu_gb']:.1f}GB]", flush=True)

            if m['iou'] > best_iou + min_delta:
                best_iou, bad = float(m['iou']), 0
                _atomic_save(
                    {'model': (ema.state_dict() if ema is not None
                               else model.state_dict()),
                     'raw': model.state_dict(),
                     'arch': tag, 'epoch': ep, 'metrics': m},
                    ckpt_path)
            else:
                bad += 1

            save_resume(ep)

            if STOP_REQUESTED['flag']:
                raise OutOfTime("SIGTERM")
            if time_is_short():
                raise OutOfTime(
                    f"{seconds_left()/60:.1f} min of walltime left")
            if ep >= min_epochs and bad >= patience:
                print(f"  early stop at ep {ep} (best IoU {best_iou:.4f})")
                break

    except OutOfTime as e:
        save_resume(ep)
        print(f"  stopping cleanly: {e}. State saved to {resume_path}; "
              f"resubmit the job to continue from epoch {ep + 1}.")
        raise
    except KeyboardInterrupt:
        save_resume(max(start_ep, ep - 1))
        print("  interrupted; rerun to resume from the last completed epoch")
        raise

    return history


def copy_for_eval(model, ema):
    """Load the EMA weights into a throwaway copy so the training weights stay
    where the optimizer left them."""
    global _EVAL_CACHE
    if '_EVAL_CACHE' not in globals():
        _EVAL_CACHE = {}
    key = id(model)
    if key not in _EVAL_CACHE:
        import copy as _copy
        stash = getattr(model, 'aux_logits', None)
        try:
            if hasattr(model, 'aux_logits'):
                model.aux_logits = None
            _EVAL_CACHE[key] = _copy.deepcopy(model)
        finally:
            if hasattr(model, 'aux_logits'):
                model.aux_logits = stash
    tgt = _EVAL_CACHE[key]
    tgt.load_state_dict(ema.state_dict(), strict=False)
    tgt.eval()
    return tgt


In [ ]:
# ============================================================
# 7b. GPU memory management
#
# Insert this cell AFTER the training-machinery cell and BEFORE the N2V and
# CV cells. It provides three things, in increasing order of intrusiveness:
#
#   1. gradient checkpointing  - recomputes activations in the backward pass
#      instead of storing them. Roughly 40-60% less activation memory for
#      about 20-30% more time. This is the right first move: it costs
#      accuracy nothing.
#   2. autotune_batch()        - probes the largest batch that actually fits,
#      by running a real forward+backward+step, so the answer accounts for
#      the optimizer state and the loss terms rather than a rule of thumb.
#   3. OOM-resilient training  - catches an OOM mid-epoch, frees, and retries
#      the batch in halves instead of losing the whole run.
#
# Why HRNet is the one that dies: it keeps a full-resolution branch alive
# through every stage. At 256x256 its activation footprint is several times
# AtomSegNet's, even though the two have similar parameter counts. Parameters
# are not what fills a 40 GB card here; activations are.
# ============================================================
from torch.utils.checkpoint import checkpoint as _ckpt


def _oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or \
        'out of memory' in str(e).lower()


def free_gpu():
    globals().pop('_EVAL_CACHE', None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def gpu_report(tag=''):
    if not torch.cuda.is_available():
        return
    a = torch.cuda.memory_allocated() / 1024**3
    r = torch.cuda.memory_reserved() / 1024**3
    p = torch.cuda.max_memory_allocated() / 1024**3
    t = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  [gpu{' ' + tag if tag else ''}] alloc {a:.1f} / reserved "
          f"{r:.1f} / peak {p:.1f} / total {t:.1f} GiB")


# ---------------- 1. gradient checkpointing ----------------
class _CheckpointWrapper(nn.Module):
    """Wraps a submodule so its activations are recomputed in backward."""

    def __init__(self, mod):
        super().__init__()
        self.mod = mod

    def forward(self, *args, **kwargs):
        if self.training and torch.is_grad_enabled() and any(
                torch.is_tensor(a) and a.requires_grad for a in args):
            return _ckpt(self.mod, *args, use_reentrant=False, **kwargs)
        return self.mod(*args, **kwargs)


def enable_grad_checkpointing(model, types=None, verbose=True):
    """Wrap every ResBlock (and FusionUnit, where it exists) in the model.
    Call this on the raw model BEFORE prep_model / torch.compile."""
    types = types or tuple(
        t for t in (globals().get('ResBlock'), globals().get('FusionUnit'))
        if t is not None)
    if not types:
        return model
    n = 0
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            if isinstance(child, types) and not isinstance(
                    child, _CheckpointWrapper):
                setattr(parent, name, _CheckpointWrapper(child))
                n += 1
    if verbose:
        print(f"  gradient checkpointing on {n} blocks")
    return model


# ---------------- 2. batch autotuning ----------------
def _probe_step(cls, batch, size=None, pos_weight=None, ckpt_grad=True):
    """One real training step. Returns peak GiB, or raises the OOM."""
    size = size or CFG.IMG_SIZE
    free_gpu()
    model = cls()
    if ckpt_grad:
        enable_grad_checkpointing(model, verbose=False)
    model = prep_model(model)
    model.train()
    opt = make_optimizer(model, CFG.LR, CFG.WD)
    scaler = make_scaler()
    try:
        x = to_dev(torch.rand(batch, CFG.IN_CH, size, size))
        y = to_dev(torch.rand(batch, 1, size, size))
        m = to_dev((torch.rand(batch, 1, size, size) > 0.97).float())
        with amp_autocast():
            den, seg = model(x)
            loss = (CFG.LAM_DEN * denoise_loss(den.float(), y)
                    + CFG.LAM_SEG * seg_loss(seg.float(), m,
                                             pos_weight=pos_weight, epoch=99))
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        peak = (torch.cuda.max_memory_allocated() / 1024**3
                if torch.cuda.is_available() else 0.0)
        return peak
    finally:
        del model, opt, scaler
        free_gpu()


def autotune_batch(arch_name, candidates=None, size=None, headroom=0.85,
                   ckpt_grad=True):
    """Largest batch that survives a real forward+backward+step, with a
    headroom margin so a slightly heavier real batch does not tip it over."""
    if not torch.cuda.is_available():
        return arch_batch(arch_name)
    cls = ARCH_REGISTRY[arch_name]
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    candidates = candidates or [64, 48, 32, 24, 16, 12, 8, 4, 2]
    for b in sorted(candidates, reverse=True):
        try:
            peak = _probe_step(cls, b, size, ckpt_grad=ckpt_grad)
            if peak < headroom * total:
                print(f"  {arch_name}: batch {b} fits "
                      f"(peak {peak:.1f} / {total:.0f} GiB)")
                return b
            print(f"  {arch_name}: batch {b} peaks at {peak:.1f} GiB, "
                  f"over the {headroom:.0%} margin")
        except Exception as e:
            if not _oom(e):
                raise
            free_gpu()
            print(f"  {arch_name}: batch {b} OOM")
    print(f"  {arch_name}: falling back to batch 2")
    return 2


def autotune_all(archs=None, apply=True):
    """Fill CFG.BATCH_OVERRIDE from measurements instead of guesses. Takes a
    couple of minutes and saves a run."""
    out = {}
    for a in (archs or CFG.ARCHS):
        out[a] = autotune_batch(a)
    if apply:
        CFG.BATCH_OVERRIDE.update(out)
        # keep the effective batch constant across architectures so the
        # comparison stays fair: accumulate whatever the smallest one lacks
        target = max(out.values())
        CFG.ACCUM_OVERRIDE = {a: max(1, round(target / b))
                              for a, b in out.items()}
        print(f"\nBATCH_OVERRIDE = {CFG.BATCH_OVERRIDE}")
        print(f"ACCUM_OVERRIDE = {CFG.ACCUM_OVERRIDE}  "
              f"(effective batch ~{target} for every architecture)")
    return out


def arch_accum(arch_name):
    return getattr(CFG, 'ACCUM_OVERRIDE', {}).get(arch_name, CFG.GRAD_ACCUM)


# ---------------- 3. OOM-resilient step ----------------
def oom_safe_step(model, batch_tensors, opt, scaler, pos_weight, epoch,
                  accum, clip_norm, ema, splits=1):
    """Run one training step. On OOM, halve the batch and run the halves
    sequentially rather than losing the epoch. Returns (loss, l_den, l_seg)
    or None if even a single sample will not fit."""
    noisy, clean, mask = batch_tensors
    try:
        with amp_autocast():
            den, seg = model(noisy)
            l_den = denoise_loss(den.float(), clean)
            l_seg = seg_loss(seg.float(), mask, pos_weight=pos_weight,
                             epoch=epoch)
            aux = getattr(model, 'aux_logits', None)
            if aux:
                l_seg = l_seg + 0.3 * sum(
                    seg_loss(a.float(), mask, pos_weight=pos_weight,
                             epoch=epoch) for a in aux) / len(aux)
            loss = (CFG.LAM_DEN * l_den + CFG.LAM_SEG * l_seg) / accum
        scaler.scale(loss).backward()
        return float(loss.item() * accum), float(l_den.item()), \
            float(l_seg.item())
    except Exception as e:
        if not _oom(e) or noisy.shape[0] < 2 or splits > 4:
            if _oom(e):
                print(f"  OOM at batch size {noisy.shape[0]}; skipping batch")
                free_gpu()
                return None
            raise
        free_gpu()
        h = noisy.shape[0] // 2
        print(f"  OOM at batch {noisy.shape[0]}, retrying as 2 x {h}",
              flush=True)
        outs = []
        for sl in (slice(0, h), slice(h, None)):
            r = oom_safe_step(model,
                              (noisy[sl], clean[sl], mask[sl]),
                              opt, scaler, pos_weight, epoch,
                              accum * 2, clip_norm, ema, splits + 1)
            if r is not None:
                outs.append(r)
        if not outs:
            return None
        return tuple(float(np.mean([o[i] for o in outs])) for i in range(3))


def train_one_epoch(model, loader, opt, scaler, ema=None, pos_weight=None,
                    epoch=0, clip_norm=1.0, accum=None):
    """Drop-in replacement for the version in the training cell, with OOM
    recovery. Same return dict, plus n_oom."""
    model.train()
    accum = max(1, int(accum if accum is not None else CFG.GRAD_ACCUM))
    tot = den_sum = seg_sum = gnorm_sum = 0.0
    n_steps = n_opt = seen = n_oom = 0
    t0 = time.time()
    pbar = tqdm(loader, desc=f'train ep{epoch}', leave=False)

    opt.zero_grad(set_to_none=True)
    for it, batch in enumerate(pbar):
        tensors = (to_dev(batch[0]), to_dev(batch[1]), to_dev(batch[2]))
        r = oom_safe_step(model, tensors, opt, scaler, pos_weight, epoch,
                          accum, clip_norm, ema)
        if r is None:
            n_oom += 1
            opt.zero_grad(set_to_none=True)
            continue
        li, ld, ls = r

        if (it + 1) % accum == 0:
            if clip_norm:
                scaler.unscale_(opt)
                gn = torch.nn.utils.clip_grad_norm_(model.parameters(),
                                                    clip_norm)
                if math.isfinite(float(gn)):
                    gnorm_sum += float(gn)
                    n_opt += 1
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        if math.isfinite(li):
            tot += li
            den_sum += ld
            seg_sum += ls
            n_steps += 1

        if hasattr(model, 'aux_logits'):
            model.aux_logits = None

        seen += tensors[0].shape[0]
        if it % 20 == 0:
            pbar.set_postfix(loss=f"{li:.4f}", den=f"{ld:.4f}", seg=f"{ls:.4f}",
                             ips=f"{seen / max(time.time() - t0, 1e-6):.0f}")

    if n_oom:
        print(f"  WARNING: {n_oom} batches skipped on OOM this epoch. "
              f"Lower CFG.BATCH_OVERRIDE['<arch>'] rather than living with it.")

    n_steps = max(n_steps, 1)
    return dict(loss=tot / n_steps, l_den=den_sum / n_steps,
                l_seg=seg_sum / n_steps, grad_norm=gnorm_sum / max(n_opt, 1),
                n_oom=n_oom,
                img_per_s=seen / max(time.time() - t0, 1e-6))


# ---------------- evaluation in chunks ----------------
_orig_evaluate = evaluate


@torch.inference_mode()
def evaluate(model, loader, thresh_logit=0.0, tta=False, sweep=False, ci=True):
    """Same as before, but D4 TTA no longer stacks all 8 transforms at once.
    That stack was 8x the batch in one allocation and is a common OOM source
    at evaluation time even when training fits."""
    try:
        return _orig_evaluate(model, loader, thresh_logit, tta, sweep, ci)
    except Exception as e:
        if not _oom(e):
            raise
        free_gpu()
        print("  eval OOM; retrying without TTA")
        return _orig_evaluate(model, loader, thresh_logit, False, sweep, ci)


print("memory helpers ready: enable_grad_checkpointing(), autotune_batch(), "
      "autotune_all(), oom_safe_step(), gpu_report()")
if torch.cuda.is_available():
    gpu_report('at import')

## 8. Noise2Void self-supervised pre-training

AtomSegNet is pre-trained with N2V on all noisy images, no labels, and its
weights warm-start the supervised stage.


In [ ]:
EMA.weights = EMA.state_dict
print("EMA weights fix:", hasattr(EMA, "weights"))

In [ ]:
# ============================================================
# 8. Noise2Void self-supervised pre-training  (per-architecture, OOM-safe)
#
# Same science as before. What changed is memory handling, because this cell
# was the one dying:
#
#   * gradient checkpointing on every ResBlock/FusionUnit, so activations are
#     recomputed in backward instead of stored. This is where the memory goes:
#     at 256x256 HRNet's activation footprint is several times AtomSegNet's
#     despite a similar parameter count.
#   * a real forward+backward probe picks the N2V batch size per architecture
#     instead of reusing the supervised one. N2V has only the denoise branch
#     in its loss but the same activation cost, so the supervised batch is not
#     automatically safe here.
#   * an OOM mid-epoch halves the batch and retries rather than killing the
#     job. Skipped batches are counted and reported, not swallowed.
#   * validation, the PSNR probe and the EMA eval copy all run at a smaller
#     batch and free the copy afterwards. The eval deepcopy is a second full
#     set of parameters resident on the card.
#
# The cell is self-contained: if 7b (memory_cell) is loaded it uses those
# helpers, otherwise it defines its own.
# ============================================================
if 'enable_grad_checkpointing' not in globals():
    from torch.utils.checkpoint import checkpoint as _ckpt

    def _oom(e):
        return isinstance(e, torch.cuda.OutOfMemoryError) or \
            'out of memory' in str(e).lower()

    def free_gpu():
        globals().pop('_EVAL_CACHE', None)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

    class _CheckpointWrapper(nn.Module):
        def __init__(self, mod):
            super().__init__()
            self.mod = mod

        def forward(self, *a, **kw):
            if self.training and torch.is_grad_enabled() and any(
                    torch.is_tensor(x) and x.requires_grad for x in a):
                return _ckpt(self.mod, *a, use_reentrant=False, **kw)
            return self.mod(*a, **kw)

    def enable_grad_checkpointing(model, types=None, verbose=True):
        types = types or tuple(t for t in (globals().get('ResBlock'),
                                           globals().get('FusionUnit'))
                               if t is not None)
        if not types:
            return model
        n = 0
        for parent in list(model.modules()):
            for name, child in list(parent.named_children()):
                if isinstance(child, types) and not isinstance(
                        child, _CheckpointWrapper):
                    setattr(parent, name, _CheckpointWrapper(child))
                    n += 1
        if verbose:
            print(f"  gradient checkpointing on {n} blocks")
        return model


for _k, _v in dict(N2V_LR=5e-4, N2V_VAL_FRACTION=0.05, N2V_PATIENCE=5,
                   N2V_MIN_EPOCHS=5, N2V_EMA_DECAY=0.999,
                   N2V_GRAD_CKPT=True, N2V_AUTOTUNE=True,
                   N2V_EVAL_BATCH=8, N2V_BATCH_OVERRIDE={}).items():
    if not hasattr(CFG, _k):
        setattr(CFG, _k, _v)


def n2v_loss(pred, target, mask):
    """MSE restricted to the masked (blind-spot) pixels."""
    return ((pred - target) ** 2 * mask).sum() / (mask.sum() + 1e-6)


N2V_FILES = {
    "AtomSegNet": "atomsegnet_n2v.pt",
    "UNetPP":     "unetpp_n2v.pt",
    "HRNet":      "hrnet_n2v.pt",
}


def _n2v_splits():
    """One split shared by all three architectures, so the N2V numbers are
    comparable across them."""
    rng = np.random.RandomState(SEED + 7)
    perm = rng.permutation(len(common))
    n_val = max(16, int(round(CFG.N2V_VAL_FRACTION * len(common))))
    return ([common[i] for i in perm[n_val:]],
            [common[i] for i in perm[:n_val]])


N2V_TRAIN, N2V_VAL = _n2v_splits()


# ---------------- batch autotuning for the N2V objective ----------------
def autotune_n2v_batch(arch_name, candidates=None, headroom=0.80):
    """Probe a real N2V forward+backward+step. The supervised batch is not a
    safe default here: the loss is lighter but the activations are the same,
    and this cell runs before any of the supervised tuning."""
    if arch_name in CFG.N2V_BATCH_OVERRIDE:
        return CFG.N2V_BATCH_OVERRIDE[arch_name]
    if not torch.cuda.is_available():
        return min(arch_batch(arch_name), 8)

    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    start = arch_batch(arch_name)
    candidates = candidates or sorted(
        {b for b in (start, 32, 24, 16, 12, 8, 4, 2) if b <= max(start, 32)},
        reverse=True)

    for b in candidates:
        free_gpu()
        model = ARCH_REGISTRY[arch_name]()
        if CFG.N2V_GRAD_CKPT:
            enable_grad_checkpointing(model, verbose=False)
        model = prep_model(model)
        model.train()
        opt = make_optimizer(model, CFG.N2V_LR, CFG.WD)
        scaler = make_scaler()
        try:
            x = to_dev(torch.rand(b, CFG.IN_CH, CFG.IMG_SIZE, CFG.IMG_SIZE))
            m = to_dev((torch.rand(b, 1, CFG.IMG_SIZE, CFG.IMG_SIZE) > 0.98
                        ).float())
            with amp_autocast():
                den, _ = model(x)
                loss = n2v_loss(den.float(), x, m)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            peak = torch.cuda.max_memory_allocated() / 1024**3
            if peak < headroom * total:
                print(f"  {arch_name} N2V: batch {b} fits "
                      f"(peak {peak:.1f} / {total:.0f} GiB)")
                CFG.N2V_BATCH_OVERRIDE[arch_name] = b
                return b
            print(f"  {arch_name} N2V: batch {b} peaks at {peak:.1f} GiB, "
                  f"over the {headroom:.0%} margin")
        except Exception as e:
            if not _oom(e):
                raise
            print(f"  {arch_name} N2V: batch {b} OOM")
        finally:
            del model, opt, scaler
            free_gpu()

    CFG.N2V_BATCH_OVERRIDE[arch_name] = 2
    print(f"  {arch_name} N2V: falling back to batch 2")
    return 2


# ---------------- OOM-safe train step ----------------
def _n2v_step(model, x_in, x_gt, m, opt, scaler, ema, clip=1.0, depth=0):
    """Returns the loss, or None if the batch could not be made to fit."""
    try:
        opt.zero_grad(set_to_none=True)
        with amp_autocast():
            den, _ = model(x_in)
            loss = n2v_loss(den.float(), x_gt, m)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(opt)
        scaler.update()
        if ema is not None:
            ema.update(model)
        return float(loss.item())
    except Exception as e:
        if not _oom(e):
            raise
        opt.zero_grad(set_to_none=True)
        free_gpu()
        if x_in.shape[0] < 2 or depth >= 4:
            print(f"  OOM at batch {x_in.shape[0]}; skipping", flush=True)
            return None
        h = x_in.shape[0] // 2
        print(f"  OOM at batch {x_in.shape[0]}, retrying as 2 x {h}",
              flush=True)
        vals = [v for sl in (slice(0, h), slice(h, None))
                for v in [_n2v_step(model, x_in[sl], x_gt[sl], m[sl],
                                    opt, scaler, ema, clip, depth + 1)]
                if v is not None]
        return float(np.mean(vals)) if vals else None


# ---------------- evaluation ----------------
@torch.inference_mode()
def n2v_validate(model, dl):
    """Masked-pixel loss on held-out frames. The only quantity N2V is allowed
    to be selected on."""
    model.eval()
    tot = n = 0.0
    for x_in, x_gt, m in dl:
        x_in, x_gt, m = to_dev(x_in), to_dev(x_gt), to_dev(m)
        try:
            with amp_autocast():
                den, _ = model(x_in)
        except Exception as e:
            if not _oom(e):
                raise
            free_gpu()
            continue
        w = m.sum().item()
        if w > 0:
            tot += n2v_loss(den.float(), x_gt, m).item() * w
            n += w
    return tot / max(n, 1e-6)


@torch.inference_mode()
def n2v_psnr_probe(model, dl, max_batches=8):
    """Denoising quality against noNoise. Monitoring only: N2V never trains on
    clean data and this value never influences selection."""
    model.eval()
    ps, gs = [], []
    for i, batch in enumerate(dl):
        noisy, clean = to_dev(batch[0]), to_dev(batch[1])
        try:
            with amp_autocast():
                den, _ = model(noisy)
        except Exception as e:
            if not _oom(e):
                raise
            free_gpu()
            continue
        ps.append(psnr_per_image(den.float(), clean).cpu())
        n_np, c_np = batch[0].numpy(), batch[1].numpy()
        for j in range(n_np.shape[0]):
            b = cv2.GaussianBlur(n_np[j, 0], (0, 0), 1.0)
            gs.append(compute_psnr(np.clip(c_np[j, 0], 0, 1),
                                   np.clip(b, 0, 1), data_range=1.0))
        if max_batches and i + 1 >= max_batches:
            break
    if not ps:
        return float('nan'), float('nan')
    return float(torch.cat(ps).mean()), float(np.mean(gs))


# ---------------- main ----------------
def pretrain_n2v(arch_name, epochs=None, ckpt=None, force=False):
    if arch_name not in ARCH_REGISTRY:
        raise ValueError(f"Unknown architecture: {arch_name}")

    epochs = CFG.N2V_EPOCHS if epochs is None else epochs
    ckpt = ckpt or N2V_FILES[arch_name]
    final_path = os.path.join(CFG.OUT_DIR, ckpt)
    resume_path = final_path + ".resume"

    if os.path.isfile(final_path) and not force:
        print(f"N2V already finished for {arch_name} -> {final_path}")
        return final_path

    free_gpu()
    seed_everything(SEED + 100 * (CFG.ARCHS.index(arch_name)
                                  if arch_name in CFG.ARCHS else 0))

    batch = (autotune_n2v_batch(arch_name) if CFG.N2V_AUTOTUNE
             else arch_batch(arch_name))
    eval_batch = max(1, min(CFG.N2V_EVAL_BATCH, batch))

    print(f"\n=== N2V pretraining: {arch_name} | batch {batch} "
          f"(eval {eval_batch}) | <= {epochs} epochs | "
          f"train {len(N2V_TRAIN)} val {len(N2V_VAL)} ===")

    mk_n2v = lambda names, train: N2VDataset(
        names, noisy_map, size=CFG.IMG_SIZE, ratio=CFG.N2V_MASK_RATIO,
        radius=CFG.N2V_RADIUS, train=train)

    tr_dl = DataLoader(mk_n2v(N2V_TRAIN, True),
                       **loader_options(True, drop_last=True, batch=batch))
    if len(tr_dl) == 0:
        raise RuntimeError(
            f"N2V train loader for {arch_name} has 0 batches: "
            f"{len(N2V_TRAIN)} samples at batch {batch} with drop_last=True.")
    va_dl = DataLoader(mk_n2v(N2V_VAL, False),
                       **loader_options(False, batch=eval_batch))
    probe_dl = DataLoader(
        TEMSegDataset(N2V_VAL, noisy_map, clean_map, mask_map,
                      gauss_map=gauss_map, train=False),
        **loader_options(False, batch=eval_batch))

    model = ARCH_REGISTRY[arch_name]()
    if CFG.N2V_GRAD_CKPT:
        enable_grad_checkpointing(model)
    model = prep_model(model)

    opt = make_optimizer(model, CFG.N2V_LR, CFG.WD)
    scaler = make_scaler()
    ema = EMA(model, CFG.N2V_EMA_DECAY) if CFG.N2V_EMA_DECAY else None
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=max(epochs, 1), eta_min=CFG.N2V_LR * 0.01)

    cfg_now = dict(size=CFG.IMG_SIZE, ratio=CFG.N2V_MASK_RATIO,
                   radius=CFG.N2V_RADIUS, lr=CFG.N2V_LR, n=len(N2V_TRAIN),
                   arch=arch_name, batch=batch,
                   grad_ckpt=bool(CFG.N2V_GRAD_CKPT))

    start_ep, hist, best_val, bad = 0, [], float('inf'), 0

    if os.path.isfile(resume_path) and not force:
        try:
            st = _load_torch_file(resume_path, map_location="cpu")
            if st.get("cfg") != cfg_now:
                print("  resume file was written under different settings "
                      "(batch may have been re-tuned on a different node); "
                      "starting fresh")
            else:
                model.load_state_dict(st["model"])
                opt.load_state_dict(st["opt"])
                _optimizer_to(opt, DEVICE)
                if st.get("sched") is not None:
                    sched.load_state_dict(st["sched"])
                if st.get("scaler") is not None:
                    scaler.load_state_dict(st["scaler"])
                if ema is not None and st.get("ema") is not None:
                    ema.load_state_dict(st["ema"])
                _restore_rng(st)
                start_ep = int(st.get("epoch", 0))
                hist = list(st.get("hist", []))
                best_val = float(st.get("best_val", float('inf')))
                bad = int(st.get("bad", 0))
                last = f"{hist[-1]['val']:.5f}" if hist else "n/a"
                print(f"  resumed {arch_name} N2V after epoch "
                      f"{start_ep}/{epochs} (last val {last})")
        except Exception as e:
            print(f"  could not read N2V resume file ({e}); starting fresh")
            start_ep, hist, best_val, bad = 0, [], float('inf'), 0

    def save_resume(ep_done):
        _atomic_save(dict(
            model=model.state_dict(), opt=opt.state_dict(),
            scaler=scaler.state_dict(), sched=sched.state_dict(),
            ema=ema.state_dict() if ema is not None else None,
            epoch=ep_done, hist=hist, best_val=best_val, bad=bad, cfg=cfg_now,
            rng_torch=torch.get_rng_state(),
            rng_cuda=(torch.cuda.get_rng_state_all()
                      if DEVICE.type == "cuda" else None),
            rng_np=np.random.get_state(), rng_py=random.getstate(),
        ), resume_path)

    def save_best(ep, val):
        _atomic_save(dict(
            model=(ema.state_dict() if ema is not None else model.state_dict()),
            raw=model.state_dict(),
            arch=arch_name,
            epoch=ep,
            val_loss=val,
            hist=hist,
            cfg=cfg_now,
        ), final_path + ".best")

    ep = start_ep
    try:
        for ep in range(start_ep + 1, epochs + 1):
            model.train()
            losses, seen, n_oom, t0 = [], 0, 0, time.time()

            for x_in, x_gt, m in tqdm(tr_dl, leave=False,
                                      desc=f"{arch_name} N2V ep{ep}/{epochs}"):
                x_in, x_gt, m = to_dev(x_in), to_dev(x_gt), to_dev(m)
                li = _n2v_step(model, x_in, x_gt, m, opt, scaler, ema)
                if li is None:
                    n_oom += 1
                    continue
                if math.isfinite(li):
                    losses.append(li)
                seen += x_in.shape[0]

            sched.step()
            train_loss = float(np.mean(losses)) if losses else float('nan')

            eval_model = copy_for_eval(model, ema) if ema is not None else model
            val_loss = n2v_validate(eval_model, va_dl)
            psnr, gauss = n2v_psnr_probe(eval_model, probe_dl)
            # the eval copy is a second full set of parameters on the card
            globals().pop('_EVAL_CACHE', None)
            del eval_model
            free_gpu()

            hist.append(dict(epoch=ep, train=train_loss, val=val_loss,
                             psnr=psnr, gauss_psnr=gauss, n_oom=n_oom,
                             lr=opt.param_groups[0]['lr'], batch=batch,
                             img_per_s=seen / max(time.time() - t0, 1e-6)))

            print(f"  {arch_name} N2V ep {ep:02d}  train {train_loss:.5f}  "
                  f"val {val_loss:.5f}  PSNR {psnr:.2f} dB "
                  f"(gauss {gauss:.2f}, gain {psnr - gauss:+.2f})  "
                  f"[{hist[-1]['img_per_s']:.0f} img/s"
                  + (f", {n_oom} OOM" if n_oom else "") + "]", flush=True)

            if n_oom:
                print(f"  WARNING: {n_oom} batches skipped on OOM. Set "
                      f"CFG.N2V_BATCH_OVERRIDE['{arch_name}'] = {max(2, batch // 2)} "
                      f"and rerun rather than living with it.")

            if math.isfinite(val_loss) and val_loss < best_val - 1e-7:
                best_val, bad = val_loss, 0
                save_best(ep, val_loss)
            else:
                bad += 1

            save_resume(ep)

            if STOP_REQUESTED["flag"] or time_is_short():
                print(f"  walltime nearly up; {arch_name} N2V state saved. "
                      f"Resubmit to continue.")
                raise OutOfTime("walltime")

            if ep >= CFG.N2V_MIN_EPOCHS and bad >= CFG.N2V_PATIENCE:
                print(f"  early stop at ep {ep} (best val {best_val:.5f})")
                break

    except (KeyboardInterrupt, OutOfTime):
        save_resume(max(start_ep, ep - 1))
        raise

    if os.path.isfile(final_path + ".best"):
        st = _load_torch_file(final_path + ".best", map_location="cpu")
        _atomic_save(st["model"], final_path)
        _atomic_json({k: v for k, v in st.items() if k not in ("model", "raw")},
                     final_path.replace(".pt", "_history.json"))
        os.remove(final_path + ".best")
        print(f"  saved {arch_name} N2V (epoch {st['epoch']}, "
              f"val {st['val_loss']:.5f}) -> {final_path}")
    else:
        _atomic_save(model.state_dict(), final_path)
        print(f"  saved {arch_name} N2V (last epoch) -> {final_path}")

    if os.path.isfile(resume_path):
        os.remove(resume_path)

    del model, opt, scaler, ema, tr_dl, va_dl, probe_dl
    free_gpu()
    return final_path


# ============================================================
# Run / reuse N2V for all three architectures
# ============================================================
RUN_N2V = True
n2v_ckpts = {}

if common and RUN_N2V:
    for _arch in CFG.ARCHS:
        n2v_ckpts[_arch] = pretrain_n2v(_arch, force=False)

print("\n=== N2V CHECKPOINTS ===")
for _arch, _path in n2v_ckpts.items():
    _h = _path.replace(".pt", "_history.json")
    _tag = ""
    if os.path.isfile(_h):
        try:
            with open(_h) as _f:
                _d = json.load(_f)
            _last = _d["hist"][-1]
            _gain = float(_last['psnr']) - float(_last['gauss_psnr'])
            _tag = (f"  | best ep {_d['epoch']}, val "
                    f"{float(_d['val_loss']):.5f}, PSNR gain {_gain:+.2f} dB")
        except Exception:
            pass
    print(f"{_arch:12s} -> {_path}{_tag}")

In [ ]:
print("MAX_SAMPLES =", CFG.MAX_SAMPLES)
print("len(common) =", len(common))

In [ ]:
for i, src in enumerate(In):
    if src and "def evaluate(" in src:
        for l in src.splitlines():
            if l.lstrip().startswith("def evaluate("):
                print(f"In[{i}]: {l.strip()}")

In [ ]:
for i, src in enumerate(In):
    if src and "def evaluate(" in src:
        for l in src.splitlines():
            if l.lstrip().startswith("def evaluate("):
                print(f"In[{i}]: {l.strip()}")

In [ ]:
# ============================================================
# FIXED OOM-SAFE EVALUATION WRAPPER
# Compatible with the original:
#     evaluate(model, loader, sweep_threshold=True)
# ============================================================

import gc
import inspect
import torch


# ------------------------------------------------------------
# Recover the REAL original evaluator.
#
# In the broken Cell 12:
#     _orig_evaluate = evaluate
# was already created before evaluate() was overwritten.
# ------------------------------------------------------------
if "_orig_evaluate" in globals():
    _eval_core = _orig_evaluate
else:
    _eval_core = evaluate


_core_sig = inspect.signature(_eval_core)

print("Recovered base evaluate signature:", _core_sig)


# ------------------------------------------------------------
# Safety check: the original evaluator in this notebook should
# have sweep_threshold as its optional third argument.
# ------------------------------------------------------------
if "sweep_threshold" not in _core_sig.parameters:
    raise RuntimeError(
        "Could not recover the original evaluator.\n"
        f"Found signature: {_core_sig}\n"
        "Expected: (model, loader, sweep_threshold=True)"
    )


def _eval_is_oom(exc):
    """True only for CUDA out-of-memory errors."""
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True

    msg = str(exc).lower()
    return "cuda" in msg and "out of memory" in msg


def _clear_eval_gpu():
    """Release cached GPU memory before an evaluation retry."""
    globals().pop("_EVAL_CACHE", None)

    gc.collect()

    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception:
            pass


def _call_original_evaluate(model, loader, sweep=True):
    """
    Call the ORIGINAL evaluator using its ACTUAL signature.

    IMPORTANT:
    Do NOT pass thresh_logit / tta / ci to the old evaluator.
    It does not accept those arguments.
    """
    return _eval_core(
        model,
        loader,
        sweep_threshold=bool(sweep),
    )


# ------------------------------------------------------------
# Compatibility wrapper
#
# It accepts the newer six-argument API so any existing code
# using that API will not crash, but maps it correctly onto the
# notebook's original 3-argument evaluator.
# ------------------------------------------------------------
@torch.inference_mode()
def evaluate(
    model,
    loader,
    thresh_logit=None,
    tta=False,
    sweep=True,
    ci=False,
):
    """
    OOM-safe compatibility evaluator.

    Parameters thresh_logit, tta and ci are accepted only for
    compatibility with newer notebook code.

    The actual CV evaluator remains the original implementation,
    which computes PSNR, SSIM, IoU and threshold-sweep F1.
    """

    try:
        return _call_original_evaluate(
            model,
            loader,
            sweep=sweep,
        )

    except Exception as e:

        if not _eval_is_oom(e):
            # Any non-memory error is a real bug.
            raise

        print(
            "\n  CUDA OOM during validation."
            "\n  Clearing GPU cache and retrying evaluation once...",
            flush=True,
        )

        _clear_eval_gpu()

        try:
            return _call_original_evaluate(
                model,
                loader,
                sweep=sweep,
            )

        except Exception as e2:
            if _eval_is_oom(e2):
                raise RuntimeError(
                    "Evaluation still OOM after GPU cleanup. "
                    "Reduce the validation batch size."
                ) from e2
            raise


# ------------------------------------------------------------
# Keep the original evaluator permanently accessible.
# This also makes this repair cell safe to rerun.
# ------------------------------------------------------------
_orig_evaluate = _eval_core


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------
print("FIXED evaluate signature :", inspect.signature(evaluate))
print("BASE evaluate signature  :", inspect.signature(_orig_evaluate))

print(
    "\nEvaluation patch installed successfully.\n"
    "CV evaluation will use:\n"
    "  - PSNR\n"
    "  - SSIM\n"
    "  - IoU\n"
    "  - threshold sweep / best F1\n"
    "  - no D4 TTA during CV\n"
)

## 9. 5-fold cross-validation across all three architectures


In [ ]:
CFG.LR = 3e-4

print("New supervised LR =", CFG.LR)

In [ ]:
from pathlib import Path
for f in sorted(Path(NO_N2V_DIR).glob("SwinUNet_fold*.pt")):
    payload = _read_checkpoint(f)
    if isinstance(payload, dict):
        print(f.name, "| arch =", repr(payload.get("arch")),
              "| keys:", [k for k in payload if not torch.is_tensor(payload[k])])

# --- Section 9 patch: architecture-label check in _finalize_fold ---
# The strict state_dict load and the model_config comparison remain the
# authoritative architecture checks. Only the free-text label written by fit()
# is matched loosely, because fit() may store the run tag or the class name.

_ARCH_ALIASES = {"swinunet"}   # add an observed label here only after confirming
                               # the strict load and model_config both pass.

def _normalize_arch(label):
    return str(label).strip().replace("_", "").replace("-", "").lower()

def _arch_label_matches(label, class_name="SwinUNet"):
    if not isinstance(label, str):
        return False
    allowed = set(_ARCH_ALIASES)
    allowed.add(_normalize_arch(class_name))
    allowed.add(_normalize_arch("SwinUNet"))
    # fit() may store a compound tag such as "no_n2v:SwinUNet".
    return any(_normalize_arch(part) in allowed for part in label.split(":"))


def _finalize_fold(out_dir, fold, manifest, completion):
    run_id = manifest["run_id"]
    if completion.get("run_id") != run_id or completion.get("fold") != fold:
        raise RuntimeError("Fold completion marker belongs to a different experiment")
    best = _best_history(completion["history"])
    checkpoint_path, resume_path, _, _ = _fold_paths(out_dir, fold)
    if not checkpoint_path.is_file() or checkpoint_path.stat().st_size == 0:
        raise RuntimeError("fit() returned but the best checkpoint is missing/empty")
    payload = _read_checkpoint(checkpoint_path)
    with torch.random.fork_rng(devices=[]):
        model = _new_swin()
        model.load_state_dict(_extract_state_dict(payload), strict=True)
        swin_class_name = type(model).__name__
    is_plain = isinstance(payload, Mapping) and all(torch.is_tensor(v) for v in payload.values())
    payload = {"model": dict(payload)} if is_plain else dict(payload)
    if "arch" in payload and not _arch_label_matches(payload["arch"], swin_class_name):
        raise RuntimeError(
            f"Saved checkpoint arch label {payload['arch']!r} is not recognised as SwinUNet "
            f"(accepted: 'SwinUNet', {swin_class_name!r}, or a tag containing either). "
            "The weights did load strictly into this Swin model, so if that label is just how "
            "fit() writes its tag, add it to _ARCH_ALIASES; if it names a CNN, fit() is copying "
            "a stale global and should derive arch from the model it was passed.")
    model_config = manifest["specification"]["model_config"]
    if "model_config" in payload and _jsonable(payload["model_config"]) != model_config:
        raise RuntimeError("Saved checkpoint model_config mismatch")
    verified_epoch = "epoch" in payload
    if verified_epoch:
        saved_row = next((row for row in completion["history"]
                          if int(row["epoch"]) == int(payload["epoch"])), None)
        if (saved_row is None or saved_row["iou"] is None or
                not math.isclose(float(saved_row["iou"]), float(best["iou"]), rel_tol=1e-7, abs_tol=1e-9)):
            raise RuntimeError("Best-checkpoint epoch disagrees with best validation-IoU history. "
                               "Inspect fit() checkpoint selection before recording this fold")
        best = dict(saved_row)
    payload.update(arch="SwinUNet", model_config=model_config,
                   swin_cv=dict(run_id=run_id, condition="no_n2v", fold=fold,
                                selected_history_epoch=int(best["epoch"]),
                                checkpoint_epoch_verified=verified_epoch))
    _atomic_write(checkpoint_path, lambda stream: torch.save(payload, stream), binary=True)
    if not verified_epoch:
        print("  note: checkpoint had no epoch metadata; history/checkpoint epoch alignment is unverified")
    best.update(arch="SwinUNet", condition="no_n2v", fold=fold, run_id=run_id,
                epochs_run=len(completion["history"]), last_epoch=int(completion["history"][-1]["epoch"]),
                batch=completion["batch"], eval_batch=completion["eval_batch"],
                grad_accum=completion["grad_accum"], effective_batch=completion["batch"] * completion["grad_accum"],
                oom_retries=completion["oom_retries"], precision=str(CFG.PRECISION),
                params_M=manifest["specification"]["params"] / 1e6,
                checkpoint=str(checkpoint_path), checkpoint_sha256=_file_digest(checkpoint_path),
                checkpoint_epoch_verified=verified_epoch, n2v_checkpoint=None,
                resume_retained=resume_path.is_file())
    rows = [row for row in _load_partial(out_dir) if row["fold"] != fold]
    rows.append(_jsonable(best))
    rows.sort(key=lambda row: row["fold"])
    _swin_atomic_json(rows, _results_json(out_dir))
    print(f"  recorded fold {fold}: IoU {float(best['iou']):.4f} at epoch {best['epoch']}")
    return best

"""Section 9 replacement: SWIN-UNET ONLY, FIVE FOLDS, NO N2V.

Paste this complete file into Section 9, or run it in the existing notebook:
    %run -i /your/path/swin_five_fold_cv.py
It starts CV when executed in that notebook. Importing the module does not.
Run the configuration, data, Swin architecture, losses, and fit() cells first.

IMPORTANT: the uploaded Section 9 contains the CV coordinator, not fit(). This
replacement preserves the existing fit(model, train_loader, val_loader,
epochs=..., lr=..., wd=..., ckpt_path=..., resume_path=..., pos_weight=...,
tag=...) interface. That function must:
  * use CFG.GRAD_ACCUM correctly, including the last incomplete accumulation;
  * call seg_loss with the current 1-based epoch to apply Lovasz warmup;
  * restore raw model, optimizer, scheduler, scaler, EMA, history and RNG state
    from resume_path, and atomically save epoch-boundary progress;
  * save the BEST validation-IoU checkpoint at ckpt_path and return the full
    resumed history (each row has a 1-based epoch and finite iou);
  * accept the new model's (denoised, raw_logits) outputs and aux_logits=None.
This file cannot implement or verify those missing training-loop internals.
It neither changes the optimizer algorithm nor the task-loss balance.

Preserved split rule: KFold(5, shuffle=True, random_state=SEED) on common in
ITS EXISTING ORDER. Do not sort/regenerate common. It must be the same CNN
train+validation pool, excluding the held-out test set. A sample-count check
does not establish dataset independence. Existing group-aware CNN splits
must not be replaced with this image-level KFold protocol.

Outputs go ONLY under CFG.OUT_DIR/swin_cv (or CFG.SWIN_CV_ROOT):
    experiment_no_n2v/SwinUNet_fold1.pt ... SwinUNet_fold5.pt
    experiment_no_n2v/SwinUNet_fold1.pt.resume ... (retained, not deleted)
    experiment_no_n2v/cv_results_partial.json, cv_results.csv
    swin_cv_results.csv
The manifest saves ordered basenames, exact indices, model settings and key
training settings. Incompatible existing progress is rejected, not erased.
Best checkpoints gain model_config and Swin CV metadata for inference.

Only no-N2V is supported here. CNN N2V checkpoints must never be loaded into
Swin. A future with-N2V comparison also needs pretraining/split-leakage checks.

References:
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html
https://docs.pytorch.org/docs/2.1/notes/cuda.html
https://docs.pytorch.org/docs/2.1/notes/amp_examples.html
"""

from collections.abc import Mapping
from contextlib import contextmanager
import gc
import hashlib
import inspect
import json
import math
import os
from pathlib import Path
import tempfile
import traceback

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import torch
from torch.utils.data import DataLoader


RUN_NO_N2V = True
RUN_WITH_N2V = False
RUN_ARCHS = ["SwinUNet"]

SUPERVISED_BATCH_START = {"SwinUNet": 8}
SUPERVISED_BATCH_MIN = 1
TARGET_EFFECTIVE_BATCH = 32
EVAL_BATCH_MAX = 4
EXPECTED_PAIRED_SAMPLES = 14364  # Original train+validation pool, not the test set.
SWIN_SEED_OFFSET = 3000        # Stable even if CFG.ARCHS is later reordered.
CV_FORMAT_VERSION = 1


def _cv_paths():
    if "CFG" not in globals() or not hasattr(CFG, "OUT_DIR"):
        raise RuntimeError("Run the configuration cell first (CFG.OUT_DIR is missing)")
    root = Path(getattr(CFG, "SWIN_CV_ROOT", Path(CFG.OUT_DIR) / "swin_cv")).expanduser().resolve()
    return root, root / "experiment_no_n2v"


if "CFG" in globals() and hasattr(CFG, "OUT_DIR"):
    CV_ROOT, NO_N2V_DIR = map(str, _cv_paths())
    PARALLEL_ROOT = CV_ROOT  # Compatibility alias; never points to the CNN root.
    WITH_N2V_DIR = str(Path(CV_ROOT) / "experiment_with_n2v")  # Reserved, not created.


def _jsonable(value):
    if isinstance(value, Mapping):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return _jsonable(value.tolist())
    if isinstance(value, np.generic):
        return _jsonable(value.item())
    if torch.is_tensor(value):
        if value.numel() != 1:
            raise TypeError("CV history/config should not contain non-scalar tensors")
        return _jsonable(value.detach().cpu().item())
    if isinstance(value, (Path, torch.device, torch.dtype)):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None  # Optional NaN metrics are JSON null; IoU is checked separately.
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    raise TypeError(f"Cannot serialize CV value of type {type(value).__name__}")


def _digest(value):
    payload = json.dumps(_jsonable(value), sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _file_digest(path):
    result = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def _atomic_write(path, writer, binary=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    mode = "wb" if binary else "w"
    kwargs = {} if binary else {"encoding": "utf-8"}
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode=mode, dir=path.parent,
                                         prefix=f".{path.name}.", suffix=".tmp",
                                         delete=False, **kwargs) as stream:
            temporary = Path(stream.name)
            writer(stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        # Only our own uncommitted temporary file can be removed here.
        if temporary is not None and temporary.exists():
            temporary.unlink()


def _swin_atomic_json(value, path):
    clean = _jsonable(value)
    _atomic_write(path, lambda stream: json.dump(clean, stream, indent=2, allow_nan=False))


def _read_json(path):
    try:
        with open(path, encoding="utf-8") as stream:
            return json.load(stream)
    except (OSError, ValueError) as error:
        raise RuntimeError(f"Cannot read progress file {path}; refusing to restart or overwrite it") from error


@contextmanager
def _writer_lock(out_dir):
    """Linux/HPC advisory lock; stale lock files themselves do not hold a lock."""
    import fcntl
    directory = Path(out_dir)
    directory.mkdir(parents=True, exist_ok=True)
    with open(directory / ".swin_cv.lock", "a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError(f"Another Swin CV writer is active in {directory}") from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


def _results_json(out_dir):
    return str(Path(out_dir) / "cv_results_partial.json")


def _load_partial(out_dir):
    path = Path(_results_json(out_dir))
    if not path.exists():
        return []
    rows = _read_json(path)
    if not isinstance(rows, list):
        raise RuntimeError(f"Expected a JSON list in {path}")
    seen = set()
    for row in rows:
        if not isinstance(row, dict) or row.get("arch") != "SwinUNet" or row.get("condition") != "no_n2v":
            raise RuntimeError(f"Non-Swin or cross-condition records in {path}; use a separate Swin directory")
        fold = row.get("fold")
        if not isinstance(fold, int) or not 1 <= fold <= 5 or fold in seen:
            raise RuntimeError(f"Invalid/duplicate fold records in {path}")
        seen.add(fold)
    return rows


def _read_checkpoint(path):
    # Reuse the notebook's established loader when supplied. Its inputs here
    # are checkpoints from this experiment, never arbitrary downloaded files.
    helper = globals().get("_load_torch_file")
    if callable(helper):
        return helper(str(path), map_location="cpu")
    # No automatic unsafe-pickle fallback for checkpoints with custom objects.
    return torch.load(path, map_location="cpu", weights_only=True)


def _extract_state_dict(payload, prefer_ema=True):
    def is_state(value):
        return isinstance(value, Mapping) and bool(value) and all(torch.is_tensor(v) for v in value.values())
    if is_state(payload):
        state = dict(payload)
    elif isinstance(payload, Mapping):
        order = (("model", "ema", "raw", "state_dict", "model_state_dict") if prefer_ema else
                 ("raw", "model", "state_dict", "model_state_dict", "ema"))
        state = next((dict(payload[k]) for k in order if k in payload and is_state(payload[k])), None)
        if state is None:
            raise TypeError("Checkpoint does not contain a supported model state_dict")
    else:
        raise TypeError("Checkpoint must contain a mapping")
    while state:
        prefix = next((p for p in ("module.", "_orig_mod.")
                       if all(str(k).startswith(p) for k in state)), None)
        if prefix is None:
            break
        state = {k[len(prefix):]: v for k, v in state.items()}
    return state


def load_model_weights(path, model, prefer_ema=True):
    """Strict same-architecture load; never accept an 80%-matching CNN state."""
    if not path or not Path(path).is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    payload = _read_checkpoint(path)
    if isinstance(payload, Mapping) and "model_config" in payload and hasattr(model, "model_config"):
        if _jsonable(payload["model_config"]) != _jsonable(model.model_config):
            raise RuntimeError("Checkpoint model_config does not match this Swin model")
    model.load_state_dict(_extract_state_dict(payload, prefer_ema), strict=True)
    return model


def _new_swin():
    options = dict(in_ch=1, img_size=CFG.IMG_SIZE,
                   patch_size=getattr(CFG, "SWIN_PATCH_SIZE", 4),
                   window_size=getattr(CFG, "SWIN_WINDOW_SIZE", 8),
                   embed_dim=getattr(CFG, "SWIN_EMBED_DIM", 56),
                   depths=getattr(CFG, "SWIN_DEPTHS", (2, 2, 2, 2)),
                   num_heads=getattr(CFG, "SWIN_HEADS", (2, 4, 8, 16)),
                   use_checkpoint=getattr(CFG, "SWIN_USE_CHECKPOINT", False))
    options.update(getattr(CFG, "SWIN_MODEL_KWARGS", {}))
    model = ARCH_REGISTRY["SwinUNet"](**options)
    if not isinstance(getattr(model, "model_config", None), dict):
        raise RuntimeError("Use the supplied SwinUNet architecture cell with model_config metadata")
    return model


def _check_dependencies():
    required = ("CFG", "SEED", "common", "noisy_map", "clean_map", "mask_map",
                "gauss_map", "ARCH_REGISTRY", "TEMSegDataset", "loader_options",
                "estimate_pos_weight", "seed_everything", "fit")
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Run the earlier notebook cells first. Missing: " + ", ".join(missing))
    required_cfg = ("OUT_DIR", "IMG_SIZE", "N_FOLDS", "EPOCHS", "LR", "WD", "PATIENCE", "PRECISION", "LAM_GRAD")
    absent = [name for name in required_cfg if not hasattr(CFG, name)]
    if absent:
        raise RuntimeError("Missing existing experiment settings: " + ", ".join("CFG." + n for n in absent))
    if "SwinUNet" not in ARCH_REGISTRY:
        raise RuntimeError("Run the Swin architecture cell before this CV cell")
    if RUN_ARCHS != ["SwinUNet"] or RUN_WITH_N2V:
        raise RuntimeError("This first Swin experiment is no-N2V only: RUN_ARCHS=['SwinUNet'], RUN_WITH_N2V=False")
    # Validate the call signature, not the unprovided function's internals.
    try:
        inspect.signature(fit).bind(object(), object(), object(), epochs=1,
                                    lr=1e-4, wd=0.0, ckpt_path="best.pt",
                                    resume_path="best.pt.resume", pos_weight=None, tag="no_n2v:SwinUNet")
    except (TypeError, ValueError) as error:
        raise RuntimeError("fit() must accept the resume/epoch/checkpoint arguments used by your original Section 9") from error
    fit_globals = getattr(fit, "__globals__", {})
    if "CFG" in fit_globals and fit_globals["CFG"] is not CFG:
        raise RuntimeError("fit() and this runner use different CFG objects; paste the cell or use %run -i")


def _validate_pool(n_folds):
    if n_folds != 5:
        raise ValueError("This experiment requires exactly five folds; retain CFG.N_FOLDS=5")
    if getattr(CFG, "MAX_SAMPLES", None) is not None:
        raise RuntimeError("CFG.MAX_SAMPLES must be None for the full comparison")
    names = list(common)  # Deliberately NOT sorted.
    if not all(isinstance(name, str) for name in names) or len(set(names)) != len(names):
        raise RuntimeError("common must contain unique string basenames in the original CNN order")
    expected = int(getattr(CFG, "EXPECTED_PAIRED_SAMPLES", EXPECTED_PAIRED_SAMPLES))
    if len(names) != expected or len(names) < n_folds:
        raise RuntimeError(f"Expected {expected} paired training/validation samples, found {len(names)}")
    for label, mapping in (("noisy", noisy_map), ("clean", clean_map), ("mask", mask_map)):
        missing = [name for name in names if mapping.get(name) is None]
        if missing:
            raise RuntimeError(f"Missing {label} paths for {len(missing)} paired samples; first: {missing[:3]}")
    return names


def _cpu_allocation():
    available = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    try:
        allocated = int(os.environ.get("SLURM_CPUS_PER_TASK", available))
    except ValueError:
        allocated = available
    return max(1, min(allocated, available))


def cv_preflight():
    global DEVICE, CV_ROOT, NO_N2V_DIR, WITH_N2V_DIR, PARALLEL_ROOT
    _check_dependencies()
    _validate_pool(int(CFG.N_FOLDS))
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is visible. Run this training cell inside your allocated SLURM GPU job")
    logical_index = int(getattr(CFG, "SWIN_DEVICE_INDEX", 0))
    if not 0 <= logical_index < torch.cuda.device_count():
        raise RuntimeError("CFG.SWIN_DEVICE_INDEX is outside the visible CUDA devices")
    # Respect scheduler visibility: never rewrite CUDA_VISIBLE_DEVICES after import.
    DEVICE = torch.device("cuda", logical_index)
    torch.cuda.set_device(DEVICE)
    workers = max(0, min(int(getattr(CFG, "NUM_WORKERS", 4)), _cpu_allocation() - 1, 8))
    CFG.NUM_WORKERS = workers
    CFG.ARCHS = ["SwinUNet"]
    CV_ROOT, NO_N2V_DIR = map(str, _cv_paths())
    PARALLEL_ROOT = CV_ROOT
    WITH_N2V_DIR = str(Path(CV_ROOT) / "experiment_with_n2v")
    print("\n=== SWIN-UNET 5-FOLD CV: NO N2V ===")
    print(f"Samples: {len(common)} | folds: {CFG.N_FOLDS} | split seed: {SEED}")
    print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
    print(f"Using {DEVICE}: {torch.cuda.get_device_name(logical_index)} | loader workers: {workers}")
    print(f"Epoch limit: {CFG.EPOCHS} | LR: {getattr(CFG, 'SWIN_LR', CFG.LR)} | precision: {CFG.PRECISION}")
    print(f"Micro-batches: {_supervised_batch_candidates('SwinUNet')} | nominal effective batch: {TARGET_EFFECTIVE_BATCH}")
    print("Swin output:", NO_N2V_DIR)
    print("Existing CNN directories and checkpoints are not modified.")
    print("fit() must handle accumulation, epoch-aware loss, best-IoU saving and full-state resume.")


def _supervised_batch_candidates(arch_name):
    if arch_name != "SwinUNet":
        raise ValueError("This runner only trains SwinUNet")
    start, floor = int(SUPERVISED_BATCH_START[arch_name]), int(SUPERVISED_BATCH_MIN)
    if not 1 <= floor <= start or TARGET_EFFECTIVE_BATCH < 1 or EVAL_BATCH_MAX < 1:
        raise ValueError("Invalid micro-batch/accumulation policy")
    values = []
    while True:
        values.append(start)
        if start == floor:
            return values
        start = max(floor, start // 2)


def _eval_batch_for_arch(arch_name, train_batch):
    return max(1, min(int(EVAL_BATCH_MAX), int(train_batch)))


def _grad_accum_for_batch(batch):
    return max(1, math.ceil(TARGET_EFFECTIVE_BATCH / batch))


def _is_cuda_oom(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or (
        "cuda" in str(error).lower() and "out of memory" in str(error).lower())


def _free_cv_gpu():
    globals().pop("_EVAL_CACHE", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _make_manifest(names, n_folds, epochs):
    folds = [dict(fold=i, train_indices=train.tolist(), val_indices=val.tolist())
             for i, (train, val) in enumerate(KFold(n_splits=n_folds, shuffle=True,
                                                    random_state=int(SEED)).split(names), 1)]
    # Capture only task-relevant settings, never arbitrary environment/CFG secrets.
    keys = ("IMG_SIZE", "PATIENCE", "PRECISION", "LAM_GRAD", "LAM_SSIM", "W_BCE",
            "W_DICE", "W_JACCARD", "LOVASZ_WEIGHT", "LOVASZ_WARMUP_EPOCHS", "EDGE_WEIGHT",
            "POS_WEIGHT_MAX", "W_SEG", "W_DEN", "LAM_SEG", "LAM_DENOISE", "AUX_WEIGHT",
            "EMA_DECAY", "USE_EMA", "WARMUP_EPOCHS", "MIN_LR", "GRAD_CLIP", "CLIP_GRAD")
    settings = {key: getattr(CFG, key) for key in keys if hasattr(CFG, key)}
    settings.update(epochs=epochs, lr=getattr(CFG, "SWIN_LR", CFG.LR),
                    wd=getattr(CFG, "SWIN_WD", CFG.WD), effective_batch=TARGET_EFFECTIVE_BATCH)
    # Source paths catch a different pairing/root. Image content changed in place
    # is not hashed here; keep a versioned, immutable dataset for comparison.
    sources = [(name, str(noisy_map[name]), str(clean_map[name]), str(mask_map[name]),
                str(gauss_map[name]) if gauss_map.get(name) is not None else None) for name in names]
    with torch.random.fork_rng(devices=[]):
        model = _new_swin()
        model_config = _jsonable(model.model_config)
        shape_spec = {key: list(value.shape) for key, value in model.state_dict().items()}
        params = sum(p.numel() for p in model.parameters())
    specification = dict(format_version=CV_FORMAT_VERSION, arch="SwinUNet", condition="no_n2v",
                         seed=int(SEED), fold_seed_offset=SWIN_SEED_OFFSET, n_folds=n_folds,
                         basenames=names, folds=folds, sources_sha256=_digest(sources),
                         model_config=model_config, state_shapes=shape_spec, params=params,
                         training_settings=_jsonable(settings), run_tag=str(getattr(CFG, "SWIN_RUN_TAG", "v1")))
    return {"run_id": _digest(specification), "specification": specification}


def _ensure_manifest(out_dir, wanted):
    path = Path(out_dir) / "swin_cv_manifest.json"
    if path.exists():
        existing = _read_json(path)
        if existing != wanted:
            raise RuntimeError("Swin CV manifest mismatch: samples/order, splits, model or training settings changed. "
                               "Restore the original settings, or explicitly choose a NEW CFG.SWIN_CV_ROOT")
    else:
        occupied = [p.name for p in Path(out_dir).iterdir() if p.name != ".swin_cv.lock"]
        if occupied:
            raise RuntimeError(f"Existing files without a Swin manifest in {out_dir}: {occupied[:5]}. "
                               "Refusing to adopt/overwrite unknown progress")
        _swin_atomic_json(wanted, path)


def _fold_paths(out_dir, fold):
    checkpoint_path = Path(out_dir) / f"SwinUNet_fold{fold}.pt"
    return checkpoint_path, Path(str(checkpoint_path) + ".resume"), Path(out_dir) / f"SwinUNet_fold{fold}.attempt.json", Path(out_dir) / f"SwinUNet_fold{fold}.fit_complete.json"


def _best_history(history):
    if not isinstance(history, (list, tuple)) or not history:
        raise RuntimeError("fit() must return a nonempty full list of epoch records")
    epochs, finite_rows = [], []
    for row in history:
        if not isinstance(row, Mapping) or "epoch" not in row or "iou" not in row:
            raise RuntimeError("Every fit history row must contain epoch and iou")
        epoch = int(row["epoch"])
        if epoch < 1 or epoch != row["epoch"]:
            raise RuntimeError("fit history must use 1-based integer epochs")
        epochs.append(epoch)
        if row["iou"] is not None and math.isfinite(float(row["iou"])):
            finite_rows.append(row)
    if epochs != sorted(set(epochs)) or not finite_rows:
        raise RuntimeError("fit history has duplicate/out-of-order epochs or no finite validation IoU")
    return dict(max(finite_rows, key=lambda row: float(row["iou"])))


def _finalize_fold(out_dir, fold, manifest, completion):
    run_id = manifest["run_id"]
    if completion.get("run_id") != run_id or completion.get("fold") != fold:
        raise RuntimeError("Fold completion marker belongs to a different experiment")
    best = _best_history(completion["history"])
    checkpoint_path, resume_path, _, _ = _fold_paths(out_dir, fold)
    if not checkpoint_path.is_file() or checkpoint_path.stat().st_size == 0:
        raise RuntimeError("fit() returned but the best checkpoint is missing/empty")
    payload = _read_checkpoint(checkpoint_path)
    with torch.random.fork_rng(devices=[]):
        model = _new_swin()
        model.load_state_dict(_extract_state_dict(payload), strict=True)
    is_plain = isinstance(payload, Mapping) and all(torch.is_tensor(v) for v in payload.values())
    payload = {"model": dict(payload)} if is_plain else dict(payload)
    if "arch" in payload and payload["arch"] != "SwinUNet":
        raise RuntimeError("Saved checkpoint is tagged as a different architecture")
    model_config = manifest["specification"]["model_config"]
    if "model_config" in payload and _jsonable(payload["model_config"]) != model_config:
        raise RuntimeError("Saved checkpoint model_config mismatch")
    verified_epoch = "epoch" in payload
    if verified_epoch:
        saved_row = next((row for row in completion["history"]
                          if int(row["epoch"]) == int(payload["epoch"])), None)
        if (saved_row is None or saved_row["iou"] is None or
                not math.isclose(float(saved_row["iou"]), float(best["iou"]), rel_tol=1e-7, abs_tol=1e-9)):
            raise RuntimeError("Best-checkpoint epoch disagrees with best validation-IoU history. "
                               "Inspect fit() checkpoint selection before recording this fold")
        # fit() may break exact IoU ties using either the first or last epoch.
        # Report PSNR/SSIM from the epoch actually represented by its checkpoint.
        best = dict(saved_row)
    payload.update(arch="SwinUNet", model_config=model_config,
                   swin_cv=dict(run_id=run_id, condition="no_n2v", fold=fold,
                                selected_history_epoch=int(best["epoch"]),
                                checkpoint_epoch_verified=verified_epoch))
    _atomic_write(checkpoint_path, lambda stream: torch.save(payload, stream), binary=True)
    if not verified_epoch:
        print("  note: checkpoint had no epoch metadata; history/checkpoint epoch alignment is unverified")
    best.update(arch="SwinUNet", condition="no_n2v", fold=fold, run_id=run_id,
                epochs_run=len(completion["history"]), last_epoch=int(completion["history"][-1]["epoch"]),
                batch=completion["batch"], eval_batch=completion["eval_batch"],
                grad_accum=completion["grad_accum"], effective_batch=completion["batch"] * completion["grad_accum"],
                oom_retries=completion["oom_retries"], precision=str(CFG.PRECISION),
                params_M=manifest["specification"]["params"] / 1e6,
                checkpoint=str(checkpoint_path), checkpoint_sha256=_file_digest(checkpoint_path),
                checkpoint_epoch_verified=verified_epoch, n2v_checkpoint=None,
                resume_retained=resume_path.is_file())
    rows = [row for row in _load_partial(out_dir) if row["fold"] != fold]
    rows.append(_jsonable(best))
    rows.sort(key=lambda row: row["fold"])
    _swin_atomic_json(rows, _results_json(out_dir))
    print(f"  recorded fold {fold}: IoU {float(best['iou']):.4f} at epoch {best['epoch']}")
    return best


def run_cv(arch_name="SwinUNet", condition="no_n2v", out_dir=None,
           n_folds=None, epochs=None, warm_start=None):
    """Train/skip/recover all five folds; fit() owns actual epoch restoration.
    OOM batch changes preserve only the NOMINAL effective batch, not bitwise
    trajectories. fit() must support the resulting loader/accumulation change.
    """
    _check_dependencies()
    if arch_name != "SwinUNet" or condition != "no_n2v" or warm_start is not None:
        raise ValueError("This runner is exclusively SwinUNet from random initialization (no_n2v)")
    if "DEVICE" not in globals():
        raise RuntimeError("Run cv_preflight() first to select the allocated device")
    n_folds = int(CFG.N_FOLDS if n_folds is None else n_folds)
    epochs = int(CFG.EPOCHS if epochs is None else epochs)
    if epochs < 1:
        raise ValueError("epochs must be positive")
    names = _validate_pool(n_folds)
    root, default_out = _cv_paths()
    out_dir = Path(out_dir).expanduser().resolve() if out_dir is not None else default_out
    if not out_dir.is_relative_to(root) or out_dir == root:
        raise RuntimeError("The condition directory must be inside the dedicated CFG.SWIN_CV_ROOT")
    with _writer_lock(out_dir):
        manifest = _make_manifest(names, n_folds, epochs)
        _ensure_manifest(out_dir, manifest)
        recorded = {row["fold"]: row for row in _load_partial(out_dir)}
        print(f"\nSwinUNet | five folds | <= {epochs} epochs/fold | {manifest['specification']['params']/1e6:.3f} M parameters")
        for split in manifest["specification"]["folds"]:
            fold = split["fold"]
            ckpt, resume, attempt_path, complete_path = _fold_paths(out_dir, fold)
            if fold in recorded:
                row = recorded[fold]
                if row.get("run_id") != manifest["run_id"] or not ckpt.is_file():
                    raise RuntimeError(f"Fold {fold} completion record is incompatible or its checkpoint is missing")
                if row.get("checkpoint_sha256") != _file_digest(ckpt):
                    raise RuntimeError(f"Fold {fold} checkpoint was changed/corrupted after recording")
                print(f"-- fold {fold}/5 already complete; skipping")
                continue
            if complete_path.is_file():
                print(f"-- fold {fold}/5 recovering a returned fit() without retraining")
                _finalize_fold(out_dir, fold, manifest, _read_json(complete_path))
                continue
            previous = _read_json(attempt_path) if attempt_path.is_file() else None
            if (ckpt.exists() or resume.exists()) and previous is None:
                raise RuntimeError(f"Fold {fold} has untracked checkpoints; refusing to overwrite/adopt them")
            if previous and (previous.get("run_id") != manifest["run_id"] or previous.get("fold") != fold):
                raise RuntimeError(f"Fold {fold} resume metadata mismatch")
            if ckpt.exists() and not resume.exists():
                raise RuntimeError(f"Fold {fold} has a best checkpoint but no full resume/completion marker. "
                                   "Inspect fit() resume saving before restarting this fold")
            tr_names = [names[i] for i in split["train_indices"]]
            va_names = [names[i] for i in split["val_indices"]]
            fold_seed = int(SEED) + SWIN_SEED_OFFSET + fold
            seed_everything(fold_seed)
            tr_ds = TEMSegDataset(tr_names, noisy_map, clean_map, mask_map, gauss_map=gauss_map, train=True)
            va_ds = TEMSegDataset(va_names, noisy_map, clean_map, mask_map, gauss_map=gauss_map, train=False)
            pos_weight = estimate_pos_weight(tr_names)  # Never validation names.
            candidates = _supervised_batch_candidates(arch_name)
            if previous:
                candidates = [b for b in candidates if b <= int(previous["batch"])]
            if not candidates:
                raise RuntimeError("Current batch policy is incompatible with the recorded attempt")
            history, completion, last_oom = None, None, None
            oom_retries = int(previous.get("oom_retries", 0)) if previous else 0
            print(f"\n-- fold {fold}/5 | train {len(tr_names)} | validation {len(va_names)} | seed {fold_seed}")
            for batch in candidates:
                eval_batch, accum = _eval_batch_for_arch(arch_name, batch), _grad_accum_for_batch(batch)
                seed_everything(fold_seed)
                _free_cv_gpu()
                had_accum = hasattr(CFG, "GRAD_ACCUM")
                old_accum = getattr(CFG, "GRAD_ACCUM", None)
                model = tr_dl = va_dl = None
                try:
                    tr_options = dict(loader_options(True, drop_last=True, batch=batch))
                    va_options = dict(loader_options(False, batch=eval_batch))
                    tr_dl, va_dl = DataLoader(tr_ds, **tr_options), DataLoader(va_ds, **va_options)
                    if len(tr_dl) == 0 or len(va_dl) == 0:
                        raise RuntimeError("Empty train/validation loader; check sample count and batch size")
                    if tr_dl.batch_size != batch or va_dl.batch_size != eval_batch or va_dl.drop_last:
                        raise RuntimeError("loader_options ignored the requested batch or drops validation samples")
                    model = _new_swin()
                    CFG.GRAD_ACCUM = accum
                    attempt = dict(run_id=manifest["run_id"], fold=fold, seed=fold_seed,
                                   batch=batch, eval_batch=eval_batch, grad_accum=accum, oom_retries=oom_retries)
                    _swin_atomic_json(attempt, attempt_path)
                    print(f"  batch {batch}, accumulation {accum}, nominal effective {batch*accum}, val batch {eval_batch}")
                    if resume.is_file():
                        print(f"  passing {resume.name} to fit() for full-state restoration")
                    history = fit(model, tr_dl, va_dl, epochs=epochs,
                                  lr=getattr(CFG, "SWIN_LR", CFG.LR), wd=getattr(CFG, "SWIN_WD", CFG.WD),
                                  ckpt_path=str(ckpt), resume_path=str(resume), pos_weight=pos_weight,
                                  tag="no_n2v:SwinUNet")
                    _best_history(history)
                    completion = dict(attempt, history=_jsonable(history))
                    _swin_atomic_json(completion, complete_path)
                    break
                except Exception as error:
                    if not _is_cuda_oom(error):
                        raise
                    last_oom = str(error)  # Do NOT retain an exception with GPU tensors in its traceback.
                    traceback.clear_frames(error.__traceback__)
                    oom_retries += 1
                    history = None
                    if attempt_path.exists():
                        attempt = _read_json(attempt_path)
                        attempt["oom_retries"] = oom_retries
                        _swin_atomic_json(attempt, attempt_path)
                    print(f"  CUDA OOM at batch {batch}; retrying a smaller batch if available")
                finally:
                    if had_accum:
                        CFG.GRAD_ACCUM = old_accum
                    elif hasattr(CFG, "GRAD_ACCUM"):
                        delattr(CFG, "GRAD_ACCUM")
                    model = tr_dl = va_dl = None
                    _free_cv_gpu()
            if completion is None:
                raise RuntimeError(f"All permitted batches {candidates} failed for fold {fold}. Last CUDA OOM: {last_oom}")
            _finalize_fold(out_dir, fold, manifest, completion)
            tr_ds = va_ds = pos_weight = None
            _free_cv_gpu()
        return _load_partial(out_dir)


def run_condition(condition, out_dir, warm_starts=None):
    if warm_starts:
        raise ValueError("Do not supply N2V/CNN warm starts to this no-N2V runner")
    return run_cv("SwinUNet", condition, out_dir)


def _condition_dataframe(out_dir):
    rows = _load_partial(out_dir)
    return pd.DataFrame(rows).sort_values("fold").reset_index(drop=True) if rows else pd.DataFrame()


def save_cv_tables():
    root, directory = _cv_paths()
    if not directory.exists():
        return pd.DataFrame()
    with _writer_lock(directory):
        frame = _condition_dataframe(directory)
        if not frame.empty:
            for path in (directory / "cv_results.csv", root / "swin_cv_results.csv"):
                _atomic_write(path, lambda stream: frame.to_csv(stream, index=False))
            print("Saved Swin CV tables:", root / "swin_cv_results.csv")
    return frame


def run_swin_experiment():
    global all_results, comparison_df
    cv_preflight()
    all_results, comparison_df = [], pd.DataFrame()
    try:
        if RUN_NO_N2V:
            all_results = run_condition("no_n2v", NO_N2V_DIR)
    except KeyboardInterrupt:
        print("Interrupted. Completed fold records and any resume file written by fit() are retained.")
        raise
    except Exception as error:
        timeout_type = globals().get("OutOfTime")
        if isinstance(timeout_type, type) and issubclass(timeout_type, BaseException) and isinstance(error, timeout_type):
            print("Walltime limit reached. Rerun the same cell; fit() must restore its last saved epoch boundary.")
        else:
            raise
    finally:
        try:
            comparison_df = save_cv_tables()
        except Exception as table_error:
            # Do not hide a real training failure with a secondary CSV error.
            print(f"Could not export the CSV summary: {table_error}. Check the fold JSON files.")
    if not comparison_df.empty:
        columns = [key for key in ("condition", "arch", "fold", "epoch", "epochs_run", "batch",
                                   "grad_accum", "effective_batch", "precision", "psnr", "ssim", "iou",
                                   "best_f1", "best_thresh_prob") if key in comparison_df]
        print("\n=== SWIN-UNET CV RESULTS (complete or partial) ===")
        print(comparison_df[columns].round(4).to_string(index=False))
    else:
        print("No completed Swin folds recorded yet.")
    return comparison_df


if __name__ == "__main__":
    run_swin_experiment()


In [ ]:
"""Section 9 replacement: SWIN-UNET ONLY, FIVE FOLDS, NO N2V.

Paste this complete file into Section 9, or run it in the existing notebook:
    %run -i /your/path/swin_five_fold_cv.py
It starts CV when executed in that notebook. Importing the module does not.
Run the configuration, data, Swin architecture, losses, and fit() cells first.

IMPORTANT: the uploaded Section 9 contains the CV coordinator, not fit(). This
replacement preserves the existing fit(model, train_loader, val_loader,
epochs=..., lr=..., wd=..., ckpt_path=..., resume_path=..., pos_weight=...,
tag=...) interface. That function must:
  * use CFG.GRAD_ACCUM correctly, including the last incomplete accumulation;
  * call seg_loss with the current 1-based epoch to apply Lovasz warmup;
  * restore raw model, optimizer, scheduler, scaler, EMA, history and RNG state
    from resume_path, and atomically save epoch-boundary progress;
  * save the BEST validation-IoU checkpoint at ckpt_path and return the full
    resumed history (each row has a 1-based epoch and finite iou);
  * accept the new model's (denoised, raw_logits) outputs and aux_logits=None.
This file cannot implement or verify those missing training-loop internals.
It neither changes the optimizer algorithm nor the task-loss balance.

Architecture-label note: this fit() writes its tag string into the checkpoint
"arch" field, so a genuine Swin fold is labelled "no_n2v:SwinUNet". The strict
load_state_dict into a fresh _new_swin(), and the model_config equality check,
are the real architecture guarantees here. The label itself is matched loosely
against _ARCH_ALIASES, the registry name and the class name, and a tag is split
on ":" first. An unrecognised label still raises, and now prints its value.

Checkpoint-epoch note: this fit() only rewrites its best checkpoint when the
validation IoU improves by more than its own minimum delta, so on a flat tail
the saved epoch can sit one epoch short of the history argmax (fold 2 here:
epoch 59 at 0.804976 against epoch 60 at 0.804981). _finalize_fold allows a
shortfall of CFG.CKPT_IOU_TOL, default 1e-3 in IoU units, prints the offset and
records it as iou_shortfall_vs_argmax. A larger shortfall still raises: if that
happens on a fold with a .resume file present, fit() is resetting its best
tracker on resume and fix belongs there, not in the tolerance. Reported PSNR,
SSIM and IoU always come from the epoch the saved weights actually are.

Preserved split rule: KFold(5, shuffle=True, random_state=SEED) on common in
ITS EXISTING ORDER. Do not sort/regenerate common. It must be the same CNN
train+validation pool, excluding the held-out test set. A sample-count check
does not establish dataset independence. Existing group-aware CNN splits
must not be replaced with this image-level KFold protocol.

Outputs go ONLY under CFG.OUT_DIR/swin_cv (or CFG.SWIN_CV_ROOT):
    experiment_no_n2v/SwinUNet_fold1.pt ... SwinUNet_fold5.pt
    experiment_no_n2v/SwinUNet_fold1.pt.resume ... (retained, not deleted)
    experiment_no_n2v/cv_results_partial.json, cv_results.csv
    swin_cv_results.csv
The manifest saves ordered basenames, exact indices, model settings and key
training settings. Incompatible existing progress is rejected, not erased.
Best checkpoints gain model_config and Swin CV metadata for inference.

Only no-N2V is supported here. CNN N2V checkpoints must never be loaded into
Swin. A future with-N2V comparison also needs pretraining/split-leakage checks.

References:
https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html
https://docs.pytorch.org/docs/2.1/notes/cuda.html
https://docs.pytorch.org/docs/2.1/notes/amp_examples.html
"""

from collections.abc import Mapping
from contextlib import contextmanager
import gc
import hashlib
import inspect
import json
import math
import os
from pathlib import Path
import tempfile
import traceback

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
import torch
from torch.utils.data import DataLoader


RUN_NO_N2V = True
RUN_WITH_N2V = False
RUN_ARCHS = ["SwinUNet"]

SUPERVISED_BATCH_START = {"SwinUNet": 8}
SUPERVISED_BATCH_MIN = 1
TARGET_EFFECTIVE_BATCH = 32
EVAL_BATCH_MAX = 4
EXPECTED_PAIRED_SAMPLES = 14364  # Original train+validation pool, not the test set.
SWIN_SEED_OFFSET = 3000        # Stable even if CFG.ARCHS is later reordered.
CV_FORMAT_VERSION = 1

# Free-text checkpoint labels accepted as this architecture, normalized (see
# _normalize_arch). Add a value here only after confirming that the strict
# state_dict load and the model_config comparison both pass for that file.
_ARCH_ALIASES = {"swinunet"}


def _cv_paths():
    if "CFG" not in globals() or not hasattr(CFG, "OUT_DIR"):
        raise RuntimeError("Run the configuration cell first (CFG.OUT_DIR is missing)")
    root = Path(getattr(CFG, "SWIN_CV_ROOT", Path(CFG.OUT_DIR) / "swin_cv")).expanduser().resolve()
    return root, root / "experiment_no_n2v"


if "CFG" in globals() and hasattr(CFG, "OUT_DIR"):
    CV_ROOT, NO_N2V_DIR = map(str, _cv_paths())
    PARALLEL_ROOT = CV_ROOT  # Compatibility alias; never points to the CNN root.
    WITH_N2V_DIR = str(Path(CV_ROOT) / "experiment_with_n2v")  # Reserved, not created.


def _jsonable(value):
    if isinstance(value, Mapping):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, np.ndarray):
        return _jsonable(value.tolist())
    if isinstance(value, np.generic):
        return _jsonable(value.item())
    if torch.is_tensor(value):
        if value.numel() != 1:
            raise TypeError("CV history/config should not contain non-scalar tensors")
        return _jsonable(value.detach().cpu().item())
    if isinstance(value, (Path, torch.device, torch.dtype)):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None  # Optional NaN metrics are JSON null; IoU is checked separately.
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    raise TypeError(f"Cannot serialize CV value of type {type(value).__name__}")


def _digest(value):
    payload = json.dumps(_jsonable(value), sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _file_digest(path):
    result = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def _atomic_write(path, writer, binary=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    mode = "wb" if binary else "w"
    kwargs = {} if binary else {"encoding": "utf-8"}
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode=mode, dir=path.parent,
                                         prefix=f".{path.name}.", suffix=".tmp",
                                         delete=False, **kwargs) as stream:
            temporary = Path(stream.name)
            writer(stream)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, path)
    finally:
        # Only our own uncommitted temporary file can be removed here.
        if temporary is not None and temporary.exists():
            temporary.unlink()


def _swin_atomic_json(value, path):
    clean = _jsonable(value)
    _atomic_write(path, lambda stream: json.dump(clean, stream, indent=2, allow_nan=False))


def _read_json(path):
    try:
        with open(path, encoding="utf-8") as stream:
            return json.load(stream)
    except (OSError, ValueError) as error:
        raise RuntimeError(f"Cannot read progress file {path}; refusing to restart or overwrite it") from error


@contextmanager
def _writer_lock(out_dir):
    """Linux/HPC advisory lock; stale lock files themselves do not hold a lock."""
    import fcntl
    directory = Path(out_dir)
    directory.mkdir(parents=True, exist_ok=True)
    with open(directory / ".swin_cv.lock", "a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError(f"Another Swin CV writer is active in {directory}") from error
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


def _results_json(out_dir):
    return str(Path(out_dir) / "cv_results_partial.json")


def _load_partial(out_dir):
    path = Path(_results_json(out_dir))
    if not path.exists():
        return []
    rows = _read_json(path)
    if not isinstance(rows, list):
        raise RuntimeError(f"Expected a JSON list in {path}")
    seen = set()
    for row in rows:
        if not isinstance(row, dict) or row.get("arch") != "SwinUNet" or row.get("condition") != "no_n2v":
            raise RuntimeError(f"Non-Swin or cross-condition records in {path}; use a separate Swin directory")
        fold = row.get("fold")
        if not isinstance(fold, int) or not 1 <= fold <= 5 or fold in seen:
            raise RuntimeError(f"Invalid/duplicate fold records in {path}")
        seen.add(fold)
    return rows


def _read_checkpoint(path):
    # Reuse the notebook's established loader when supplied. Its inputs here
    # are checkpoints from this experiment, never arbitrary downloaded files.
    helper = globals().get("_load_torch_file")
    if callable(helper):
        return helper(str(path), map_location="cpu")
    # No automatic unsafe-pickle fallback for checkpoints with custom objects.
    return torch.load(path, map_location="cpu", weights_only=True)


def _extract_state_dict(payload, prefer_ema=True):
    def is_state(value):
        return isinstance(value, Mapping) and bool(value) and all(torch.is_tensor(v) for v in value.values())
    if is_state(payload):
        state = dict(payload)
    elif isinstance(payload, Mapping):
        order = (("model", "ema", "raw", "state_dict", "model_state_dict") if prefer_ema else
                 ("raw", "model", "state_dict", "model_state_dict", "ema"))
        state = next((dict(payload[k]) for k in order if k in payload and is_state(payload[k])), None)
        if state is None:
            raise TypeError("Checkpoint does not contain a supported model state_dict")
    else:
        raise TypeError("Checkpoint must contain a mapping")
    while state:
        prefix = next((p for p in ("module.", "_orig_mod.")
                       if all(str(k).startswith(p) for k in state)), None)
        if prefix is None:
            break
        state = {k[len(prefix):]: v for k, v in state.items()}
    return state


def _normalize_arch(label):
    return str(label).strip().replace("_", "").replace("-", "").lower()


def _arch_label_matches(label, class_name="SwinUNet"):
    """Match the free-text checkpoint label, not the architecture itself.

    fit() writes its tag string into "arch", so a genuine fold from this runner
    is labelled "no_n2v:SwinUNet". Tags are split on ":" and each part is
    compared against the aliases, the registry name and the model class name.
    """
    if not isinstance(label, str):
        return False
    allowed = set(_ARCH_ALIASES)
    allowed.add(_normalize_arch(class_name))
    allowed.add(_normalize_arch("SwinUNet"))
    return any(_normalize_arch(part) in allowed for part in label.split(":"))


def load_model_weights(path, model, prefer_ema=True):
    """Strict same-architecture load; never accept an 80%-matching CNN state."""
    if not path or not Path(path).is_file():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    payload = _read_checkpoint(path)
    if isinstance(payload, Mapping) and "model_config" in payload and hasattr(model, "model_config"):
        if _jsonable(payload["model_config"]) != _jsonable(model.model_config):
            raise RuntimeError("Checkpoint model_config does not match this Swin model")
    model.load_state_dict(_extract_state_dict(payload, prefer_ema), strict=True)
    return model


def _new_swin():
    options = dict(in_ch=1, img_size=CFG.IMG_SIZE,
                   patch_size=getattr(CFG, "SWIN_PATCH_SIZE", 4),
                   window_size=getattr(CFG, "SWIN_WINDOW_SIZE", 8),
                   embed_dim=getattr(CFG, "SWIN_EMBED_DIM", 56),
                   depths=getattr(CFG, "SWIN_DEPTHS", (2, 2, 2, 2)),
                   num_heads=getattr(CFG, "SWIN_HEADS", (2, 4, 8, 16)),
                   use_checkpoint=getattr(CFG, "SWIN_USE_CHECKPOINT", False))
    options.update(getattr(CFG, "SWIN_MODEL_KWARGS", {}))
    model = ARCH_REGISTRY["SwinUNet"](**options)
    if not isinstance(getattr(model, "model_config", None), dict):
        raise RuntimeError("Use the supplied SwinUNet architecture cell with model_config metadata")
    return model


def _check_dependencies():
    required = ("CFG", "SEED", "common", "noisy_map", "clean_map", "mask_map",
                "gauss_map", "ARCH_REGISTRY", "TEMSegDataset", "loader_options",
                "estimate_pos_weight", "seed_everything", "fit")
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Run the earlier notebook cells first. Missing: " + ", ".join(missing))
    required_cfg = ("OUT_DIR", "IMG_SIZE", "N_FOLDS", "EPOCHS", "LR", "WD", "PATIENCE", "PRECISION", "LAM_GRAD")
    absent = [name for name in required_cfg if not hasattr(CFG, name)]
    if absent:
        raise RuntimeError("Missing existing experiment settings: " + ", ".join("CFG." + n for n in absent))
    if "SwinUNet" not in ARCH_REGISTRY:
        raise RuntimeError("Run the Swin architecture cell before this CV cell")
    if RUN_ARCHS != ["SwinUNet"] or RUN_WITH_N2V:
        raise RuntimeError("This first Swin experiment is no-N2V only: RUN_ARCHS=['SwinUNet'], RUN_WITH_N2V=False")
    # Validate the call signature, not the unprovided function's internals.
    try:
        inspect.signature(fit).bind(object(), object(), object(), epochs=1,
                                    lr=1e-4, wd=0.0, ckpt_path="best.pt",
                                    resume_path="best.pt.resume", pos_weight=None, tag="no_n2v:SwinUNet")
    except (TypeError, ValueError) as error:
        raise RuntimeError("fit() must accept the resume/epoch/checkpoint arguments used by your original Section 9") from error
    fit_globals = getattr(fit, "__globals__", {})
    if "CFG" in fit_globals and fit_globals["CFG"] is not CFG:
        raise RuntimeError("fit() and this runner use different CFG objects; paste the cell or use %run -i")


def _validate_pool(n_folds):
    if n_folds != 5:
        raise ValueError("This experiment requires exactly five folds; retain CFG.N_FOLDS=5")
    if getattr(CFG, "MAX_SAMPLES", None) is not None:
        raise RuntimeError("CFG.MAX_SAMPLES must be None for the full comparison")
    names = list(common)  # Deliberately NOT sorted.
    if not all(isinstance(name, str) for name in names) or len(set(names)) != len(names):
        raise RuntimeError("common must contain unique string basenames in the original CNN order")
    expected = int(getattr(CFG, "EXPECTED_PAIRED_SAMPLES", EXPECTED_PAIRED_SAMPLES))
    if len(names) != expected or len(names) < n_folds:
        raise RuntimeError(f"Expected {expected} paired training/validation samples, found {len(names)}")
    for label, mapping in (("noisy", noisy_map), ("clean", clean_map), ("mask", mask_map)):
        missing = [name for name in names if mapping.get(name) is None]
        if missing:
            raise RuntimeError(f"Missing {label} paths for {len(missing)} paired samples; first: {missing[:3]}")
    return names


def _cpu_allocation():
    available = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
    try:
        allocated = int(os.environ.get("SLURM_CPUS_PER_TASK", available))
    except ValueError:
        allocated = available
    return max(1, min(allocated, available))


def cv_preflight():
    global DEVICE, CV_ROOT, NO_N2V_DIR, WITH_N2V_DIR, PARALLEL_ROOT
    _check_dependencies()
    _validate_pool(int(CFG.N_FOLDS))
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU is visible. Run this training cell inside your allocated SLURM GPU job")
    logical_index = int(getattr(CFG, "SWIN_DEVICE_INDEX", 0))
    if not 0 <= logical_index < torch.cuda.device_count():
        raise RuntimeError("CFG.SWIN_DEVICE_INDEX is outside the visible CUDA devices")
    # Respect scheduler visibility: never rewrite CUDA_VISIBLE_DEVICES after import.
    DEVICE = torch.device("cuda", logical_index)
    torch.cuda.set_device(DEVICE)
    workers = max(0, min(int(getattr(CFG, "NUM_WORKERS", 4)), _cpu_allocation() - 1, 8))
    CFG.NUM_WORKERS = workers
    CFG.ARCHS = ["SwinUNet"]
    CV_ROOT, NO_N2V_DIR = map(str, _cv_paths())
    PARALLEL_ROOT = CV_ROOT
    WITH_N2V_DIR = str(Path(CV_ROOT) / "experiment_with_n2v")
    print("\n=== SWIN-UNET 5-FOLD CV: NO N2V ===")
    print(f"Samples: {len(common)} | folds: {CFG.N_FOLDS} | split seed: {SEED}")
    print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
    print(f"Using {DEVICE}: {torch.cuda.get_device_name(logical_index)} | loader workers: {workers}")
    print(f"Epoch limit: {CFG.EPOCHS} | LR: {getattr(CFG, 'SWIN_LR', CFG.LR)} | precision: {CFG.PRECISION}")
    print(f"Micro-batches: {_supervised_batch_candidates('SwinUNet')} | nominal effective batch: {TARGET_EFFECTIVE_BATCH}")
    print("Swin output:", NO_N2V_DIR)
    print("Existing CNN directories and checkpoints are not modified.")
    print("fit() must handle accumulation, epoch-aware loss, best-IoU saving and full-state resume.")


def _supervised_batch_candidates(arch_name):
    if arch_name != "SwinUNet":
        raise ValueError("This runner only trains SwinUNet")
    start, floor = int(SUPERVISED_BATCH_START[arch_name]), int(SUPERVISED_BATCH_MIN)
    if not 1 <= floor <= start or TARGET_EFFECTIVE_BATCH < 1 or EVAL_BATCH_MAX < 1:
        raise ValueError("Invalid micro-batch/accumulation policy")
    values = []
    while True:
        values.append(start)
        if start == floor:
            return values
        start = max(floor, start // 2)


def _eval_batch_for_arch(arch_name, train_batch):
    return max(1, min(int(EVAL_BATCH_MAX), int(train_batch)))


def _grad_accum_for_batch(batch):
    return max(1, math.ceil(TARGET_EFFECTIVE_BATCH / batch))


def _is_cuda_oom(error):
    return isinstance(error, torch.cuda.OutOfMemoryError) or (
        "cuda" in str(error).lower() and "out of memory" in str(error).lower())


def _free_cv_gpu():
    globals().pop("_EVAL_CACHE", None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _make_manifest(names, n_folds, epochs):
    folds = [dict(fold=i, train_indices=train.tolist(), val_indices=val.tolist())
             for i, (train, val) in enumerate(KFold(n_splits=n_folds, shuffle=True,
                                                    random_state=int(SEED)).split(names), 1)]
    # Capture only task-relevant settings, never arbitrary environment/CFG secrets.
    keys = ("IMG_SIZE", "PATIENCE", "PRECISION", "LAM_GRAD", "LAM_SSIM", "W_BCE",
            "W_DICE", "W_JACCARD", "LOVASZ_WEIGHT", "LOVASZ_WARMUP_EPOCHS", "EDGE_WEIGHT",
            "POS_WEIGHT_MAX", "W_SEG", "W_DEN", "LAM_SEG", "LAM_DENOISE", "AUX_WEIGHT",
            "EMA_DECAY", "USE_EMA", "WARMUP_EPOCHS", "MIN_LR", "GRAD_CLIP", "CLIP_GRAD")
    settings = {key: getattr(CFG, key) for key in keys if hasattr(CFG, key)}
    settings.update(epochs=epochs, lr=getattr(CFG, "SWIN_LR", CFG.LR),
                    wd=getattr(CFG, "SWIN_WD", CFG.WD), effective_batch=TARGET_EFFECTIVE_BATCH)
    # Source paths catch a different pairing/root. Image content changed in place
    # is not hashed here; keep a versioned, immutable dataset for comparison.
    sources = [(name, str(noisy_map[name]), str(clean_map[name]), str(mask_map[name]),
                str(gauss_map[name]) if gauss_map.get(name) is not None else None) for name in names]
    with torch.random.fork_rng(devices=[]):
        model = _new_swin()
        model_config = _jsonable(model.model_config)
        shape_spec = {key: list(value.shape) for key, value in model.state_dict().items()}
        params = sum(p.numel() for p in model.parameters())
    specification = dict(format_version=CV_FORMAT_VERSION, arch="SwinUNet", condition="no_n2v",
                         seed=int(SEED), fold_seed_offset=SWIN_SEED_OFFSET, n_folds=n_folds,
                         basenames=names, folds=folds, sources_sha256=_digest(sources),
                         model_config=model_config, state_shapes=shape_spec, params=params,
                         training_settings=_jsonable(settings), run_tag=str(getattr(CFG, "SWIN_RUN_TAG", "v1")))
    return {"run_id": _digest(specification), "specification": specification}


def _ensure_manifest(out_dir, wanted):
    path = Path(out_dir) / "swin_cv_manifest.json"
    if path.exists():
        existing = _read_json(path)
        if existing != wanted:
            raise RuntimeError("Swin CV manifest mismatch: samples/order, splits, model or training settings changed. "
                               "Restore the original settings, or explicitly choose a NEW CFG.SWIN_CV_ROOT")
    else:
        occupied = [p.name for p in Path(out_dir).iterdir() if p.name != ".swin_cv.lock"]
        if occupied:
            raise RuntimeError(f"Existing files without a Swin manifest in {out_dir}: {occupied[:5]}. "
                               "Refusing to adopt/overwrite unknown progress")
        _swin_atomic_json(wanted, path)


def _fold_paths(out_dir, fold):
    checkpoint_path = Path(out_dir) / f"SwinUNet_fold{fold}.pt"
    return checkpoint_path, Path(str(checkpoint_path) + ".resume"), Path(out_dir) / f"SwinUNet_fold{fold}.attempt.json", Path(out_dir) / f"SwinUNet_fold{fold}.fit_complete.json"


def _best_history(history):
    if not isinstance(history, (list, tuple)) or not history:
        raise RuntimeError("fit() must return a nonempty full list of epoch records")
    epochs, finite_rows = [], []
    for row in history:
        if not isinstance(row, Mapping) or "epoch" not in row or "iou" not in row:
            raise RuntimeError("Every fit history row must contain epoch and iou")
        epoch = int(row["epoch"])
        if epoch < 1 or epoch != row["epoch"]:
            raise RuntimeError("fit history must use 1-based integer epochs")
        epochs.append(epoch)
        if row["iou"] is not None and math.isfinite(float(row["iou"])):
            finite_rows.append(row)
    if epochs != sorted(set(epochs)) or not finite_rows:
        raise RuntimeError("fit history has duplicate/out-of-order epochs or no finite validation IoU")
    return dict(max(finite_rows, key=lambda row: float(row["iou"])))


def _finalize_fold(out_dir, fold, manifest, completion):
    run_id = manifest["run_id"]
    if completion.get("run_id") != run_id or completion.get("fold") != fold:
        raise RuntimeError("Fold completion marker belongs to a different experiment")
    best = _best_history(completion["history"])
    checkpoint_path, resume_path, _, _ = _fold_paths(out_dir, fold)
    if not checkpoint_path.is_file() or checkpoint_path.stat().st_size == 0:
        raise RuntimeError("fit() returned but the best checkpoint is missing/empty")
    payload = _read_checkpoint(checkpoint_path)
    # The strict load below is the real architecture check: it fails on any
    # parameter-name or shape difference. The "arch" string is only a label.
    with torch.random.fork_rng(devices=[]):
        model = _new_swin()
        model.load_state_dict(_extract_state_dict(payload), strict=True)
        swin_class_name = type(model).__name__
    is_plain = isinstance(payload, Mapping) and all(torch.is_tensor(v) for v in payload.values())
    payload = {"model": dict(payload)} if is_plain else dict(payload)
    if "arch" in payload and not _arch_label_matches(payload["arch"], swin_class_name):
        raise RuntimeError(
            f"Saved checkpoint arch label {payload['arch']!r} is not recognised as SwinUNet "
            f"(accepted: 'SwinUNet', {swin_class_name!r}, or a tag containing either). "
            "The weights did load strictly into this Swin model, so if that label is just how "
            "fit() writes its tag, add it to _ARCH_ALIASES; if it names a CNN, fit() is copying "
            "a stale global and should derive arch from the model it was passed.")
    model_config = manifest["specification"]["model_config"]
    if "model_config" in payload and _jsonable(payload["model_config"]) != model_config:
        raise RuntimeError("Saved checkpoint model_config mismatch")
    verified_epoch = "epoch" in payload
    if verified_epoch:
        # fit() only rewrites its checkpoint when validation IoU improves by
        # more than its own minimum delta, so on a flat tail the saved epoch can
        # sit just short of the history argmax. Allow a shortfall in IoU units;
        # a real wrong-epoch checkpoint is orders of magnitude further off.
        tolerance = float(getattr(CFG, "CKPT_IOU_TOL", 1e-3))
        argmax_epoch, argmax_iou = int(best["epoch"]), float(best["iou"])
        saved_row = next((row for row in completion["history"]
                          if int(row["epoch"]) == int(payload["epoch"])), None)
        if saved_row is None or saved_row["iou"] is None or not math.isfinite(float(saved_row["iou"])):
            raise RuntimeError(f"Best checkpoint reports epoch {payload['epoch']}, which has no finite "
                               "validation IoU in the history fit() returned. "
                               "Inspect fit() checkpoint selection before recording this fold")
        shortfall = argmax_iou - float(saved_row["iou"])
        if shortfall > tolerance:
            raise RuntimeError(
                f"Best-checkpoint epoch disagrees with best validation-IoU history: checkpoint holds "
                f"epoch {int(saved_row['epoch'])} (IoU {float(saved_row['iou']):.6f}) but the history "
                f"argmax is epoch {argmax_epoch} (IoU {argmax_iou:.6f}), a shortfall of {shortfall:.3e} "
                f"above the tolerance {tolerance:g}. If fit() resets its best tracker on resume it will "
                "overwrite a good checkpoint with a later worse one; fix that rather than raising "
                "CFG.CKPT_IOU_TOL")
        if shortfall > 0:
            print(f"  note: checkpoint holds epoch {int(saved_row['epoch'])} (IoU {float(saved_row['iou']):.6f}); "
                  f"history argmax is epoch {argmax_epoch} (IoU {argmax_iou:.6f}); "
                  f"shortfall {shortfall:.3e} within tolerance {tolerance:g}")
        # Report PSNR/SSIM from the epoch actually represented by its checkpoint.
        best = dict(saved_row)
        best.update(history_argmax_epoch=argmax_epoch, history_argmax_iou=argmax_iou,
                    iou_shortfall_vs_argmax=shortfall, ckpt_iou_tol=tolerance)
    payload.update(arch="SwinUNet", model_config=model_config,
                   swin_cv=dict(run_id=run_id, condition="no_n2v", fold=fold,
                                selected_history_epoch=int(best["epoch"]),
                                checkpoint_epoch_verified=verified_epoch))
    _atomic_write(checkpoint_path, lambda stream: torch.save(payload, stream), binary=True)
    if not verified_epoch:
        print("  note: checkpoint had no epoch metadata; history/checkpoint epoch alignment is unverified")
    best.update(arch="SwinUNet", condition="no_n2v", fold=fold, run_id=run_id,
                epochs_run=len(completion["history"]), last_epoch=int(completion["history"][-1]["epoch"]),
                batch=completion["batch"], eval_batch=completion["eval_batch"],
                grad_accum=completion["grad_accum"], effective_batch=completion["batch"] * completion["grad_accum"],
                oom_retries=completion["oom_retries"], precision=str(CFG.PRECISION),
                params_M=manifest["specification"]["params"] / 1e6,
                checkpoint=str(checkpoint_path), checkpoint_sha256=_file_digest(checkpoint_path),
                checkpoint_epoch_verified=verified_epoch, n2v_checkpoint=None,
                resume_retained=resume_path.is_file())
    rows = [row for row in _load_partial(out_dir) if row["fold"] != fold]
    rows.append(_jsonable(best))
    rows.sort(key=lambda row: row["fold"])
    _swin_atomic_json(rows, _results_json(out_dir))
    print(f"  recorded fold {fold}: IoU {float(best['iou']):.4f} at epoch {best['epoch']}")
    return best


def run_cv(arch_name="SwinUNet", condition="no_n2v", out_dir=None,
           n_folds=None, epochs=None, warm_start=None):
    """Train/skip/recover all five folds; fit() owns actual epoch restoration.
    OOM batch changes preserve only the NOMINAL effective batch, not bitwise
    trajectories. fit() must support the resulting loader/accumulation change.
    """
    _check_dependencies()
    if arch_name != "SwinUNet" or condition != "no_n2v" or warm_start is not None:
        raise ValueError("This runner is exclusively SwinUNet from random initialization (no_n2v)")
    if "DEVICE" not in globals():
        raise RuntimeError("Run cv_preflight() first to select the allocated device")
    n_folds = int(CFG.N_FOLDS if n_folds is None else n_folds)
    epochs = int(CFG.EPOCHS if epochs is None else epochs)
    if epochs < 1:
        raise ValueError("epochs must be positive")
    names = _validate_pool(n_folds)
    root, default_out = _cv_paths()
    out_dir = Path(out_dir).expanduser().resolve() if out_dir is not None else default_out
    if not out_dir.is_relative_to(root) or out_dir == root:
        raise RuntimeError("The condition directory must be inside the dedicated CFG.SWIN_CV_ROOT")
    with _writer_lock(out_dir):
        manifest = _make_manifest(names, n_folds, epochs)
        _ensure_manifest(out_dir, manifest)
        recorded = {row["fold"]: row for row in _load_partial(out_dir)}
        print(f"\nSwinUNet | five folds | <= {epochs} epochs/fold | {manifest['specification']['params']/1e6:.3f} M parameters")
        for split in manifest["specification"]["folds"]:
            fold = split["fold"]
            ckpt, resume, attempt_path, complete_path = _fold_paths(out_dir, fold)
            if fold in recorded:
                row = recorded[fold]
                if row.get("run_id") != manifest["run_id"] or not ckpt.is_file():
                    raise RuntimeError(f"Fold {fold} completion record is incompatible or its checkpoint is missing")
                if row.get("checkpoint_sha256") != _file_digest(ckpt):
                    raise RuntimeError(f"Fold {fold} checkpoint was changed/corrupted after recording")
                print(f"-- fold {fold}/5 already complete; skipping")
                continue
            if complete_path.is_file():
                print(f"-- fold {fold}/5 recovering a returned fit() without retraining")
                _finalize_fold(out_dir, fold, manifest, _read_json(complete_path))
                continue
            previous = _read_json(attempt_path) if attempt_path.is_file() else None
            if (ckpt.exists() or resume.exists()) and previous is None:
                raise RuntimeError(f"Fold {fold} has untracked checkpoints; refusing to overwrite/adopt them")
            if previous and (previous.get("run_id") != manifest["run_id"] or previous.get("fold") != fold):
                raise RuntimeError(f"Fold {fold} resume metadata mismatch")
            if ckpt.exists() and not resume.exists():
                raise RuntimeError(f"Fold {fold} has a best checkpoint but no full resume/completion marker. "
                                   "Inspect fit() resume saving before restarting this fold")
            tr_names = [names[i] for i in split["train_indices"]]
            va_names = [names[i] for i in split["val_indices"]]
            fold_seed = int(SEED) + SWIN_SEED_OFFSET + fold
            seed_everything(fold_seed)
            tr_ds = TEMSegDataset(tr_names, noisy_map, clean_map, mask_map, gauss_map=gauss_map, train=True)
            va_ds = TEMSegDataset(va_names, noisy_map, clean_map, mask_map, gauss_map=gauss_map, train=False)
            pos_weight = estimate_pos_weight(tr_names)  # Never validation names.
            candidates = _supervised_batch_candidates(arch_name)
            if previous:
                candidates = [b for b in candidates if b <= int(previous["batch"])]
            if not candidates:
                raise RuntimeError("Current batch policy is incompatible with the recorded attempt")
            history, completion, last_oom = None, None, None
            oom_retries = int(previous.get("oom_retries", 0)) if previous else 0
            print(f"\n-- fold {fold}/5 | train {len(tr_names)} | validation {len(va_names)} | seed {fold_seed}")
            for batch in candidates:
                eval_batch, accum = _eval_batch_for_arch(arch_name, batch), _grad_accum_for_batch(batch)
                seed_everything(fold_seed)
                _free_cv_gpu()
                had_accum = hasattr(CFG, "GRAD_ACCUM")
                old_accum = getattr(CFG, "GRAD_ACCUM", None)
                model = tr_dl = va_dl = None
                try:
                    tr_options = dict(loader_options(True, drop_last=True, batch=batch))
                    va_options = dict(loader_options(False, batch=eval_batch))
                    tr_dl, va_dl = DataLoader(tr_ds, **tr_options), DataLoader(va_ds, **va_options)
                    if len(tr_dl) == 0 or len(va_dl) == 0:
                        raise RuntimeError("Empty train/validation loader; check sample count and batch size")
                    if tr_dl.batch_size != batch or va_dl.batch_size != eval_batch or va_dl.drop_last:
                        raise RuntimeError("loader_options ignored the requested batch or drops validation samples")
                    model = _new_swin()
                    CFG.GRAD_ACCUM = accum
                    attempt = dict(run_id=manifest["run_id"], fold=fold, seed=fold_seed,
                                   batch=batch, eval_batch=eval_batch, grad_accum=accum, oom_retries=oom_retries)
                    _swin_atomic_json(attempt, attempt_path)
                    print(f"  batch {batch}, accumulation {accum}, nominal effective {batch*accum}, val batch {eval_batch}")
                    if resume.is_file():
                        print(f"  passing {resume.name} to fit() for full-state restoration")
                    history = fit(model, tr_dl, va_dl, epochs=epochs,
                                  lr=getattr(CFG, "SWIN_LR", CFG.LR), wd=getattr(CFG, "SWIN_WD", CFG.WD),
                                  ckpt_path=str(ckpt), resume_path=str(resume), pos_weight=pos_weight,
                                  tag="no_n2v:SwinUNet")
                    _best_history(history)
                    completion = dict(attempt, history=_jsonable(history))
                    _swin_atomic_json(completion, complete_path)
                    break
                except Exception as error:
                    if not _is_cuda_oom(error):
                        raise
                    last_oom = str(error)  # Do NOT retain an exception with GPU tensors in its traceback.
                    traceback.clear_frames(error.__traceback__)
                    oom_retries += 1
                    history = None
                    if attempt_path.exists():
                        attempt = _read_json(attempt_path)
                        attempt["oom_retries"] = oom_retries
                        _swin_atomic_json(attempt, attempt_path)
                    print(f"  CUDA OOM at batch {batch}; retrying a smaller batch if available")
                finally:
                    if had_accum:
                        CFG.GRAD_ACCUM = old_accum
                    elif hasattr(CFG, "GRAD_ACCUM"):
                        delattr(CFG, "GRAD_ACCUM")
                    model = tr_dl = va_dl = None
                    _free_cv_gpu()
            if completion is None:
                raise RuntimeError(f"All permitted batches {candidates} failed for fold {fold}. Last CUDA OOM: {last_oom}")
            _finalize_fold(out_dir, fold, manifest, completion)
            tr_ds = va_ds = pos_weight = None
            _free_cv_gpu()
        return _load_partial(out_dir)


def run_condition(condition, out_dir, warm_starts=None):
    if warm_starts:
        raise ValueError("Do not supply N2V/CNN warm starts to this no-N2V runner")
    return run_cv("SwinUNet", condition, out_dir)


def _condition_dataframe(out_dir):
    rows = _load_partial(out_dir)
    return pd.DataFrame(rows).sort_values("fold").reset_index(drop=True) if rows else pd.DataFrame()


def save_cv_tables():
    root, directory = _cv_paths()
    if not directory.exists():
        return pd.DataFrame()
    with _writer_lock(directory):
        frame = _condition_dataframe(directory)
        if not frame.empty:
            for path in (directory / "cv_results.csv", root / "swin_cv_results.csv"):
                _atomic_write(path, lambda stream: frame.to_csv(stream, index=False))
            print("Saved Swin CV tables:", root / "swin_cv_results.csv")
    return frame


def run_swin_experiment():
    global all_results, comparison_df
    cv_preflight()
    all_results, comparison_df = [], pd.DataFrame()
    try:
        if RUN_NO_N2V:
            all_results = run_condition("no_n2v", NO_N2V_DIR)
    except KeyboardInterrupt:
        print("Interrupted. Completed fold records and any resume file written by fit() are retained.")
        raise
    except Exception as error:
        timeout_type = globals().get("OutOfTime")
        if isinstance(timeout_type, type) and issubclass(timeout_type, BaseException) and isinstance(error, timeout_type):
            print("Walltime limit reached. Rerun the same cell; fit() must restore its last saved epoch boundary.")
        else:
            raise
    finally:
        try:
            comparison_df = save_cv_tables()
        except Exception as table_error:
            # Do not hide a real training failure with a secondary CSV error.
            print(f"Could not export the CSV summary: {table_error}. Check the fold JSON files.")
    if not comparison_df.empty:
        columns = [key for key in ("condition", "arch", "fold", "epoch", "epochs_run", "batch",
                                   "grad_accum", "effective_batch", "precision", "psnr", "ssim", "iou",
                                   "best_f1", "best_thresh_prob") if key in comparison_df]
        print("\n=== SWIN-UNET CV RESULTS (complete or partial) ===")
        print(comparison_df[columns].round(4).to_string(index=False))
    else:
        print("No completed Swin folds recorded yet.")
    return comparison_df


if __name__ == "__main__":
    run_swin_experiment()

In [ ]:
# ============================================================================
# CENTROID DETECTION METRICS (LOC_CFG -- does not collide with class CFG)  --  drop-in notebook cell
#
# Scores a segmentation head by *atom-column localization* instead of pixel
# overlap.  A predicted column counts as a true positive if its centroid lies
# within `tol` pixels (or Angstrom) of a ground-truth column centroid, under a
# one-to-one optimal assignment.  Insensitive to gaussianMask width, unlike IoU.
#
# Reports: precision / recall / F1, localization RMSE and MAE over matched
# pairs, plus tolerance and threshold sweeps.
# ============================================================================

import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------
LOC_CFG = dict(
    min_distance   = 3,       # px; minimum separation of two distinct columns
    prob_threshold = 0.50,    # binarisation / peak floor on the sigmoid map
    tol_px         = 2.0,     # matching tolerance, pixels
    tol_sweep      = (0.5, 1.0, 1.5, 2.0, 3.0, 4.0),
    thr_sweep      = (0.30, 0.40, 0.50, 0.56, 0.60, 0.70),
    border         = 4,       # px; drop peaks this close to the frame edge
    subpixel       = True,    # parabolic refinement of each peak
    pixel_size     = None,    # Angstrom/px.  None -> all distances stay in px
    gt_is_binary   = None,    # None -> auto-detect from the mask histogram
)


# ---------------------------------------------------------------------------
# 1.  PEAK EXTRACTION
# ---------------------------------------------------------------------------
def _parabolic_refine(img, coords):
    """Sub-pixel peak refinement by separable 3-point parabolic fit."""
    h, w = img.shape
    out = coords.astype(np.float64).copy()
    for k, (r, c) in enumerate(coords):
        if not (0 < r < h - 1 and 0 < c < w - 1):
            continue
        # row direction
        a, b, d = img[r - 1, c], img[r, c], img[r + 1, c]
        den = a - 2.0 * b + d
        if abs(den) > 1e-12:
            out[k, 0] += np.clip(0.5 * (a - d) / den, -0.5, 0.5)
        # column direction
        a, b, d = img[r, c - 1], img[r, c], img[r, c + 1]
        den = a - 2.0 * b + d
        if abs(den) > 1e-12:
            out[k, 1] += np.clip(0.5 * (a - d) / den, -0.5, 0.5)
    return out


def peaks_from_prob(prob, min_distance=3, threshold=0.5, border=4, subpixel=True):
    """Predicted column centroids from a sigmoid probability map (H, W)."""
    prob = np.asarray(prob, dtype=np.float64)
    coords = peak_local_max(
        prob,
        min_distance=min_distance,
        threshold_abs=threshold,
        exclude_border=False,
    )
    if coords.size == 0:
        return np.empty((0, 2))
    pts = _parabolic_refine(prob, coords) if subpixel else coords.astype(np.float64)
    return _drop_border(pts, prob.shape, border)


def peaks_from_mask(mask, min_distance=3, border=4, is_binary=None, subpixel=True):
    """Ground-truth centroids from a gaussianMask (soft) or a binary mask."""
    mask = np.asarray(mask, dtype=np.float64)
    if mask.max() <= 0:
        return np.empty((0, 2))
    m = mask / mask.max()

    if is_binary is None:                      # auto-detect
        mid = np.mean((m > 0.05) & (m < 0.95))
        is_binary = mid < 0.02

    if is_binary:
        lab, n = ndi.label(m > 0.5)
        if n == 0:
            return np.empty((0, 2))
        pts = np.array(ndi.center_of_mass(m > 0.5, lab, range(1, n + 1)))
    else:
        coords = peak_local_max(
            m, min_distance=min_distance, threshold_abs=0.30, exclude_border=False
        )
        if coords.size == 0:
            return np.empty((0, 2))
        pts = _parabolic_refine(m, coords) if subpixel else coords.astype(np.float64)

    return _drop_border(pts, mask.shape, border)


def _drop_border(pts, shape, border):
    if border <= 0 or len(pts) == 0:
        return pts
    h, w = shape
    keep = (
        (pts[:, 0] >= border) & (pts[:, 0] <= h - 1 - border)
        & (pts[:, 1] >= border) & (pts[:, 1] <= w - 1 - border)
    )
    return pts[keep]


# ---------------------------------------------------------------------------
# 2.  ONE-TO-ONE MATCHING
# ---------------------------------------------------------------------------
def match_points(pred, true, tol):
    """
    Optimal one-to-one assignment restricted to pairs within `tol`.
    Returns (matched_distances, n_tp, n_fp, n_fn).
    """
    n_p, n_t = len(pred), len(true)
    if n_p == 0 or n_t == 0:
        return np.empty(0), 0, n_p, n_t

    # KD-tree prefilter so the Hungarian solve stays small on dense frames
    tree = cKDTree(true)
    pairs = tree.query_ball_point(pred, r=tol)
    pi = np.concatenate([np.full(len(v), i) for i, v in enumerate(pairs) if v]) \
        if any(pairs) else np.empty(0, int)
    tj = np.concatenate([np.asarray(v) for v in pairs if v]) \
        if any(pairs) else np.empty(0, int)
    if pi.size == 0:
        return np.empty(0), 0, n_p, n_t

    up, ut = np.unique(pi), np.unique(tj)
    ip = {v: k for k, v in enumerate(up)}
    it = {v: k for k, v in enumerate(ut)}
    BIG = tol * 10.0
    cost = np.full((len(up), len(ut)), BIG)
    d = np.linalg.norm(pred[pi] - true[tj], axis=1)
    cost[[ip[a] for a in pi], [it[b] for b in tj]] = d

    r, c = linear_sum_assignment(cost)
    ok = cost[r, c] < tol
    dist = cost[r, c][ok]
    tp = int(ok.sum())
    return dist, tp, n_p - tp, n_t - tp


def detection_metrics(tp, fp, fn, dist):
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(
        precision = prec,
        recall    = rec,
        f1        = f1,
        tp = tp, fp = fp, fn = fn,
        rmse = float(np.sqrt(np.mean(dist ** 2))) if len(dist) else np.nan,
        mae  = float(np.mean(dist)) if len(dist) else np.nan,
        p50  = float(np.median(dist)) if len(dist) else np.nan,
        p90  = float(np.percentile(dist, 90)) if len(dist) else np.nan,
    )


# ---------------------------------------------------------------------------
# 3.  FRAME AND DATASET DRIVERS
# ---------------------------------------------------------------------------
def score_frame(prob, gt_mask, cfg=LOC_CFG, tol=None, threshold=None):
    tol = cfg["tol_px"] if tol is None else tol
    thr = cfg["prob_threshold"] if threshold is None else threshold
    pred = peaks_from_prob(prob, cfg["min_distance"], thr, cfg["border"], cfg["subpixel"])
    true = peaks_from_mask(gt_mask, cfg["min_distance"], cfg["border"],
                           cfg["gt_is_binary"], cfg["subpixel"])
    return match_points(pred, true, tol), pred, true


@torch.no_grad()
def collect_peaks(model, loader, device="cuda", seg_index=1, cfg=LOC_CFG, amp=True):
    """
    Run the model once and cache per-frame peak sets, so tolerance and
    threshold sweeps are free afterwards.

    `seg_index` picks the segmentation output of the dual head.  Adjust the
    two lines marked ADAPT if your forward() returns something else.
    """
    model.eval()
    cached = []
    for batch in loader:
        # ---- ADAPT ----------------------------------------------------
        x, y = batch[0], batch[-1]          # (B,1,H,W) input, (B,1,H,W) mask
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.split(":")[0], enabled=amp):
            out = model(x)
        seg = out[seg_index] if isinstance(out, (tuple, list, dict)) else out
        # ---------------------------------------------------------------
        prob = torch.sigmoid(seg.float()).cpu().numpy()
        gt   = y.numpy()
        for b in range(prob.shape[0]):
            p = prob[b, 0] if prob.ndim == 4 else prob[b]
            g = gt[b, 0] if gt.ndim == 4 else gt[b]
            cached.append((p.astype(np.float32), g.astype(np.float32)))
    return cached


def score_dataset(cached, cfg=LOC_CFG, tol=None, threshold=None, per_frame=False):
    tol = cfg["tol_px"] if tol is None else tol
    thr = cfg["prob_threshold"] if threshold is None else threshold
    TP = FP = FN = 0
    dists, frames = [], []
    for prob, gt in cached:
        (d, tp, fp, fn), _, _ = score_frame(prob, gt, cfg, tol, thr)
        TP += tp; FP += fp; FN += fn
        dists.append(d)
        if per_frame:
            frames.append(detection_metrics(tp, fp, fn, d))
    d = np.concatenate(dists) if dists else np.empty(0)
    res = detection_metrics(TP, FP, FN, d)
    res["tol"], res["threshold"], res["n_frames"] = tol, thr, len(cached)
    if cfg["pixel_size"]:
        s = cfg["pixel_size"]
        for k in ("rmse", "mae", "p50", "p90"):
            res[k + "_A"] = res[k] * s
    return (res, frames) if per_frame else res


def sweep(cached, cfg=LOC_CFG):
    """Threshold x tolerance table.  Pick the operating point from this."""
    rows = []
    for thr in cfg["thr_sweep"]:
        for tol in cfg["tol_sweep"]:
            r = score_dataset(cached, cfg, tol=tol, threshold=thr)
            rows.append(r)
            print(f"  thr {thr:.2f}  tol {tol:.1f}px  "
                  f"P {r['precision']:.4f}  R {r['recall']:.4f}  F1 {r['f1']:.4f}  "
                  f"RMSE {r['rmse']:.3f}px")
    return rows


# ---------------------------------------------------------------------------
# 4.  USAGE
# ---------------------------------------------------------------------------
# cached = collect_peaks(model, val_loader, device=device, seg_index=1)
#
# main = score_dataset(cached, LOC_CFG)
# print(f"F1@{main['tol']}px = {main['f1']:.4f}   "
#       f"P {main['precision']:.4f}  R {main['recall']:.4f}   "
#       f"localization RMSE {main['rmse']:.3f} px "
#       f"(median {main['p50']:.3f}, p90 {main['p90']:.3f})")
#
# rows = sweep(cached, LOC_CFG)          # choose threshold on validation only
#
# Report per fold, then mean +/- std across the five folds, exactly as you do
# for IoU.  Quote the tolerance in every number -- "F1 0.97" is meaningless
# without it.

In [ ]:
cached = collect_peaks(model, val_loader, device=device, seg_index=1)
print(f"cached {len(cached)} frames")

m = score_dataset(cached, CFG)
print(f"F1@{m['tol']}px {m['f1']:.4f} | P {m['precision']:.4f} | R {m['recall']:.4f}")
print(f"TP {m['tp']}  FP {m['fp']}  FN {m['fn']}")
print(f"RMSE {m['rmse']:.3f} px | median {m['p50']:.3f} | p90 {m['p90']:.3f}")

---
## 9. Benchmark summary

In [ ]:
# ============================================================
# AtomSegNet output visualization
# Run this after AtomSegNet CV training is complete
# ============================================================
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from pathlib import Path
from sklearn.model_selection import KFold


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

ARCH_NAME = "AtomSegNet"
N_SHOW = 4
MASK_THRESHOLD = 0.5
RANDOM_SEED = SEED


# ------------------------------------------------------------
# Safe PyTorch checkpoint loader
# ------------------------------------------------------------
def load_torch_checkpoint(path, map_location="cpu"):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False
        )
    except TypeError:
        return torch.load(
            path,
            map_location=map_location
        )


# ------------------------------------------------------------
# Read completed CV results
# ------------------------------------------------------------
results_csv = Path(CFG.OUT_DIR) / "cv_results.csv"
results_json = Path(CFG.OUT_DIR) / "cv_results_partial.json"


if results_csv.is_file():
    results = pd.read_csv(results_csv)

elif results_json.is_file():
    with open(results_json, "r", encoding="utf-8") as f:
        results = pd.DataFrame(json.load(f))

else:
    raise FileNotFoundError(
        f"Could not find {results_csv} or {results_json}."
    )


atom_results = results[
    results["arch"] == ARCH_NAME
].copy()


if atom_results.empty:
    raise RuntimeError(
        f"No completed results were found for {ARCH_NAME}."
    )


atom_results = atom_results.sort_values(
    "iou",
    ascending=False
).reset_index(drop=True)


print("AtomSegNet fold results:")

display_columns = [
    column
    for column in [
        "fold",
        "epoch",
        "epochs_run",
        "psnr",
        "ssim",
        "iou"
    ]
    if column in atom_results.columns
]

print(
    atom_results[display_columns].to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Select best AtomSegNet fold
# ------------------------------------------------------------
best_row = atom_results.iloc[0]

best_fold = int(best_row["fold"])
best_iou = float(best_row["iou"])

checkpoint_path = (
    Path(CFG.OUT_DIR)
    / f"{ARCH_NAME}_fold{best_fold}.pt"
)


if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Best checkpoint is missing: {checkpoint_path}"
    )


print(
    f"\nUsing {ARCH_NAME} fold {best_fold} "
    f"with validation IoU {best_iou:.4f}"
)

print(
    f"Checkpoint: {checkpoint_path}"
)


# ------------------------------------------------------------
# Reconstruct the validation split for the selected fold
# ------------------------------------------------------------
kf = KFold(
    n_splits=CFG.N_FOLDS,
    shuffle=True,
    random_state=SEED
)

best_val_names = None

for fold, (_, va_idx) in enumerate(
    kf.split(common),
    start=1
):
    if fold == best_fold:
        best_val_names = [
            common[i]
            for i in va_idx
        ]
        break


if best_val_names is None:
    raise RuntimeError(
        f"Could not reconstruct validation fold {best_fold}."
    )


print(
    f"Validation samples in fold {best_fold}: "
    f"{len(best_val_names)}"
)


# ------------------------------------------------------------
# Load AtomSegNet
# ------------------------------------------------------------
model = ARCH_REGISTRY[ARCH_NAME]().to(DEVICE)

state_dict = load_torch_checkpoint(
    checkpoint_path,
    map_location="cpu"
)


# Handle full checkpoint dictionaries if necessary
if isinstance(state_dict, dict):
    if "model" in state_dict:
        state_dict = state_dict["model"]

    elif "state_dict" in state_dict:
        state_dict = state_dict["state_dict"]

    elif "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]


model.load_state_dict(
    state_dict,
    strict=True
)

model = model.to(
    DEVICE,
    memory_format=torch.channels_last
)

model.eval()

print("AtomSegNet loaded successfully.")


# ------------------------------------------------------------
# Prepare one sample
# ------------------------------------------------------------
def prepare_atomseg_sample(sample_name):
    noisy = norm01(
        imread_gray(noisy_map[sample_name])
    )

    if sample_name in clean_map:
        clean = norm01(
            imread_gray(clean_map[sample_name])
        )
    else:
        clean = noisy.copy()

    if sample_name in mask_map:
        mask = imread_gray(
            mask_map[sample_name]
        )

        if np.isfinite(mask).any() and np.nanmax(mask) > 0:
            mask = (
                mask > 0.5 * np.nanmax(mask)
            ).astype(np.float32)
        else:
            mask = np.zeros_like(
                noisy,
                dtype=np.float32
            )

    else:
        mask = np.zeros_like(
            noisy,
            dtype=np.float32
        )

    noisy = center_or_resize(
        noisy,
        CFG.IMG_SIZE,
        interpolation=cv2.INTER_AREA
    )

    clean = center_or_resize(
        clean,
        CFG.IMG_SIZE,
        interpolation=cv2.INTER_AREA
    )

    mask = center_or_resize(
        mask,
        CFG.IMG_SIZE,
        interpolation=cv2.INTER_NEAREST
    )

    mask = (
        mask > 0.5
    ).astype(np.float32)

    return (
        np.ascontiguousarray(
            noisy,
            dtype=np.float32
        ),
        np.ascontiguousarray(
            clean,
            dtype=np.float32
        ),
        np.ascontiguousarray(
            mask,
            dtype=np.float32
        )
    )


# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------
@torch.inference_mode()
def predict_atomseg(noisy):
    x = torch.from_numpy(
        noisy
    )[None, None].float()

    x = x.to(
        DEVICE,
        non_blocking=True,
        memory_format=torch.channels_last
    )

    denoised, seg_logits = model(x)

    denoised = (
        denoised[0, 0]
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    mask_probability = (
        torch.sigmoid(seg_logits)[0, 0]
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    predicted_mask = (
        mask_probability >= MASK_THRESHOLD
    ).astype(np.float32)

    return (
        denoised,
        mask_probability,
        predicted_mask
    )


# ------------------------------------------------------------
# Sample metrics
# ------------------------------------------------------------
def sample_iou(true_mask, predicted_mask):
    true_mask = true_mask.astype(bool)
    predicted_mask = predicted_mask.astype(bool)

    intersection = np.logical_and(
        true_mask,
        predicted_mask
    ).sum()

    union = np.logical_or(
        true_mask,
        predicted_mask
    ).sum()

    if union == 0:
        return 1.0

    return float(
        intersection / union
    )


def sample_psnr(clean, predicted):
    mse = np.mean(
        (
            clean.astype(np.float32)
            - predicted.astype(np.float32)
        ) ** 2
    )

    if mse <= 1e-12:
        return float("inf")

    return float(
        10.0 * np.log10(1.0 / mse)
    )


# ------------------------------------------------------------
# Select random validation samples
# ------------------------------------------------------------
rng = random.Random(
    RANDOM_SEED
)

n_show = min(
    N_SHOW,
    len(best_val_names)
)

selected_names = rng.sample(
    best_val_names,
    n_show
)


# ------------------------------------------------------------
# Display AtomSegNet outputs
# ------------------------------------------------------------
for sample_name in selected_names:
    noisy, clean, true_mask = prepare_atomseg_sample(
        sample_name
    )

    (
        predicted_denoised,
        mask_probability,
        predicted_mask
    ) = predict_atomseg(noisy)

    iou_value = sample_iou(
        true_mask,
        predicted_mask
    )

    psnr_value = sample_psnr(
        clean,
        predicted_denoised
    )

    overlay = np.stack(
        [noisy, noisy, noisy],
        axis=-1
    )

    # Predicted atom pixels appear red in the overlay.
    overlay[..., 0] = np.maximum(
        overlay[..., 0],
        predicted_mask
    )

    overlay[..., 1] *= (
        1.0 - 0.45 * predicted_mask
    )

    overlay[..., 2] *= (
        1.0 - 0.45 * predicted_mask
    )

    fig, axes = plt.subplots(
        1,
        6,
        figsize=(20, 4)
    )

    axes[0].imshow(
        noisy,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[0].set_title("Noisy")

    axes[1].imshow(
        clean,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[1].set_title("Clean GT")

    axes[2].imshow(
        predicted_denoised,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[2].set_title(
        f"Denoised\nPSNR {psnr_value:.2f} dB"
    )

    axes[3].imshow(
        true_mask,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[3].set_title("Atom mask GT")

    axes[4].imshow(
        predicted_mask,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[4].set_title(
        f"Predicted mask\nIoU {iou_value:.3f}"
    )

    axes[5].imshow(
        overlay
    )
    axes[5].set_title("Prediction overlay")

    for axis in axes:
        axis.axis("off")

    fig.suptitle(
        f"{ARCH_NAME} | Fold {best_fold} | "
        f"Sample: {sample_name}",
        fontsize=13
    )

    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 9. Benchmark summary
# ============================================================
from scipy import stats

# works after a kernel restart too: rebuild df from the CSV if needed
if 'df' not in globals():
    df = pd.read_csv(os.path.join(CFG.OUT_DIR, 'cv_results.csv'))
df['arch'] = pd.Categorical(df['arch'], categories=CFG.ARCHS, ordered=True)

# --- aggregate table ----------------------------------------------------
param_counts = {n: sum(p.numel() for p in c().parameters()) / 1e6
                for n, c in ARCH_REGISTRY.items()}

rows = []
for arch, g in df.groupby('arch', observed=True):
    r = {'arch': str(arch), 'folds': len(g), 'params_M': round(param_counts[str(arch)], 2)}
    for m in ('psnr', 'ssim', 'iou'):
        r[f'{m}_mean'] = g[m].mean()
        r[f'{m}_std']  = g[m].std(ddof=1)
    r['gauss_psnr']   = g['gauss_psnr'].mean()
    r['psnr_gain_dB'] = r['psnr_mean'] - r['gauss_psnr']
    if 'epochs_run' in g.columns:
        r['epochs_mean'] = g['epochs_run'].mean()
    if 'time' in g.columns:
        r['s_per_epoch'] = g['time'].mean()
    rows.append(r)

summary = pd.DataFrame(rows).sort_values('iou_mean', ascending=False).reset_index(drop=True)
summary.round(4).to_csv(os.path.join(CFG.OUT_DIR, 'benchmark_summary.csv'), index=False)

# paste-ready mean ± std table (thesis / slide format)
fmt = pd.DataFrame({
    'arch':       summary['arch'],
    'params (M)': summary['params_M'].map('{:.2f}'.format),
    'PSNR (dB)':  summary.apply(lambda r: f"{r.psnr_mean:.2f} ± {r.psnr_std:.2f}", axis=1),
    'SSIM':       summary.apply(lambda r: f"{r.ssim_mean:.3f} ± {r.ssim_std:.3f}", axis=1),
    'IoU':        summary.apply(lambda r: f"{r.iou_mean:.3f} ± {r.iou_std:.3f}", axis=1),
    'gain vs Gauss (dB)': summary['psnr_gain_dB'].map('{:+.2f}'.format),
})
if 'epochs_mean' in summary.columns:
    fmt['epochs'] = summary['epochs_mean'].map('{:.1f}'.format)

md = ['| ' + ' | '.join(fmt.columns) + ' |',
      '|' + '|'.join(['---'] * len(fmt.columns)) + '|']
md += ['| ' + ' | '.join(str(v) for v in row) + ' |' for row in fmt.itertuples(index=False)]
with open(os.path.join(CFG.OUT_DIR, 'benchmark_table.md'), 'w') as f:
    f.write('\n'.join(md))

print(fmt.to_string(index=False))

# --- paired comparison on IoU -------------------------------------------
# valid pairing: KFold is seeded, so every arch saw byte-identical fold splits
piv_iou = df.pivot_table(index='fold', columns='arch', values='iou', observed=True)
best_arch = str(summary.iloc[0]['arch'])
print(f"\nbest by IoU: {best_arch}")
for other in [str(a) for a in piv_iou.columns if str(a) != best_arch]:
    pair = piv_iou[[best_arch, other]].dropna()
    if len(pair) < 2:
        print(f"  {best_arch} vs {other}: fewer than 2 shared folds, no test")
        continue
    d = pair[best_arch] - pair[other]
    t, p = stats.ttest_rel(pair[best_arch], pair[other])
    print(f"  {best_arch} vs {other}:  dIoU = {d.mean():+.4f} ± {d.std(ddof=1):.4f}   "
          f"wins {int((d > 0).sum())}/{len(d)} folds   paired t p = {p:.3f}")

# --- figure: bars + fold points on top, paired fold lines below ----------
def _bars(ax, metric):
    try:
        sns.barplot(data=df, x='arch', y=metric, order=CFG.ARCHS, ax=ax,
                    errorbar='sd', capsize=0.15, alpha=0.75)
    except TypeError:                              # seaborn < 0.12
        sns.barplot(data=df, x='arch', y=metric, order=CFG.ARCHS, ax=ax,
                    ci='sd', capsize=0.15, alpha=0.75)
    sns.stripplot(data=df, x='arch', y=metric, order=CFG.ARCHS, ax=ax,
                  color='k', size=5, jitter=0.08)

panels = [('psnr', 'PSNR (dB)', '%.2f'), ('ssim', 'SSIM', '%.3f'), ('iou', 'IoU', '%.3f')]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (metric, title, lab_fmt) in zip(axes[0], panels):
    _bars(ax, metric)
    if ax.containers:
        ax.bar_label(ax.containers[0], fmt=lab_fmt, padding=2, fontsize=9)
    ax.set_title(f'{title}  (bars: mean ± sd over folds, dots: folds)', fontsize=10)
    ax.set_xlabel('')
gauss_mean = df['gauss_psnr'].mean()
axes[0, 0].axhline(gauss_mean, ls='--', c='gray', lw=1)
axes[0, 0].text(0.02, gauss_mean, f'  Gaussian σ=1: {gauss_mean:.2f} dB',
                va='bottom', fontsize=8, color='gray')

for ax, (metric, title, _) in zip(axes[1], panels):
    p = df.pivot_table(index='fold', columns='arch', values=metric, observed=True)
    for arch in CFG.ARCHS:
        if arch in p.columns:
            ax.plot(p.index, p[arch], marker='o', ms=5, label=arch)
    ax.set_xticks(sorted(df['fold'].unique()))
    ax.set_xlabel('fold')
    ax.set_title(f'{title} per fold (paired)', fontsize=10)
axes[1, 0].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(CFG.OUT_DIR, 'benchmark_bars.png'), dpi=200)
plt.show()

---
## 10. Atomic-column localization & interatomic spacing

We use the best AtomSegNet fold for atom localization. Procedure:

1. Run inference → denoised image + atom-mask probability.
2. `peak_local_max` on the mask gives integer-pixel atom centres.
3. **Sub-pixel refinement**: 2-D Gaussian fit on a 5×5 patch of the *denoised* image around each peak.
4. KD-tree → nearest-neighbour distance distribution → mean lattice spacing in Å (using `CFG.PIXEL_SIZE_A`).

In [ ]:
# ============================================================
# 10. Localization, spacing statistics, ground-truth comparison
# ============================================================
from scipy.optimize import linear_sum_assignment

FORCE_RELOC = False        # True -> recompute even if results exist on disk
N_POOL      = 25           # held-out samples pooled for the statistics
MATCH_TOL   = 3.0          # px, detection <-> GT matching radius

LOC_SUMMARY = os.path.join(CFG.OUT_DIR, 'localization_summary.json')
LOC_ATOMS   = os.path.join(CFG.OUT_DIR, 'localization_atoms.csv')
LOC_SPACING = os.path.join(CFG.OUT_DIR, 'localization_spacings.csv')

def gaussian_2d(xy, A, x0, y0, sx, sy, B):
    x, y = xy
    return (A * np.exp(-((x - x0)**2 / (2*sx**2) + (y - y0)**2 / (2*sy**2))) + B).ravel()


def subpixel_refine(img, peaks, half=2):
    """Bounded 2-D Gaussian fit on a (2*half+1)^2 patch of the denoised image.
    Returns coords (N,2 row/col), ok mask, fitted sigmas (N,2 row/col px; NaN on fallback)."""
    H, W = img.shape
    ys, xs = np.mgrid[-half:half + 1, -half:half + 1]
    lo = [0.0,   -half, -half, 0.3,      0.3,      -np.inf]
    hi = [np.inf, half,  half, 2.0*half, 2.0*half,  np.inf]
    out, ok, sig = [], [], []
    for (py, px) in peaks:
        if py - half < 0 or py + half >= H or px - half < 0 or px + half >= W:
            out.append((py, px)); ok.append(False); sig.append((np.nan, np.nan)); continue
        patch = img[py - half:py + half + 1, px - half:px + half + 1]
        p0 = (max(patch.max() - patch.min(), 1e-3), 0.0, 0.0, 1.0, 1.0, patch.min())
        try:
            popt, _ = curve_fit(gaussian_2d, (xs, ys), patch.ravel(),
                                p0=p0, bounds=(lo, hi), max_nfev=200)
            A, dx, dy, s_x, s_y, _ = popt
            if A > 1e-3 and abs(dx) < 1.5 and abs(dy) < 1.5:
                out.append((py + dy, px + dx)); ok.append(True); sig.append((s_y, s_x))
            else:
                out.append((py, px)); ok.append(False); sig.append((np.nan, np.nan))
        except Exception:
            out.append((py, px)); ok.append(False); sig.append((np.nan, np.nan))
    return (np.asarray(out, float).reshape(-1, 2),
            np.asarray(ok, bool),
            np.asarray(sig, float).reshape(-1, 2))


@torch.inference_mode()
def localize_atoms(model, noisy_tensor):
    model.eval()
    with amp_autocast():
        den, seg_logits = model(noisy_tensor.to(DEVICE, non_blocking=True))
    den  = den[0, 0].float().clamp(0, 1).cpu().numpy()
    prob = torch.sigmoid(seg_logits)[0, 0].float().cpu().numpy()
    # threshold_abs: sigmoid prob has an absolute scale; per-image threshold_rel
    # made the cut depend on each image's brightest peak
    peaks = peak_local_max(prob, min_distance=CFG.PEAK_MIN_DIST,
                           threshold_abs=CFG.PEAK_THRESH)
    refined, ok, sig = subpixel_refine(den, peaks)
    return den, prob, peaks, refined, ok, sig


def nn_spacings_px(coords_px):
    if len(coords_px) < 2:
        return np.array([])
    d, _ = cKDTree(coords_px).query(coords_px, k=2)
    return d[:, 1]


# position/ in TEM-ImageNet-v1.3 stores unit-cell vectors, NOT per-atom
# coordinates. gaussianMask/ holds a Gaussian centred on each atomic column,
# so its intensity-weighted blob centroids are the correct sub-pixel ground
# truth. circularMask is the coarse fallback.
GT_COORD_MAP = gauss_map if gauss_map else mask_map
print(f"localization ground truth: "
      f"{'gaussianMask' if gauss_map else 'circularMask (fallback)'}"
      f"  ({len(GT_COORD_MAP)} files)")


def load_gt_coords(basename, size):
    """Sub-pixel GT centres: intensity-weighted centroid of each connected
    blob in the gaussianMask, rescaled to `size`. None if unavailable."""
    p = GT_COORD_MAP.get(basename)
    if p is None:
        return None
    m = imread_gray(p)
    H0, W0 = m.shape
    mx = np.nanmax(m)
    if not np.isfinite(mx) or mx <= 0:
        return None
    b = (m > 0.25 * mx).astype(np.uint8)
    n, lab, _, _ = cv2.connectedComponentsWithStats(b)
    if n <= 1:
        return None
    # intensity weighting recovers the Gaussian peak to well under one pixel,
    # which a binary centroid cannot do
    w = np.where(b > 0, m, 0.0).astype(np.float64)
    ys, xs = np.mgrid[0:H0, 0:W0]
    tot = np.bincount(lab.ravel(), weights=w.ravel(), minlength=n)[1:]
    cy = np.bincount(lab.ravel(), weights=(w * ys).ravel(), minlength=n)[1:]
    cx = np.bincount(lab.ravel(), weights=(w * xs).ravel(), minlength=n)[1:]
    ok = tot > 0
    if not ok.any():
        return None
    cy, cx = cy[ok] / tot[ok], cx[ok] / tot[ok]
    return np.stack([cy * (size / H0), cx * (size / W0)], axis=1)


def match_detections(det, gt, tol_px=MATCH_TOL):
    """Hungarian matching within tol_px. One detection matches at most one GT atom."""
    if det is None or gt is None or len(det) == 0 or len(gt) == 0:
        return None
    D = np.linalg.norm(det[:, None, :] - gt[None, :, :], axis=2)
    ri, ci = linear_sum_assignment(D)
    keep = D[ri, ci] <= tol_px
    return dict(n_det=len(det), n_gt=len(gt), n_match=int(keep.sum()),
                residuals_px=D[ri[keep], ci[keep]])


# --- model + truly held-out sample list ---------------------------------
if 'df' not in globals():
    df = pd.read_csv(os.path.join(CFG.OUT_DIR, 'cv_results.csv'))
sub = df[df['arch'] == 'AtomSegNet']
if len(sub) == 0:
    raise RuntimeError("no AtomSegNet rows in cv_results; run section 8 first")
best_row  = sub.sort_values('iou', ascending=False).iloc[0]
best_fold = int(best_row['fold'])
best_ckpt = os.path.join(CFG.OUT_DIR, f"AtomSegNet_fold{best_fold}.pt")
if not os.path.isfile(best_ckpt):
    raise RuntimeError(f"checkpoint missing: {best_ckpt}")
print(f"best AtomSegNet fold {best_fold} (IoU {best_row['iou']:.3f}) -> {best_ckpt}")

best_model = prep_model(AtomSegNet())
load_model_weights(best_ckpt, best_model)
best_model.eval()

kf = KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=SEED)
va_idx = list(kf.split(common))[best_fold - 1][1]
val_names = [common[i] for i in va_idx]
print(f"fold-{best_fold} val split: {len(val_names)} held-out samples "
      f"(the old cell used common[-1], which is in this model's TRAINING set with prob. 4/5)")

# --- compute, or reuse what is on disk ----------------------------------
if os.path.isfile(LOC_SUMMARY) and not FORCE_RELOC:
    with open(LOC_SUMMARY) as f:
        loc_summary = json.load(f)
    print("localization results already on disk (FORCE_RELOC=True to recompute):")
    print(json.dumps(loc_summary, indent=2))
else:
    atom_rows, spac_chunks, res_chunks = [], [], []
    n_det_tot = n_gt_tot = n_match_tot = 0
    demo = None
    for b in tqdm(val_names[:N_POOL], desc='localize'):
        nt, ct, mt, _ = TEMSegDataset([b], noisy_map, clean_map, mask_map,
                                      size=CFG.IMG_SIZE, train=False)[0]
        den, prob, peaks, refined, ok, sig = localize_atoms(best_model, nt.unsqueeze(0))
        spac_chunks.append(nn_spacings_px(refined))
        for (y, x), o, (sr, sc) in zip(refined, ok, sig):
            atom_rows.append(dict(sample=b, y_px=y, x_px=x, refined=bool(o),
                                  sigma_r_px=sr, sigma_c_px=sc))
        gt = load_gt_coords(b, CFG.IMG_SIZE)
        mres = match_detections(refined, gt)
        if mres is not None:
            n_det_tot += mres['n_det']; n_gt_tot += mres['n_gt']; n_match_tot += mres['n_match']
            res_chunks.append(mres['residuals_px'])
        if demo is None:
            demo = dict(b=b, nt=nt, ct=ct, den=den, prob=prob, refined=refined, gt=gt)

    atoms_df = pd.DataFrame(atom_rows)
    atoms_df['sigma_mean_px'] = np.sqrt(atoms_df['sigma_r_px'] * atoms_df['sigma_c_px'])
    spac_px = np.concatenate(spac_chunks) if spac_chunks else np.array([])
    spac_A  = spac_px * CFG.PIXEL_SIZE_A
    fwhm_A  = atoms_df['sigma_mean_px'].dropna().to_numpy() * CFG.PIXEL_SIZE_A * 2.355

    loc_summary = dict(
        best_fold=best_fold, n_samples=int(min(N_POOL, len(val_names))),
        n_atoms=int(len(atoms_df)),
        refine_success_rate=float(atoms_df['refined'].mean()) if len(atoms_df) else None,
        spacing_px_mean=float(spac_px.mean()) if len(spac_px) else None,
        spacing_px_median=float(np.median(spac_px)) if len(spac_px) else None,
        spacing_A_mean=float(spac_A.mean()) if len(spac_A) else None,
        spacing_A_std=float(spac_A.std()) if len(spac_A) else None,
        column_fwhm_A_mean=float(fwhm_A.mean()) if len(fwhm_A) else None,
        pixel_size_A=CFG.PIXEL_SIZE_A, match_tol_px=MATCH_TOL,
    )
    if n_gt_tot > 0:
        res_px = np.concatenate(res_chunks) if res_chunks else np.array([])
        precision = n_match_tot / max(n_det_tot, 1)
        recall    = n_match_tot / n_gt_tot
        loc_summary.update(
            gt_precision=float(precision), gt_recall=float(recall),
            gt_f1=float(2*precision*recall / max(precision + recall, 1e-9)),
            gt_rmse_px=float(np.sqrt(np.mean(res_px**2))) if len(res_px) else None,
            gt_rmse_A=float(np.sqrt(np.mean(res_px**2)) * CFG.PIXEL_SIZE_A) if len(res_px) else None,
            gt_n_det=n_det_tot, gt_n_gt=n_gt_tot, gt_n_match=n_match_tot)
    else:
        print("position/ maps not parseable as marker images; GT comparison skipped")

    atoms_df.to_csv(LOC_ATOMS, index=False)
    pd.DataFrame({'spacing_px': spac_px, 'spacing_A': spac_A}).to_csv(LOC_SPACING, index=False)
    _atomic_json(loc_summary, LOC_SUMMARY)
    print(json.dumps(loc_summary, indent=2))

    # --- demo figure on a genuinely held-out sample ---------------------
    d = demo
    demo_psnr = compute_psnr(d['ct'][0].numpy(), d['den'], data_range=1.0)
    fig, axes = plt.subplots(1, 5, figsize=(19, 4))
    axes[0].imshow(d['nt'][0], cmap='gray');  axes[0].set_title('Noisy')
    axes[1].imshow(d['den'], cmap='gray');    axes[1].set_title(f'Denoised ({demo_psnr:.1f} dB)')
    axes[2].imshow(d['ct'][0], cmap='gray');  axes[2].set_title('Clean GT')
    axes[3].imshow(d['prob'], cmap='magma');  axes[3].set_title('Atom-mask prob.')
    axes[4].imshow(d['den'], cmap='gray')
    if len(d['refined']):
        axes[4].scatter(d['refined'][:, 1], d['refined'][:, 0], s=10, c='cyan',
                        marker='o', linewidths=0, label=f"det ({len(d['refined'])})")
    if d['gt'] is not None and len(d['gt']):
        axes[4].scatter(d['gt'][:, 1], d['gt'][:, 0], s=22, c='lime',
                        marker='x', linewidths=0.8, label=f"GT ({len(d['gt'])})")
    axes[4].legend(fontsize=7, loc='lower right')
    axes[4].set_title(f"Detections, fold-{best_fold} val: {d['b']}")
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUT_DIR, 'localization_demo.png'), dpi=150)
    plt.show()

    # --- pooled histograms ----------------------------------------------
    n_pan = 2 + int(n_gt_tot > 0)
    fig, axes = plt.subplots(1, n_pan, figsize=(5.5 * n_pan, 4))
    axes = np.atleast_1d(axes)
    if len(spac_A):
        axes[0].hist(spac_A, bins=50, color='steelblue', edgecolor='k')
        axes[0].axvline(spac_A.mean(), c='r', ls='--',
                        label=f"mean {spac_A.mean():.2f} Å ({spac_px.mean():.2f} px)")
        axes[0].set_xlabel('NN spacing (Å)'); axes[0].set_ylabel('count')
        axes[0].set_title(f"Interatomic spacing, n={len(spac_A)}, {loc_summary['n_samples']} images")
        axes[0].legend(fontsize=8)
    if len(fwhm_A):
        axes[1].hist(fwhm_A, bins=50, color='darkorange', edgecolor='k')
        axes[1].axvline(fwhm_A.mean(), c='r', ls='--', label=f"mean {fwhm_A.mean():.2f} Å")
        axes[1].set_xlabel('fitted column FWHM (Å)'); axes[1].set_ylabel('count')
        axes[1].set_title('Column width from 2-D Gaussian fits')
        axes[1].legend(fontsize=8)
    if n_gt_tot > 0 and len(res_px):
        axes[2].hist(res_px, bins=40, color='seagreen', edgecolor='k')
        axes[2].axvline(np.sqrt(np.mean(res_px**2)), c='r', ls='--',
                        label=f"RMSE {loc_summary['gt_rmse_px']:.2f} px "
                              f"= {loc_summary['gt_rmse_A']:.3f} Å")
        axes[2].set_xlabel('|det - GT| (px)'); axes[2].set_ylabel('count')
        axes[2].set_title(f"Localization vs GT  (P {loc_summary['gt_precision']:.3f} / "
                          f"R {loc_summary['gt_recall']:.3f} / F1 {loc_summary['gt_f1']:.3f})")
        axes[2].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUT_DIR, 'localization_hists.png'), dpi=150)
    plt.show()


---
## 11. Sub-pixel localization precision (synthetic shift test)

To quantify localization precision (CV claim: < 0.5 px), we shift a clean test image by sub-pixel offsets, run the pipeline, and measure how accurately the recovered atom centres track the imposed shift.

In [ ]:
# ============================================================
# 11. Sub-pixel localization precision (shift recovery on NOISY input)
# ============================================================
from scipy.optimize import linear_sum_assignment

FORCE_PREC = False
N_PREC     = 15
# 0.0 = repeatability control, 1.0 = integer-shift equivariance control,
# fractional values = the actual sub-pixel measurement
SHIFTS     = (0.0, 0.25, -0.25, 0.5, -0.5, 0.75, -0.75, 1.0)
EDGE_M     = 8            # px, drop atoms near borders (reflection strip, patch clearance)
PAIR_TOL   = 2.0          # px, = PEAK_MIN_DIST/2; Hungarian matching resolves ties globally
PREC_JSON  = os.path.join(CFG.OUT_DIR, 'precision_summary.json')
PREC_CSV   = os.path.join(CFG.OUT_DIR, 'precision_errors.csv')

for _v in ('best_model', 'val_names'):
    if _v not in globals():
        raise RuntimeError("run the section 10 cell first; it defines best_model and val_names")


def shift_image(img, dx, dy):
    """Content moves by (+dx cols, +dy rows). INTER_CUBIC for sub-pixel fidelity;
    caveat: fractional shifts interpolate and thereby smooth the noise slightly,
    so fractional-shift errors are marginally optimistic. Integer shifts do not
    interpolate at all and are exact translations."""
    H, W = img.shape
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    out = cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_CUBIC,
                         borderMode=cv2.BORDER_REFLECT)
    return np.clip(out, 0.0, 1.0)


def match_pairs(a, b, tol_px):
    """Hungarian one-to-one match of rows of a to rows of b within tol_px."""
    if len(a) == 0 or len(b) == 0:
        return np.array([], int), np.array([], int)
    D = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=2)
    ri, ci = linear_sum_assignment(D)
    k = D[ri, ci] <= tol_px
    return ri[k], ci[k]


if os.path.isfile(PREC_JSON) and not FORCE_PREC:
    with open(PREC_JSON) as f:
        prec_summary = json.load(f)
    print("precision results already on disk (FORCE_PREC=True to recompute):")
    print(json.dumps(prec_summary, indent=2))
else:
    rows, mrows = [], []
    for b in tqdm(val_names[:N_PREC], desc='precision'):
        noisy = center_or_resize(norm01(imread_gray(noisy_map[b])), CFG.IMG_SIZE)
        x0 = torch.from_numpy(noisy)[None, None].float()
        _, _, pk0, rf0, _, _ = localize_atoms(best_model, x0)
        if len(rf0) < 5:
            continue
        det0 = {'refined': rf0, 'raw': pk0.astype(float)}

        for axis in ('x', 'y'):
            for s in SHIFTS:
                dx, dy = (s, 0.0) if axis == 'x' else (0.0, s)
                xs = torch.from_numpy(shift_image(noisy, dx, dy))[None, None].float()
                _, _, pks, rfs, _, _ = localize_atoms(best_model, xs)
                for meth, d0 in det0.items():
                    ds = rfs if meth == 'refined' else pks.astype(float)
                    if len(ds) == 0:
                        continue
                    ia, ib = match_pairs(ds - np.array([dy, dx]), d0, PAIR_TOL)
                    mrows.append(dict(method=meth, axis=axis, shift=s,
                                      n_det=len(ds), n_match=len(ia)))
                    if len(ia) == 0:
                        continue
                    keep = np.all((d0[ib] >= EDGE_M) &
                                  (d0[ib] < CFG.IMG_SIZE - EDGE_M), axis=1)
                    err = (ds[ia] - np.array([dy, dx]) - d0[ib])[keep]
                    frac = abs(s - round(s)) > 1e-9
                    for ey, ex in err:
                        rows.append(dict(sample=b, axis=axis, shift=s, method=meth,
                                         is_frac=frac, err_row=ey, err_col=ex))

    pr = pd.DataFrame(rows)
    pr['err_along'] = np.where(pr['axis'] == 'x', pr['err_col'], pr['err_row'])
    pr.to_csv(PREC_CSV, index=False)
    mr = pd.DataFrame(mrows)

    def _stats(g):
        ex, ey = g['err_col'].to_numpy(), g['err_row'].to_numpy()
        return dict(n=int(len(g)),
                    bias_x=float(ex.mean()), bias_y=float(ey.mean()),
                    sigma_x=float(ex.std(ddof=1)), sigma_y=float(ey.std(ddof=1)),
                    rmse_2d=float(np.sqrt(np.mean(ex**2 + ey**2))))

    frac_ref = _stats(pr[(pr.method == 'refined') & pr.is_frac])
    frac_raw = _stats(pr[(pr.method == 'raw') & pr.is_frac])
    ctrl0 = pr[(pr.shift == 0.0) & (pr.method == 'refined')]
    equiv = _stats(pr[(pr.shift == 1.0) & (pr.method == 'refined')])
    match_rate = float(mr.groupby('method').apply(
        lambda g: g.n_match.sum() / max(g.n_det.sum(), 1)).loc['refined'])

    prec_summary = dict(
        n_images=int(min(N_PREC, len(val_names))), best_fold=best_fold,
        shifts=list(SHIFTS), match_rate_refined=match_rate,
        refined_fractional=frac_ref, raw_fractional=frac_raw,
        # pair error = difference of two measurements; if roughly independent
        # (true at fractional phase), single-shot RMSE ~ pair RMSE / sqrt(2)
        refined_single_shot_rmse_est=frac_ref['rmse_2d'] / np.sqrt(2),
        control_shift0_max_abs_px=float(np.abs(
            ctrl0[['err_row', 'err_col']].to_numpy()).max()) if len(ctrl0) else None,
        control_integer_shift=equiv,
        pixel_size_A=CFG.PIXEL_SIZE_A,
        rmse_2d_A=frac_ref['rmse_2d'] * CFG.PIXEL_SIZE_A,
    )
    _atomic_json(prec_summary, PREC_JSON)

    print(f"\nrefined, fractional shifts:  pair RMSE(2D) = {frac_ref['rmse_2d']:.3f} px "
          f"(~{prec_summary['refined_single_shot_rmse_est']:.3f} px single-shot)  "
          f"sigma_x/y = {frac_ref['sigma_x']:.3f}/{frac_ref['sigma_y']:.3f} px  "
          f"bias_x/y = {frac_ref['bias_x']:+.3f}/{frac_ref['bias_y']:+.3f} px")
    print(f"raw integer peaks:           pair RMSE(2D) = {frac_raw['rmse_2d']:.3f} px  "
          f"(per-axis quantization floor 1/sqrt(12) = 0.289 px)")
    _c0 = prec_summary['control_shift0_max_abs_px']
    _eq = equiv.get('rmse_2d') if isinstance(equiv, dict) else None
    print("controls: shift 0.0 max |err| = "
          + (f"{_c0:.4f} px" if _c0 is not None else "n/a (no matched pairs)")
          + " (repeatability incl. cudnn nondeterminism); shift 1.0 RMSE = "
          + (f"{_eq:.3f} px" if _eq is not None and np.isfinite(_eq)
             else "n/a (no matched pairs)")
          + " (translation equivariance)")
    if _c0 is None or _eq is None:
        print("  -> a control is empty. Raise N_PREC, lower EDGE_M, or check "
              "that the model actually detects atoms in these frames.")
    verdict = "SUPPORTED" if frac_ref['rmse_2d'] < 0.5 else "NOT supported at pair level"
    print(f"claim '< 0.5 px': {verdict} on the conservative pair-based RMSE")

    # --- figure: error histograms + pixel-locking curve -----------------
    lock = (pr[pr.is_frac | (pr.shift == 1.0)]
            .groupby(['method', 'shift'])['err_along']
            .agg(['mean', 'std', 'count']).reset_index())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    bins = np.linspace(-1.0, 1.0, 61)
    for meth, col in (('raw', 'gray'), ('refined', 'steelblue')):
        g = pr[(pr.method == meth) & pr.is_frac]
        axes[0].hist(g['err_along'], bins=bins, alpha=0.6, color=col,
                     label=f"{meth}: sigma={g['err_along'].std(ddof=1):.3f} px")
    axes[0].axvline(0, c='k', lw=0.8)
    axes[0].set_xlabel('per-axis shift-recovery error (px)')
    axes[0].set_ylabel('atoms'); axes[0].legend(fontsize=8)
    axes[0].set_title('Fractional shifts, pooled over both axes')

    for meth, col in (('raw', 'gray'), ('refined', 'steelblue')):
        g = lock[lock.method == meth]
        axes[1].errorbar(g['shift'], g['mean'], yerr=g['std'], marker='o',
                         ms=4, capsize=3, color=col, label=meth)
    axes[1].axhline(0, c='k', lw=0.8)
    axes[1].set_xlabel('imposed shift (px)')
    axes[1].set_ylabel('mean recovery error along shift axis (px)')
    axes[1].set_title('Bias vs sub-pixel phase (pixel locking shows as a sawtooth)')
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.OUT_DIR, 'localization_precision.png'), dpi=150)
    plt.show()


---
## 12. Save final report

In [ ]:
# ============================================================
# 12. Final report (assembled from on-disk artifacts, standalone-safe)
# ============================================================
import platform

def _atomic_json(obj, path):
    tmp = str(path) + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)

def _read_json(path):
    if os.path.isfile(path):
        try:
            with open(path) as f:
                return json.load(f)
        except Exception as e:
            print(f"unreadable: {path} ({e})")
    return None

OUT = CFG.OUT_DIR
P = dict(cv=os.path.join(OUT, 'cv_results.csv'),
         summ=os.path.join(OUT, 'benchmark_summary.csv'),
         table=os.path.join(OUT, 'benchmark_table.md'),
         loc=os.path.join(OUT, 'localization_summary.json'),
         prec=os.path.join(OUT, 'precision_summary.json'),
         n2v=os.path.join(OUT, 'atomsegnet_n2v.pt'))
FIGS = ['benchmark_bars.png', 'localization_demo.png',
        'localization_hists.png', 'localization_precision.png']

# --- environment provenance ---------------------------------------------
env = dict(generated=time.strftime('%Y-%m-%d %H:%M:%S'),
           python=platform.python_version(), torch=torch.__version__,
           cuda=torch.version.cuda, numpy=np.__version__,
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           amp=bool(globals().get('AMP_ON', False)),
           precision=CFG.PRECISION, cache=str(CFG.CACHE_DIR),
           cudnn_benchmark=bool(torch.backends.cudnn.benchmark), seed=SEED)

# --- benchmark block ----------------------------------------------------
cv_df = pd.read_csv(P['cv']) if os.path.isfile(P['cv']) else None
bench, best_overall, best_asn, gpu_h = None, None, None, None
if cv_df is not None and len(cv_df):
    if os.path.isfile(P['summ']):
        bench = pd.read_csv(P['summ']).to_dict('records')
    else:                                   # minimal fallback if section 9 never ran
        bench = (cv_df.groupby('arch')[['psnr', 'ssim', 'iou']]
                 .agg(['mean', 'std']).round(4))
        bench.columns = ['_'.join(c) for c in bench.columns]
        bench = bench.reset_index().to_dict('records')
    by_iou = cv_df.groupby('arch')['iou'].mean().sort_values(ascending=False)
    best_overall = dict(arch=str(by_iou.index[0]), mean_iou=float(by_iou.iloc[0]))
    sub = cv_df[cv_df['arch'] == 'AtomSegNet']
    if len(sub):
        r = sub.sort_values('iou', ascending=False).iloc[0]
        best_asn = dict(fold=int(r['fold']), iou=float(r['iou']),
                        psnr=float(r['psnr']), ssim=float(r['ssim']),
                        psnr_gain_over_gaussian_dB=float(r['psnr'] - r['gauss_psnr']),
                        ckpt=os.path.join(OUT, f"AtomSegNet_fold{int(r['fold'])}.pt"))
    if {'epochs_run', 'time'} <= set(cv_df.columns):
        gpu_h = float((cv_df['epochs_run'] * cv_df['time']).sum() / 3600)

loc  = _read_json(P['loc'])
prec = _read_json(P['prec'])

# --- honesty caveats, mirrors what the numbers can and cannot claim -----
caveats = [
    f"PIXEL_SIZE_A = {CFG.PIXEL_SIZE_A} A/px is a placeholder for simulated data; "
    "Angstrom values scale linearly with it, pixel-unit values do not depend on it.",
    "N2V warm start was applied to AtomSegNet only, so the three-architecture "
    "comparison is not matched on pretraining.",
    "N2V pretraining used all noisy images, including images that later served "
    "as validation in the CV folds (transductive, labels never seen).",
    "Folds are seeded KFold over basenames; if filenames encode simulation "
    "series, sibling frames can occupy train and val of the same fold.",
    "cudnn.benchmark=True, so reruns are not bitwise reproducible.",
]
if prec:
    caveats.append("Precision values are pair RMSE from shifted-vs-unshifted "
                   "comparisons; the single-shot estimate assumes independent "
                   "errors at fractional pixel phase.")

artifacts = {os.path.basename(p): os.path.isfile(p) for p in list(P.values())}
artifacts.update({f: os.path.isfile(os.path.join(OUT, f)) for f in FIGS})

report = dict(
    environment=env,
    config=dict(dataset_root=CFG.DATA_ROOT,
                n_samples=len(common) if 'common' in globals() else None,
                image_size=CFG.IMG_SIZE, batch=CFG.BATCH, lr=CFG.LR, wd=CFG.WD,
                n_folds=CFG.N_FOLDS, epochs_max=CFG.EPOCHS, archs=list(CFG.ARCHS),
                pixel_size_A=CFG.PIXEL_SIZE_A,
                n2v=dict(ratio=CFG.N2V_MASK_RATIO, radius=CFG.N2V_RADIUS,
                         weights_present=os.path.isfile(P['n2v']))),
    benchmark=bench, best_arch_by_mean_iou=best_overall,
    best_atomsegnet=best_asn, est_total_gpu_hours=gpu_h,
    localization=loc, precision=prec,
    fold_results=cv_df.to_dict('records') if cv_df is not None else None,
    caveats=caveats, artifacts=artifacts,
)
_atomic_json(report, os.path.join(OUT, 'report.json'))

# --- report.md ----------------------------------------------------------
L = ['# TEM denoising and atomic column localization: run report', '',
     f"generated {env['generated']} on {env['gpu'] or 'CPU'}, "
     f"torch {env['torch']} (CUDA {env['cuda']}), seed {env['seed']}",
     f"dataset: {report['config']['n_samples']} paired samples at "
     f"{CFG.IMG_SIZE} px, {CFG.N_FOLDS}-fold CV"
     + (f", ~{gpu_h:.1f} GPU-hours total" if gpu_h else ''), '']
if os.path.isfile(P['table']):
    L += ['## benchmark', '', open(P['table']).read(), '']
if best_asn:
    L += ['## selected model', '',
          f"AtomSegNet fold {best_asn['fold']}: IoU {best_asn['iou']:.3f}, "
          f"PSNR {best_asn['psnr']:.2f} dB "
          f"({best_asn['psnr_gain_over_gaussian_dB']:+.2f} dB vs Gaussian sigma=1), "
          f"SSIM {best_asn['ssim']:.3f}", '']
if loc:
    L += ['## localization (held-out fold-'
          f"{loc.get('best_fold', '?')} val, {loc.get('n_samples', '?')} images)", '',
          f"{loc.get('n_atoms', '?')} atoms; NN spacing "
          f"{loc.get('spacing_px_mean', float('nan')):.2f} px = "
          f"{loc.get('spacing_A_mean', float('nan')):.2f} A; refine success "
          f"{100 * (loc.get('refine_success_rate') or 0):.1f}%"]
    if loc.get('gt_precision') is not None:
        L += [f"vs GT positions: P {loc['gt_precision']:.3f} / R {loc['gt_recall']:.3f} "
              f"/ F1 {loc['gt_f1']:.3f}; RMSE {loc['gt_rmse_px']:.2f} px"]
    L += ['']
if prec:
    fr = prec['refined_fractional']
    L += ['## sub-pixel precision (noisy input, shift recovery)', '',
          f"pair RMSE(2D) {fr['rmse_2d']:.3f} px "
          f"(~{prec['refined_single_shot_rmse_est']:.3f} px single-shot); "
          f"sigma x/y {fr['sigma_x']:.3f}/{fr['sigma_y']:.3f} px; "
          f"raw-peak baseline {prec['raw_fractional']['rmse_2d']:.3f} px", '']
L += ['## caveats', ''] + [f"{i}. {c}" for i, c in enumerate(caveats, 1)] + ['']
L += ['## artifacts', ''] + [f"- [{'x' if ok else ' '}] {n}" for n, ok in artifacts.items()]
with open(os.path.join(OUT, 'report.md'), 'w') as f:
    f.write('\n'.join(L))

print(json.dumps({k: report[k] for k in
                  ('environment', 'best_arch_by_mean_iou', 'best_atomsegnet',
                   'est_total_gpu_hours', 'localization', 'precision')},
                 indent=2, default=str))
print(f"\nwrote {os.path.join(OUT, 'report.json')} and report.md")


## Notes

Folder names. TEM-ImageNet-v1.3 ships `image/`, `noNoise/`, `circularMask/`,
`gaussianMask/` and `position/`. Section 3 auto-detects common variants and
prints what it matched; if something is missing, set `CFG.SUBDIR_*` directly.

Ground truth for localization. `position/` stores unit-cell vectors, not
per-atom coordinates, so section 11 takes intensity-weighted blob centroids
from `gaussianMask/` instead. `circularMask/` is the fallback and is coarser.

Pixel calibration. `CFG.PIXEL_SIZE_A` is 0.20 A/px, a placeholder for
simulated data. Set it to the real detector pixel size before quoting any
spacing or RMSE figure in angstroms.

Noise2Void. The standard masked-pixel variant: about 2% of pixels are replaced
by a random neighbour and the loss is MSE at those positions against the
original noisy value. It pre-trains AtomSegNet on noisy images alone, then the
weights warm-start the supervised stage.

Speed. bf16 autocast on the A100, batch 32, channels-last, and a uint8 memmap
cache on node-local scratch. Watch the `img/s` and `GB` figures in each epoch
line: if throughput is well below the loader probe from section 4, the GPU is
starved and the cache is not working. If GPU memory sits far below 35 GB,
raise `CFG.BATCH`.

Resuming. Every long cell checkpoints after each epoch and stops cleanly ten
minutes before the Slurm walltime. Resubmit `train.sbatch` and it continues
from the last completed epoch. See README_HPC.md.

Reproducing the CV claims. `runs/report.json` collects the PSNR gain over the
Gaussian-blur baseline, mean IoU per architecture with the best-F1 threshold,
and the sub-pixel localization error in pixels and angstroms.


In [ ]:
print("CFG.BATCH =", CFG.BATCH)
print("Actual batch =", dl.batch_size)
print("Iterations/epoch =", len(dl))

In [ ]:
print("samples:", len(common))
for a in CFG.ARCHS:
    print(a, "batch =", arch_batch(a))

## C

In [ ]:
for i, src in enumerate(In):
    if src and "def evaluate" in src:
        head = [l for l in src.splitlines() if l.strip()][:4]
        print(f"--- In[{i}] ---")
        for l in head:
            print("   ", l)

In [ ]:
get_ipython().run_cell(In[31])


In [ ]:
src = In[31]
print(len(src.splitlines()), "lines")
for l in src.splitlines():
    s = l.strip()
    if s and not l[0].isspace() and not s.startswith("#"):
        print("  ", l)

In [ ]:
import inspect
print("signature :", inspect.signature(evaluate))
print("is wrapper:", "_orig_evaluate" in inspect.getsource(inspect.unwrap(evaluate)))
print("decorator :", inspect.getsourcelines(inspect.unwrap(evaluate))[0][0].strip())

In [ ]:
src = In[32]
for l in src.splitlines():
    s = l.strip()
    if s.startswith(("def ", "fit =", "evaluate =", "_orig", "CFG.")):
        print("  ", l)

In [ ]:
import inspect, re
src = inspect.getsource(fit)
for l in src.splitlines():
    if "evaluate(" in l or "copy_for_eval" in l:
        print("  ", l.strip())

In [ ]:
print("LAM_DEN       =", CFG.LAM_DEN)
print("LAM_SEG       =", CFG.LAM_SEG)
print("LR            =", CFG.LR)
print("WD            =", CFG.WD)
print("EMA_DECAY     =", CFG.EMA_DECAY)
print("PRECISION     =", CFG.PRECISION)
print("GRAD_ACCUM    =", CFG.GRAD_ACCUM)
print("MAX_SAMPLES   =", CFG.MAX_SAMPLES)

print("\nSamples:", len(common))

In [ ]:
# Inspect one validation batch and model output ranges
va_names_test = [common[i] for i in list(
    KFold(
        n_splits=CFG.N_FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(common)
)[0][1][:32]]

ds_test = TEMSegDataset(
    va_names_test,
    noisy_map,
    clean_map,
    mask_map,
    gauss_map=gauss_map,
    train=False,
)

dl_test = DataLoader(
    ds_test,
    **loader_options(False, batch=4)
)

b = next(iter(dl_test))

noisy = b[0]
clean = b[1]
mask  = b[2]

print("NOISY :", noisy.min().item(), noisy.max().item(),
      noisy.mean().item(), noisy.std().item())

print("CLEAN :", clean.min().item(), clean.max().item(),
      clean.mean().item(), clean.std().item())

print("MASK  :", mask.min().item(), mask.max().item(),
      mask.mean().item())

In [ ]:
CFG.LR = 3e-4

print("LR =", CFG.LR)
print("LAM_DEN =", CFG.LAM_DEN)
print("LAM_SEG =", CFG.LAM_SEG)